# Perfiles ciudadanos de participación electoral en México

Reto 1 del Hackathon INE 2026, eje de segmentación sociodemográfica.

Este cuaderno agrupa a los municipios de México según cómo participan en las elecciones federales de 2009 a 2024. Usa los Conteos Censales de Participación Ciudadana del INE, que son datos agregados y no contienen información personal. Los perfiles describen la población de cada municipio, no a personas individuales.

El proceso corre de principio a fin sin intervención manual.

| Paso | Qué se hace |
|---|---|
| 1 a 4 | Se cargan los archivos del INE y se elige un solo tipo de archivo, para no contar dos veces a la misma población |
| 5 | Se arma una tabla con una fila por municipio y elección, y se valida contra las cifras del INE |
| 6 | Cada municipio se empareja con su polígono del INEGI |
| 7 y 8 | Se eligen las variables y el número de perfiles, y se agrupan los municipios con K-Means |
| 9 y 10 | Se describe cada perfil y se obtienen reglas sencillas para reconocerlo |
| 11 y 12 | Se revisa si los perfiles forman regiones, a nivel municipal y estatal |
| 13 | Se analiza cómo cambia cada municipio de 2009 a 2024 |
| 14 | Se proponen estrategias por perfil y municipios prioritarios |
| 15 y 16 | Se generan el dashboard y el paquete de entregables |

**Notación**

- `LN`: lista nominal (personas que podían votar)
- `SV`: sí votó
- `NV`: no votó
- `NS`: no se sabe si votó (registros sin información)
- Participación = `SV / (SV + NV)`, la misma fórmula que usa la plataforma de Conteos Censales del INE
- pp: puntos porcentuales (diferencia entre dos porcentajes)

## 1. Librerías y entorno

Detecta si el cuaderno corre en Google Colab, instala las librerías que falten y, en Colab, conecta Google Drive.

In [ ]:
# Librerías
import sys, os, subprocess, importlib, warnings

warnings.filterwarnings('ignore')
EN_COLAB = 'google.colab' in sys.modules


def asegurar_paquete(modulo):
    """Instala un paquete sólo si todavía no está disponible."""
    try:
        importlib.import_module(modulo)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', modulo], check=False)


for paquete in ['geopandas', 'libpysal', 'esda', 'plotly', 'shapely']:
    asegurar_paquete(paquete)

if EN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')

import re, glob, json, shutil, zipfile, unicodedata, difflib, textwrap, time, html
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import ConvexHull
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from libpysal.weights import Queen, KNN, attach_islands, w_subset
from esda.moran import Moran, Moran_Local
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, HTML

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}, pandas {pd.__version__}, geopandas {gpd.__version__}")

## 2. Configuración

Normalmente sólo hay que cambiar `EQUIPO`. Aquí también están las cifras de participación publicadas por el INE, que se usan en el paso 5 para validar los cálculos.

In [ ]:
# Parámetros generales
EQUIPO = "THE DATA SCIENTISTS, Universidad Veracruzana"  # aparece en el resumen metodológico
NOMBRE_ZIP = "INE_BASES_COMPRIMIDAS.zip"  # se busca en todo Google Drive
RUTA_ZIP = os.environ.get('RETO1_ZIP')  # opcional: ruta directa al ZIP

BASE = '/content' if EN_COLAB else os.getcwd()
DIR_DATOS = f'{BASE}/data_electoral_extraida'
DIR_SALIDA = f'{BASE}/salidas_reto1'
os.makedirs(DIR_SALIDA, exist_ok=True)

# Segmentación: la convocatoria pide de 4 a 6 perfiles
K_RANGO = (4, 5, 6)
K_FORZADO = None  # por ejemplo 5, para fijar el número de perfiles
SEMILLA = 42

# Si el paso 4 elige mal el tipo de archivo, se puede fijar aquí con su nombre exacto
FAMILIA_FORZADA = None

# Para cambiar a mano el nombre de los perfiles: {0: 'Nombre', 1: '...'}
NOMBRES_MANUALES = {}

# Polígonos municipales del INEGI (Marco Geoestadístico 2022)
URL_GEO = os.environ.get(
    'RETO1_GEO', 'https://raw.githubusercontent.com/PhantomInsights/mexico-geojson/main/2022/mexico.zip'
)

# Participación nacional publicada, para validar los cálculos.
# Plataforma de Conteos Censales de Participación Ciudadana del INE: votaron / (votaron + no votaron)
REF_INE = {2009: 44.1, 2012: 62.1, 2015: 47.1, 2018: 62.4, 2021: 51.8, 2024: 59.7}
# Cómputos distritales (otra fuente, sólo como referencia)
REF_COMPUTOS = {2009: 44.6, 2012: 63.1, 2015: 47.7, 2018: 63.4, 2021: 52.7, 2024: 61.0}
ELECCION_TIPO = {
    2009: 'Intermedia',
    2012: 'Presidencial',
    2015: 'Intermedia',
    2018: 'Presidencial',
    2021: 'Intermedia',
    2024: 'Presidencial',
}

MOSTRAR_DASHBOARD_EN_NOTEBOOK = True

## 3. Descompresión de los datos

El archivo `INE_BASES_COMPRIMIDAS.zip` contiene un ZIP por elección. Cada uno se extrae en su propia carpeta para no perder el año.

In [ ]:
# Localizar y descomprimir el ZIP
def localizar_zip():
    if RUTA_ZIP and os.path.exists(RUTA_ZIP):
        return RUTA_ZIP
    for raiz in ['/content/drive/MyDrive', '/content', os.getcwd()]:
        hallados = glob.glob(f"{raiz}/**/{NOMBRE_ZIP}", recursive=True)
        if hallados:
            return hallados[0]
    raise FileNotFoundError(f"No se encontró '{NOMBRE_ZIP}'. Súbelo a Drive o define RUTA_ZIP.")


def extraer_recursivo(zip_path, destino):
    """Extrae cada ZIP anidado en una carpeta con su propio nombre, para no perder el año."""
    pendientes, n = [(zip_path, destino)], 0
    while pendientes:
        zp, dst = pendientes.pop()
        os.makedirs(dst, exist_ok=True)
        try:
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(dst)
            n += 1
        except Exception as e:
            print(f"  No se pudo abrir {os.path.basename(zp)}: {e}")
            continue
        if zp != zip_path:
            os.remove(zp)
        for z in glob.glob(f"{dst}/**/*.zip", recursive=True):
            pendientes.append((z, os.path.splitext(z)[0]))
    return n


ruta = localizar_zip()
print("ZIP localizado:", ruta)
if os.path.exists(DIR_DATOS):
    shutil.rmtree(DIR_DATOS)
os.makedirs(DIR_DATOS)
copia = os.path.join(DIR_DATOS, os.path.basename(ruta))
shutil.copy(ruta, copia)
n = extraer_recursivo(copia, os.path.join(DIR_DATOS, 'base'))
os.remove(copia)
ARCHIVOS = sorted(
    glob.glob(f"{DIR_DATOS}/**/*.csv", recursive=True) + glob.glob(f"{DIR_DATOS}/**/*.CSV", recursive=True)
)
print(f"{n} archivos ZIP descomprimidos, {len(ARCHIVOS)} archivos CSV")

## 4. Inventario de archivos y elección de la fuente

El INE entrega los mismos datos en varios cortes (por sección, por municipio, por sexo y edad). Si se usaran todos juntos, la misma población se contaría varias veces. Aquí se clasifica cada archivo por nivel geográfico y por tipo, y se elige un solo tipo que cubra todo el país en todas las elecciones. Las claves de estado, municipio, distrito y sección se usan como identificadores y nunca como variables.

Cómo leer el resultado: la tabla muestra cada tipo de archivo con sus años, filas y lista nominal total. Abajo aparece el tipo elegido para la tabla municipal y el que se usa para los datos por sexo.

In [ ]:
# Funciones para leer y clasificar los archivos
ARCHIVOS = sorted(set(ARCHIVOS))


def slug(t):
    t = unicodedata.normalize('NFKD', str(t)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '_', t.lower()).strip('_')


def norm_col(c):
    t = unicodedata.normalize('NFKD', str(c)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^A-Z0-9]+', '_', t.upper()).strip('_')


NOMBRES_ESTADO = [
    'aguascalientes',
    'baja_california_sur',
    'baja_california',
    'campeche',
    'coahuila',
    'colima',
    'chiapas',
    'chihuahua',
    'ciudad_de_mexico',
    'cdmx',
    'distrito_federal',
    'durango',
    'guanajuato',
    'guerrero',
    'hidalgo',
    'jalisco',
    'estado_de_mexico',
    'mexico',
    'michoacan',
    'morelos',
    'nayarit',
    'nuevo_leon',
    'oaxaca',
    'puebla',
    'queretaro',
    'quintana_roo',
    'san_luis_potosi',
    'sinaloa',
    'sonora',
    'tabasco',
    'tamaulipas',
    'tlaxcala',
    'veracruz',
    'yucatan',
    'zacatecas',
    'de_zaragoza',
    'de_ocampo',
    'de_ignacio_de_la_llave',
    'de_arteaga',
]


def patron_familia(ruta):
    s = slug(os.path.splitext(os.path.basename(ruta))[0])
    s = re.sub(r'(19|20)\d{2}', '', s)
    for e in sorted(NOMBRES_ESTADO, key=len, reverse=True):
        s = re.sub(rf'(^|_){e}(_|$)', '_', s)
    s = re.sub(r'\d+', '', s)
    return re.sub(r'_+', '_', s).strip('_') or 'sin_patron'


def leer_csv(ruta):
    for enc in ('utf-8', 'utf-8-sig', 'latin-1'):
        try:
            d = pd.read_csv(ruta, encoding=enc, low_memory=False)
            if d.shape[1] == 1:
                d = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
            d.columns = [norm_col(c) for c in d.columns]
            return d
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  No se pudo leer {os.path.basename(ruta)}: {e}")
            return None
    return None


# Columnas identificadoras conocidas (nunca entran como variables)
RE_EDO = re.compile(r'^(EDOCVE|CVE_?EDO|CVE_?ENT|ID_?ENTIDAD|ENTIDAD|ID_?ESTADO|EDO)$')
RE_MUN = re.compile(r'^(MPIOCVE|CVE_?MUN|CVE_?MPIO|ID_?MUNICIPIO|MUNICIPIO_?CVE|MPIO)$')
RE_SECC = re.compile(r'^(SECCION|SECC|ID_?SECCION|SECCION_?ELECTORAL|CVE_?SECC)$')
RE_DEF = re.compile(r'^(DEF_?N|DTTO_?FED|DISTRITO_?FEDERAL|ID_?DISTRITO|DF)$')
RE_DEL = re.compile(r'^(DEL_?N|DTTO_?LOC|DISTRITO_?LOCAL|DL)$')
RE_ANIO = re.compile(r'^(AELEC|ANIO|ANO|YEAR|ANIO_?ELECCION)$')
RE_SEXO = re.compile(r'^(SEXO|SEX|GENERO)$')
# El nombre del municipio viene en la columna MPIONOM
RE_NOMMUN = re.compile(r'^(MUNICIPIO|NOM_?MUN|NOMBRE_?MUNICIPIO|NOM_?MPIO|DESC_?MUNICIPIO|MPIONOM)$')


def buscar(cols, rx):
    return next((c for c in cols if rx.match(c)), None)

In [ ]:
# Leer cada archivo y resumir su contenido
INVENTARIO, DATOS = [], {}
t0 = time.time()
for ruta in ARCHIVOS:
    d = leer_csv(ruta)
    if d is None or d.empty:
        continue
    cols = list(d.columns)
    c_edo, c_mun, c_secc = buscar(cols, RE_EDO), buscar(cols, RE_MUN), buscar(cols, RE_SECC)
    c_def, c_del, c_anio, c_sexo = (
        buscar(cols, RE_DEF),
        buscar(cols, RE_DEL),
        buscar(cols, RE_ANIO),
        buscar(cols, RE_SEXO),
    )
    if c_anio and pd.to_numeric(d[c_anio], errors='coerce').notna().any():
        anio = int(pd.to_numeric(d[c_anio], errors='coerce').mode().iloc[0])
    else:
        m = re.search(r'(2009|2012|2015|2018|2021|2024)', os.path.relpath(ruta, DIR_DATOS))
        anio = int(m.group(0)) if m else None
    if c_secc:
        nivel = 'seccion'
    elif c_mun and pd.to_numeric(d[c_mun], errors='coerce').notna().mean() > 0.9:
        nivel = 'municipio'
    elif c_del:
        nivel = 'distrito_local'
    elif c_def:
        nivel = 'distrito_federal'
    else:
        nivel = 'entidad'
    fam = f"{nivel}|{patron_familia(ruta)}"
    INVENTARIO.append(
        dict(
            archivo=os.path.relpath(ruta, DIR_DATOS),
            familia=fam,
            nivel=nivel,
            anio=anio,
            filas=len(d),
            columnas=len(cols),
            edo=c_edo,
            mun=c_mun,
            sexo=c_sexo,
            tiene_LN='LN' in cols,
            tiene_SV='SV' in cols,
            LN_total=pd.to_numeric(d['LN'], errors='coerce').sum() if 'LN' in cols else np.nan,
        )
    )
    DATOS[ruta] = d
INVENTARIO = pd.DataFrame(INVENTARIO)


def dato_inventario(ruta, campo):
    """Devuelve la familia o el año de un archivo, según el inventario."""
    return INVENTARIO.loc[INVENTARIO.archivo == os.path.relpath(ruta, DIR_DATOS), campo].iloc[0]


print(f"{len(INVENTARIO)} archivos leídos en {time.time() - t0:.0f} s\n")

resumen = (
    INVENTARIO.groupby('familia')
    .agg(
        nivel=('nivel', 'first'),
        archivos=('archivo', 'size'),
        anios=('anio', lambda s: sorted(set(s.dropna().astype(int)))),
        filas=('filas', 'sum'),
        LN_total=('LN_total', 'sum'),
        con_mun=('mun', lambda s: s.notna().mean()),
        con_sexo=('sexo', lambda s: s.notna().any()),
        con_SV=('tiene_SV', 'mean'),
    )
    .sort_values('filas', ascending=False)
)
display(resumen)

In [ ]:
# Elegir el tipo de archivo que se usará
candidatas = resumen[(resumen.con_mun > 0.9) & (resumen.con_SV > 0.9)].copy()
if FAMILIA_FORZADA:
    FAMILIA = FAMILIA_FORZADA
elif candidatas.empty:
    raise ValueError("Ninguna familia tiene clave de municipio + LN + SV. Revisa la tabla y fija FAMILIA_FORZADA.")
else:
    candidatas['n_anios'] = candidatas.anios.map(len)
    candidatas['LN_por_anio'] = candidatas.LN_total / candidatas.n_anios
    maxln = candidatas.LN_por_anio.max()
    candidatas['completa'] = candidatas.LN_por_anio >= 0.97 * maxln  # descarta cortes parciales (p. ej. un solo sexo)
    orden_nivel = {'municipio': 0, 'distrito_local': 1, 'distrito_federal': 2, 'seccion': 3, 'entidad': 4}
    candidatas['o'] = candidatas.nivel.map(orden_nivel)
    FAMILIA = candidatas.sort_values(
        ['completa', 'n_anios', 'con_sexo', 'o'], ascending=[False, False, True, True]
    ).index[0]
FAM_SEXO = next((f for f in resumen.index if resumen.loc[f, 'con_sexo'] and resumen.loc[f, 'con_mun'] > 0.9), None)
# Variante: el sexo viene en el nombre del archivo (HOMBRES / MUJERES)
FAM_H = next((f for f in resumen.index if re.search(r'hombre', f) and resumen.loc[f, 'con_mun'] > 0.9), None)
FAM_M = next((f for f in resumen.index if re.search(r'mujer', f) and resumen.loc[f, 'con_mun'] > 0.9), None)
print(f"\nTipo de archivo elegido para la tabla municipal: {FAMILIA} ({resumen.loc[FAMILIA, 'archivos']} archivos)")
if FAM_SEXO or (FAM_H and FAM_M):
    print(f"Tipo de archivo con datos por sexo: {FAM_SEXO or (FAM_H + ' + ' + FAM_M)}")
else:
    print("No hay datos por sexo, así que se omite la brecha de género.")

## 5. Tabla municipio por elección

Se suman los conteos de cada municipio en cada elección y a partir de ellos se calculan todas las tasas. La participación se calcula como `SV / (SV + NV)`. Los registros `NS`, de los que no se sabe si la persona votó, no entran en el cálculo, igual que en la plataforma del INE. Nunca se promedian porcentajes.

Además de la participación total se calculan:

- la participación respecto al país en la misma elección (`part_rel`), para comparar elecciones intermedias con presidenciales
- la brecha de cada grupo de edad respecto al total de su municipio
- la composición de la lista nominal por edad y por tipo de sección (urbana, mixta y rural)
- la brecha entre mujeres y hombres

Cómo leer la validación: la participación calculada debe coincidir con la de la plataforma de Conteos Censales del INE. Los cómputos distritales son otra fuente y difieren alrededor de 1 pp.

In [ ]:
# Sumar los conteos por municipio y elección
def construir_panel(familia):
    partes = []
    for ruta, d in DATOS.items():
        if dato_inventario(ruta, 'familia') != familia:
            continue
        cols = list(d.columns)
        c_edo, c_mun = buscar(cols, RE_EDO), buscar(cols, RE_MUN)
        c_anio, c_nom, c_sexo = buscar(cols, RE_ANIO), buscar(cols, RE_NOMMUN), buscar(cols, RE_SEXO)
        x = d.copy()
        if c_sexo is not None:  # si hay filas de total, quedarse sólo con ellas
            tot = x[c_sexo].astype(str).str.upper().str.contains('TOT|AMBOS|^T$')
            if tot.any():
                x = x[tot]
        x['_anio'] = pd.to_numeric(x[c_anio], errors='coerce') if c_anio else dato_inventario(ruta, 'anio')
        x['_edo'] = pd.to_numeric(x[c_edo], errors='coerce')
        x['_mun'] = pd.to_numeric(x[c_mun], errors='coerce')
        x['_nom'] = x[c_nom].astype(str).str.strip() if c_nom else np.nan
        partes.append(x)
    panel = pd.concat(partes, ignore_index=True, sort=False)
    panel = panel[panel._edo.between(1, 32) & (panel._mun > 0) & panel._anio.notna()]
    conteo = [c for c in panel.columns if re.match(r'^(LN|SV|NV|NS)(_|$)', c) or re.match(r'^N_SECC', c)]
    for c in conteo:
        panel[c] = pd.to_numeric(panel[c], errors='coerce')
    g = panel.groupby(['_anio', '_edo', '_mun'])
    municipal = g[conteo].sum(min_count=1)
    municipal['nombre'] = g['_nom'].agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else np.nan)
    municipal = municipal.reset_index().rename(columns={'_anio': 'anio', '_edo': 'edo', '_mun': 'mun'})
    municipal[['anio', 'edo', 'mun']] = municipal[['anio', 'edo', 'mun']].astype(int)
    municipal['cvegeo'] = municipal.edo.map('{:02d}'.format) + municipal.mun.map('{:03d}'.format)
    return municipal, conteo


panel, CONTEOS = construir_panel(FAMILIA)
print(
    f"Tabla: {len(panel):,} filas municipio-elección, {panel.cvegeo.nunique():,} municipios, años {sorted(int(a) for a in panel.anio.unique())}"
)

In [ ]:
# Tasas de participación por edad y tipo de sección
EDADES = sorted(
    {m.group(1) for c in CONTEOS for m in [re.match(r'^LN_(E\d+.*)$', c)] if m},
    key=lambda s: int(re.search(r'\d+', s).group()),
)
TIPOS = sorted({m.group(1) for c in CONTEOS for m in [re.match(r'^LN_TS_([A-Z]+)$', c)] if m})


def lim_inf(e):
    return int(re.search(r'\d+', e).group())


GRUPOS = {
    'jov': [e for e in EDADES if lim_inf(e) < 30],
    'adu': [e for e in EDADES if 30 <= lim_inf(e) < 50],
    'mad': [e for e in EDADES if 50 <= lim_inf(e) < 70],
    'may': [e for e in EDADES if lim_inf(e) >= 70],
}
print("Grupos de edad:", EDADES, "| tipos de sección:", TIPOS)


def tasa(df, sufijos, base='SV'):
    num = sum(df.get(f'{base}_{s}', 0) for s in sufijos)
    den = sum(df.get(f'SV_{s}', 0) + df.get(f'NV_{s}', 0) for s in sufijos)
    return 100 * num / den.replace(0, np.nan) if isinstance(den, pd.Series) else np.nan


panel['part'] = 100 * panel.SV / (panel.SV + panel.NV).replace(0, np.nan)
PART_NACIONAL = panel.groupby('anio').apply(lambda d: 100 * d.SV.sum() / (d.SV.sum() + d.NV.sum()))
panel['part_nac'] = panel.anio.map(PART_NACIONAL)
panel['part_rel'] = panel.part - panel.part_nac
for gk, sufs in GRUPOS.items():
    if sufs:
        panel[f'part_{gk}'] = tasa(panel, sufs)
        panel[f'brecha_{gk}'] = panel[f'part_{gk}'] - panel.part
for e in EDADES:
    panel[f'part_{e}'] = tasa(panel, [e])
if GRUPOS['jov']:
    panel['pct_ln_jov'] = 100 * sum(panel[f'LN_{e}'] for e in GRUPOS['jov']) / panel.LN
if GRUPOS['may'] or GRUPOS['mad']:
    mayores = [e for e in EDADES if lim_inf(e) >= 60]
    panel['pct_ln_60'] = 100 * sum(panel[f'LN_{e}'] for e in mayores) / panel.LN
for t in TIPOS:
    panel[f'pct_ln_{t}'] = 100 * panel[f'LN_TS_{t}'] / panel.LN
    panel[f'part_ts_{t}'] = tasa(panel, [f'TS_{t}'])
if 'R' in TIPOS:
    panel['pct_rural'] = panel.pct_ln_R
if 'U' in TIPOS:
    panel['pct_urbano'] = panel.pct_ln_U
elif 'R' in TIPOS and 'M' in TIPOS:
    panel['pct_urbano'] = 100 - panel.pct_ln_R - panel.pct_ln_M
panel['log_ln'] = np.log10(panel.LN)
panel['abstencion'] = panel.NV

In [ ]:
# Brecha entre mujeres y hombres
def panel_sexo():
    def sumar(fams, filtro=None):
        out = []
        for ruta, d in DATOS.items():
            fam = dato_inventario(ruta, 'familia')
            if fam not in fams:
                continue
            x = d.copy()
            x['_fam'] = fam
            x['_anio'] = (
                pd.to_numeric(x[buscar(x.columns, RE_ANIO)], errors='coerce')
                if buscar(x.columns, RE_ANIO)
                else dato_inventario(ruta, 'anio')
            )
            x['_edo'] = pd.to_numeric(x[buscar(x.columns, RE_EDO)], errors='coerce')
            x['_mun'] = pd.to_numeric(x[buscar(x.columns, RE_MUN)], errors='coerce')
            out.append(x)
        return pd.concat(out, ignore_index=True) if out else None

    if FAM_SEXO:
        x = sumar([FAM_SEXO])
        cs = buscar(x.columns, RE_SEXO)
        s = x[cs].astype(str).str.upper().str[0]
        x['_sx'] = np.where(s.isin(['H', '1']), 'H', np.where(s.isin(['M', '2']), 'M', None))
    elif FAM_H and FAM_M:
        x = sumar([FAM_H, FAM_M])
        x['_sx'] = np.where(x._fam == FAM_H, 'H', 'M')
    else:
        return None
    x = x[x._sx.notna()]
    t = x.groupby(['_anio', '_edo', '_mun', '_sx'])[['LN', 'SV', 'NV']].sum().unstack('_sx')
    r = pd.DataFrame(
        {
            'part_muj': 100 * t[('SV', 'M')] / (t[('SV', 'M')] + t[('NV', 'M')]),
            'part_hom': 100 * t[('SV', 'H')] / (t[('SV', 'H')] + t[('NV', 'H')]),
        }
    )
    r['brecha_genero'] = r.part_muj - r.part_hom
    r = r.reset_index()
    r.columns = ['anio', 'edo', 'mun', 'part_muj', 'part_hom', 'brecha_genero']
    r[['anio', 'edo', 'mun']] = r[['anio', 'edo', 'mun']].astype(int)
    return r


SX = panel_sexo()
if SX is not None:
    panel = panel.merge(SX, on=['anio', 'edo', 'mun'], how='left')
    print(f"Brecha de género disponible ({SX.brecha_genero.notna().mean():.0%} de municipio-elección)")

In [ ]:
# Validación contra las cifras del INE
val = pd.DataFrame(
    {'Participación calculada (%)': PART_NACIONAL.round(2), 'INE, Conteos Censales (%)': pd.Series(REF_INE)}
)
val['Diferencia (pp)'] = (val.iloc[:, 0] - val.iloc[:, 1]).round(2)
val['Cómputos distritales (%)'] = pd.Series(REF_COMPUTOS)
val['LN (millones)'] = panel.groupby('anio').LN.sum().div(1e6).round(2)
val['Municipios'] = panel.groupby('anio').cvegeo.nunique()
print("\nValidación: la participación calculada debe coincidir con la plataforma de Conteos Censales del INE.")
print("Los cómputos distritales son otra fuente (conteo de votos), por eso difieren alrededor de 1 pp.")
display(val)

## 6. Mapas municipales y emparejamiento INE con INEGI

Se cargan los polígonos municipales del INEGI. Las claves de municipio del INE no coinciden con las del INEGI en muchos estados (por ejemplo, la clave 089 de Veracruz es Xalapa para el INE y Jáltipan para el INEGI). Por eso cada municipio se empareja con su polígono por nombre, dentro de su estado.

Cómo leer el resultado: se muestran tres tablas de revisión. La primera lista los municipios sin polígono, que deben ser sólo los creados después de 2022. La segunda lista los emparejados con un nombre parecido pero no idéntico, para revisarlos. La tercera muestra el municipio con más lista nominal de cada estado, que debe ser su ciudad principal.

In [ ]:
# Cargar los polígonos municipales
def cargar_geo():
    destino = f'{BASE}/geo_municipios'
    os.makedirs(destino, exist_ok=True)
    js = glob.glob(f'{destino}/*.json') + glob.glob(f'{destino}/*.geojson')
    if not js:
        if os.path.exists(URL_GEO):
            shutil.copy(URL_GEO, f'{destino}/geo.zip')
        else:
            import urllib.request

            print("Descargando polígonos municipales...")
            urllib.request.urlretrieve(URL_GEO, f'{destino}/geo.zip')
        with zipfile.ZipFile(f'{destino}/geo.zip') as z:
            z.extractall(destino)
        js = glob.glob(f'{destino}/*.json') + glob.glob(f'{destino}/*.geojson')
    g = gpd.read_file(js[0])
    g['cvegeo'] = g['CVEGEO'].astype(str).str.zfill(5)
    g['nom_mun'] = g.get('NOMGEO', g.get('NOM_MUN'))
    g['nom_ent'] = g['NOM_ENT']
    return g[['cvegeo', 'nom_mun', 'nom_ent', 'geometry']].to_crs(4326)


GEO = cargar_geo()
ENTIDADES = GEO.assign(e=GEO.cvegeo.str[:2].astype(int)).groupby('e').nom_ent.first().to_dict()
print(f"Polígonos: {len(GEO):,} municipios")

In [ ]:
# Emparejar cada municipio del INE con su polígono
# Orden: 1) nombre idéntico, 2) un nombre contiene al otro, 3) nombre parecido. Cada polígono se usa una vez.
def limpio(t):
    return re.sub(r'\s+', ' ', slug(t).replace('_', ' ')).strip()


llave = (
    panel.groupby('cvegeo')
    .agg(edo=('edo', 'first'), nombre=('nombre', lambda s: s.dropna().iloc[-1] if s.notna().any() else np.nan))
    .reset_index()
)
geo_idx = GEO.assign(edo=GEO.cvegeo.str[:2].astype(int), k=GEO.nom_mun.map(limpio))
print(f"Municipios con nombre en los datos del INE: {llave.nombre.notna().mean():.1%}")

mapa_clave, metodo = {}, {}
for edo, sub in llave.groupby('edo'):
    cands = geo_idx[geo_idx.edo == edo].sort_values('cvegeo')
    libres = {}  # nombre INEGI -> claves INEGI aún sin usar
    for k_, c_ in zip(cands.k, cands.cvegeo):  # (hay nombres repetidos, p. ej. en Oaxaca)
        libres.setdefault(k_, []).append(c_)

    def tomar(nombre):
        c = libres[nombre].pop(0)
        if not libres[nombre]:
            del libres[nombre]
        return c

    pendientes = []
    for _, r in sub.sort_values('cvegeo').iterrows():  # 1) nombre idéntico
        k = limpio(r.nombre) if pd.notna(r.nombre) else ''
        if k and k in libres:
            mapa_clave[r.cvegeo], metodo[r.cvegeo] = tomar(k), 'nombre idéntico'
        else:
            pendientes.append((r.cvegeo, k))
    resto = []
    for cve, k in pendientes:  # 2) un nombre contiene al otro
        opciones = [n for n in libres if k and (k in n or n in k)]
        if len(opciones) == 1:
            mapa_clave[cve], metodo[cve] = tomar(opciones[0]), 'nombre contenido'
        else:
            resto.append((cve, k))
    for cve, k in resto:  # 3) nombre parecido
        m = difflib.get_close_matches(k, list(libres), n=1, cutoff=0.75) if k else []
        if m:
            mapa_clave[cve], metodo[cve] = tomar(m[0]), 'nombre parecido'

panel['cvegeo_inegi'] = panel.cvegeo.map(mapa_clave)
cob = panel.drop_duplicates('cvegeo').cvegeo_inegi.notna().mean()
distinta = np.mean([mapa_clave[c] != c for c in mapa_clave]) if mapa_clave else 0
print(f"Municipios con polígono: {cob:.1%}, métodos: {pd.Series(metodo).value_counts().to_dict()}")
print(f"La clave del INE es distinta a la del INEGI en {distinta:.1%} de los municipios emparejados")

# Revisión 1: municipios que se quedaron sin polígono
sin_poligono = llave[~llave.cvegeo.isin(mapa_clave)]
if len(sin_poligono):
    print(f"\nSin polígono ({len(sin_poligono)}), normalmente municipios creados después del marco INEGI 2022:")
    display(sin_poligono.assign(estado=sin_poligono.edo.map(ENTIDADES))[['estado', 'cvegeo', 'nombre']])

# Revisión 2: emparejamientos que no son idénticos, para revisarlos a ojo
nom_inegi = GEO.set_index('cvegeo').nom_mun
revisar = llave[llave.cvegeo.map(metodo).isin(['nombre contenido', 'nombre parecido'])].copy()
if len(revisar):
    revisar['nombre_INEGI'] = revisar.cvegeo.map(mapa_clave).map(nom_inegi)
    revisar['metodo'] = revisar.cvegeo.map(metodo)
    print(f"\nEmparejados por nombre no idéntico ({len(revisar)}), revisa que cada par sea el mismo municipio:")
    display(revisar[['cvegeo', 'nombre', 'nombre_INEGI', 'metodo']])

# Revisión 3: el municipio con más lista nominal de cada estado debe ser su ciudad principal
ult = panel[panel.anio == panel.anio.max()]
mayores = ult.loc[ult.groupby('edo').LN.idxmax(), ['edo', 'nombre', 'cvegeo_inegi', 'LN']]
mayores['estado'] = mayores.edo.map(ENTIDADES)
mayores['polígono INEGI'] = mayores.cvegeo_inegi.map(nom_inegi)
print("\nMunicipio con más lista nominal en cada estado (el nombre del INE y el del polígono deben coincidir):")
display(mayores[['estado', 'nombre', 'polígono INEGI', 'LN']].reset_index(drop=True))
if cob < 0.97:
    print("Cobertura baja: revisa la tabla de municipios sin polígono.")

In [ ]:
# Geometría para mapas y matriz de vecinos
GEO_SIMPLE = GEO[['cvegeo', 'geometry']].copy()
GEO_SIMPLE['geometry'] = GEO_SIMPLE.geometry.simplify(0.01, preserve_topology=True)
GEOJSON_TXT = re.sub(r'(\d+\.\d{3})\d+', r'\1', GEO_SIMPLE.to_json(drop_id=True))


def pesos_espaciales(cves):
    sub = GEO[GEO.cvegeo.isin(cves)].reset_index(drop=True)
    w = Queen.from_dataframe(sub, use_index=False, silence_warnings=True)
    if w.islands:
        cent = np.c_[sub.geometry.representative_point().x, sub.geometry.representative_point().y]
        w = attach_islands(w, KNN(cent, k=1))
    w.transform = 'r'
    return w, sub.cvegeo.tolist()

## 7. Variables de segmentación y panorama nacional

Se eligen las variables con las que se comparan los municipios. Se excluyen los municipios con menos de 100 personas en la lista nominal, los valores extremos se recortan a los percentiles 1 y 99, y todas las variables se estandarizan para que ninguna pese más sólo por sus unidades.

Cómo leer el resultado: la tabla resume cada variable. La gráfica muestra la participación por edad en cada elección federal (línea continua para presidenciales y punteada para intermedias).

In [ ]:
# Variables, recorte de extremos y estandarización
ETIQ = {
    'part_rel': 'Participación vs. nacional (pp)',
    'brecha_jov': 'Brecha jóvenes 18–29 (pp)',
    'brecha_adu': 'Brecha adultos 30–49 (pp)',
    'brecha_mad': 'Brecha 50–69 (pp)',
    'brecha_may': 'Brecha 70+ (pp)',
    'pct_ln_jov': '% de LN con 18–29 años',
    'pct_ln_60': '% de LN con 60+ años',
    'pct_rural': '% de LN en secciones rurales',
    'pct_urbano': '% de LN en secciones urbanas',
    'log_ln': 'Tamaño (log10 LN)',
    'brecha_genero': 'Brecha mujeres − hombres (pp)',
    'part': 'Participación (%)',
}
CANDIDATAS = [
    'part_rel',
    'brecha_jov',
    'brecha_mad',
    'brecha_may',
    'pct_ln_jov',
    'pct_ln_60',
    'pct_rural',
    'pct_urbano',
    'log_ln',
    'brecha_genero',
]
VARIABLES_SEGMENTACION = [c for c in CANDIDATAS if c in panel and panel[c].notna().mean() > 0.8 and panel[c].std() > 0]
print("Variables de segmentación:", VARIABLES_SEGMENTACION)

datos_modelo = panel[panel.LN >= 100].dropna(subset=['part']).copy()  # municipios con lista nominal mínima
variables = datos_modelo[VARIABLES_SEGMENTACION].copy()
variables = variables.fillna(variables.median())
limites_winsor = {c: variables[c].quantile([0.01, 0.99]).values for c in VARIABLES_SEGMENTACION}  # winsorización 1–99 %
for c in VARIABLES_SEGMENTACION:
    variables[c] = variables[c].clip(*limites_winsor[c])
escalador = StandardScaler().fit(variables)
variables_estandarizadas = escalador.transform(variables)
print(
    f"Matriz de segmentación: {variables_estandarizadas.shape[0]:,} municipio-elección × {variables_estandarizadas.shape[1]} variables"
)
display(datos_modelo[VARIABLES_SEGMENTACION].describe().T.round(2))

In [ ]:
# Participación nacional por edad
ANIOS = sorted(datos_modelo.anio.unique())


def participacion_nacional(anio, sufijo):
    """Participación del país en una elección para un grupo (edad o tipo de sección)."""
    d = datos_modelo[datos_modelo.anio == anio]
    return 100 * d[f'SV_{sufijo}'].sum() / (d[f'SV_{sufijo}'].sum() + d[f'NV_{sufijo}'].sum())


CURVA_NAC = pd.DataFrame({a: {e: participacion_nacional(a, e) for e in EDADES} for a in ANIOS})
TS_NAC = pd.DataFrame({a: {t: participacion_nacional(a, f'TS_{t}') for t in TIPOS} for a in ANIOS}) if TIPOS else None
ETQ_EDAD = {e: re.sub(r'^E', '', e).replace('_', '–').replace('MAS', '+') for e in EDADES}
ETQ_TS = {'U': 'Urbana', 'M': 'Mixta', 'R': 'Rural'}

f = go.Figure()
for a in ANIOS:
    f.add_trace(
        go.Scatter(
            x=[ETQ_EDAD[e] for e in EDADES],
            y=CURVA_NAC[a],
            mode='lines+markers',
            name=f"{a} ({ELECCION_TIPO.get(a, '')})",
            line=dict(dash='solid' if ELECCION_TIPO.get(a) == 'Presidencial' else 'dot', width=2.5),
        )
    )
f.update_layout(
    title='Participación por edad, elecciones federales',
    xaxis_title='Edad',
    yaxis_title='% que votó',
    template='plotly_white',
    height=420,
)
f.show()

## 8. Número de perfiles y algoritmo

Se comparan tres algoritmos de agrupamiento (K-Means, mezcla gaussiana y Ward jerárquico) para 2 a 8 grupos, con cuatro criterios:

- silhouette: qué tan separados quedan los grupos (de -1 a 1, mayor es mejor)
- Calinski-Harabasz: mayor es mejor
- Davies-Bouldin: menor es mejor
- estabilidad: si al repetir el ajuste con el 70 % de los datos salen los mismos grupos (de 0 a 1, mayor es mejor)

El número final se elige entre 4 y 6, que es lo que pide la convocatoria, combinando los cuatro criterios. Los perfiles se ordenan de mayor a menor participación, así que el perfil 1 es el más participativo.

In [ ]:
# Comparar algoritmos y número de grupos
generador = np.random.default_rng(SEMILLA)
n = len(variables_estandarizadas)
muestra = generador.choice(n, size=min(n, 6000), replace=False)  # muestra para las métricas más lentas
METRICAS = []
for k in range(2, 9):
    km = KMeans(k, n_init=20, random_state=SEMILLA).fit(variables_estandarizadas)
    etiquetas = km.labels_
    gm = GaussianMixture(k, covariance_type='full', n_init=3, random_state=SEMILLA).fit(variables_estandarizadas)
    etiquetas_gmm = gm.predict(variables_estandarizadas)
    muestra_ward = muestra[:4000]
    etiquetas_ward = AgglomerativeClustering(k, linkage='ward').fit_predict(variables_estandarizadas[muestra_ward])
    aris = []
    for b in range(15):  # estabilidad: volver a ajustar con el 70 % de los datos
        sb = generador.choice(n, size=int(0.7 * n), replace=False)
        kb = KMeans(k, n_init=5, random_state=b).fit(variables_estandarizadas[sb])
        aris.append(adjusted_rand_score(etiquetas, kb.predict(variables_estandarizadas)))
    METRICAS.append(
        dict(
            k=k,
            silhouette=silhouette_score(variables_estandarizadas[muestra], etiquetas[muestra]),
            calinski=calinski_harabasz_score(variables_estandarizadas, etiquetas),
            davies=davies_bouldin_score(variables_estandarizadas, etiquetas),
            estabilidad=np.mean(aris),
            estabilidad_sd=np.std(aris),
            bic_gmm=gm.bic(variables_estandarizadas),
            ari_kmeans_gmm=adjusted_rand_score(etiquetas, etiquetas_gmm),
            ari_kmeans_ward=adjusted_rand_score(etiquetas[muestra_ward], etiquetas_ward),
            sil_gmm=(
                silhouette_score(variables_estandarizadas[muestra], etiquetas_gmm[muestra])
                if len(set(etiquetas_gmm[muestra])) > 1
                else np.nan
            ),
            sil_ward=silhouette_score(variables_estandarizadas[muestra_ward], etiquetas_ward),
        )
    )
    print(
        f"k={k}: silhouette {METRICAS[-1]['silhouette']:.3f}, estabilidad {METRICAS[-1]['estabilidad']:.3f}, "
        f"Davies-Bouldin {METRICAS[-1]['davies']:.2f}, acuerdo con GMM {METRICAS[-1]['ari_kmeans_gmm']:.2f} y con Ward {METRICAS[-1]['ari_kmeans_ward']:.2f}"
    )
METRICAS = pd.DataFrame(METRICAS).set_index('k')

In [ ]:
# Modelo final
def zs(s, menor_mejor=False):
    """Estandariza una métrica; si menor es mejor, le cambia el signo."""
    s = (s - s.mean()) / (s.std() + 1e-9)
    return -s if menor_mejor else s


metricas_rango = METRICAS.loc[list(K_RANGO)]
puntaje = (
    zs(metricas_rango.silhouette)
    + zs(metricas_rango.estabilidad)
    + 0.5 * zs(metricas_rango.calinski)
    + 0.5 * zs(metricas_rango.davies, True)
)
num_perfiles = K_FORZADO or int(puntaje.idxmax())
print(
    f"\nNúmero de perfiles elegido: k = {num_perfiles} (puntaje compuesto dentro de {K_RANGO}: {puntaje.round(2).to_dict()})"
)

MODELO_KMEANS = KMeans(num_perfiles, n_init=50, random_state=SEMILLA).fit(variables_estandarizadas)
etiquetas = MODELO_KMEANS.labels_
# Ordenar perfiles de mayor a menor participación relativa (perfil 1 = el más participativo)
orden = (
    pd.Series(MODELO_KMEANS.cluster_centers_[:, VARIABLES_SEGMENTACION.index('part_rel')])
    .sort_values(ascending=False)
    .index
)
remap = {old: new for new, old in enumerate(orden)}
datos_modelo['perfil'] = pd.Series(etiquetas).map(remap).values
CENTROS_Z = pd.DataFrame(MODELO_KMEANS.cluster_centers_[orden], columns=VARIABLES_SEGMENTACION)
print(datos_modelo.perfil.value_counts().sort_index().rename('municipio-elección').to_frame().T)

## 9. Descripción y nombre de cada perfil

Los indicadores de cada perfil se calculan sumando los conteos de sus municipios (es decir, ponderados por población) y se expresan como diferencia respecto al país. El nombre de cada perfil se genera a partir de sus rasgos más distintivos y se puede cambiar con `NOMBRES_MANUALES` en el paso 2.

In [ ]:
# Indicadores de cada perfil
PALETA = [
    '#2a78d6',
    '#eb6834',
    '#14a3a3',
    '#4a3aa7',
    '#d9a600',
    '#a8326f',
]  # validada para daltonismo (todas las parejas)
ULT = max(ANIOS)


def agregado(df):
    """Indicadores de un conjunto de municipios, calculados desde los conteos (ponderados por ciudadanos)."""
    # Todas las tasas usan como base SV + NV
    r = {'LN': df.LN.sum(), 'SV': df.SV.sum(), 'part': 100 * df.SV.sum() / (df.SV.sum() + df.NV.sum())}
    for gk, sufs in GRUPOS.items():
        if sufs:
            r[f'part_{gk}'] = (
                100
                * sum(df[f'SV_{e}'].sum() for e in sufs)
                / sum(df[f'SV_{e}'].sum() + df[f'NV_{e}'].sum() for e in sufs)
            )
            r[f'brecha_{gk}'] = r[f'part_{gk}'] - r['part']
    for e in EDADES:
        r[f'part_{e}'] = 100 * df[f'SV_{e}'].sum() / (df[f'SV_{e}'].sum() + df[f'NV_{e}'].sum())
    for t in TIPOS:
        r[f'pct_ln_{t}'] = 100 * df[f'LN_TS_{t}'].sum() / df.LN.sum()
        r[f'part_ts_{t}'] = 100 * df[f'SV_TS_{t}'].sum() / max(df[f'SV_TS_{t}'].sum() + df[f'NV_TS_{t}'].sum(), 1)
    if GRUPOS['jov']:
        r['pct_ln_jov'] = 100 * sum(df[f'LN_{e}'].sum() for e in GRUPOS['jov']) / df.LN.sum()
    if 'pct_ln_60' in df:
        r['pct_ln_60'] = 100 * sum(df[f'LN_{e}'].sum() for e in EDADES if lim_inf(e) >= 60) / df.LN.sum()
    if 'brecha_genero' in df and df.brecha_genero.notna().any():
        w = df.LN.where(df.brecha_genero.notna(), 0)
        r['brecha_genero'] = np.average(df.brecha_genero.fillna(0), weights=w)
    return r


# Indicadores por perfil: promedio, en todas las elecciones, de la diferencia perfil menos país
INDICADORES_PAIS = {a: agregado(datos_modelo[datos_modelo.anio == a]) for a in ANIOS}
filas = []
for p in range(num_perfiles):
    por_anio = []
    for a in ANIOS:
        s = datos_modelo[(datos_modelo.perfil == p) & (datos_modelo.anio == a)]
        if len(s):
            por_anio.append(
                {
                    k: v - INDICADORES_PAIS[a].get(k, 0) if not k.startswith(('LN', 'SV')) else v
                    for k, v in agregado(s).items()
                }
            )
    rel = pd.DataFrame(por_anio).mean()
    u = datos_modelo[(datos_modelo.perfil == p) & (datos_modelo.anio == ULT)]
    filas.append(
        dict(
            perfil=p,
            n_obs=int((datos_modelo.perfil == p).sum()),
            n_mun_ult=len(u),
            LN_ult=u.LN.sum(),
            pct_LN_ult=100 * u.LN.sum() / datos_modelo[datos_modelo.anio == ULT].LN.sum(),
            **{f'd_{k}': v for k, v in rel.items() if k not in ('LN', 'SV')},
            **{f'u_{k}': v for k, v in agregado(u).items()} if len(u) else {},
            med_rural=(
                datos_modelo.loc[datos_modelo.perfil == p, 'pct_rural'].median()
                if 'pct_rural' in datos_modelo
                else np.nan
            ),
            med_urbano=(
                datos_modelo.loc[datos_modelo.perfil == p, 'pct_urbano'].median()
                if 'pct_urbano' in datos_modelo
                else np.nan
            ),
        )
    )
PERFILES = pd.DataFrame(filas).set_index('perfil')
PERFILES['n_mun_ult'] = PERFILES.n_mun_ult.astype(int)
pais_ultima = INDICADORES_PAIS[ULT]

In [ ]:
# Nombre automático de cada perfil
def nombrar(p):
    z = CENTROS_Z.loc[p]
    c = PERFILES.loc[p]
    if c.med_rural >= 55:
        sust, fem, terr = 'Comunidades rurales', True, 'rural'
    elif c.med_urbano >= 60:
        sust, fem, terr = (
            ('Grandes ciudades', True, 'urbano') if z.get('log_ln', 0) > 0.8 else ('Ciudades medias', True, 'urbano')
        )
    else:
        sust, fem, terr = 'Municipios en transición', False, 'mixto'
    a = 'as' if fem else 'os'
    zp = z['part_rel']
    adj = (
        f'muy participativ{a}'
        if zp > 0.6
        else (
            f'participativ{a}'
            if zp > 0.2
            else (
                'con abstención alta'
                if zp < -0.6
                else 'de baja participación' if zp < -0.2 else 'de participación media'
            )
        )
    )
    clausulas = []
    if 'brecha_jov' in z:
        clausulas.append(
            (abs(z.brecha_jov), 'con juventud alejada de las urnas' if z.brecha_jov < 0 else 'con jóvenes que sí votan')
        )
    if 'brecha_may' in z:
        clausulas.append(
            (
                abs(z.brecha_may),
                'donde las personas mayores dejan de votar' if z.brecha_may < 0 else 'con personas mayores muy activas',
            )
        )
    if 'pct_ln_60' in z:
        clausulas.append((abs(z.pct_ln_60) * 0.9, f'envejecid{a}' if z.pct_ln_60 > 0 else 'con padrón joven'))
    if 'brecha_genero' in z:
        clausulas.append(
            (
                abs(z.brecha_genero) * 0.9,
                'donde las mujeres votan más' if z.brecha_genero > 0 else 'con brecha de género a favor de hombres',
            )
        )
    clausulas = [t for s, t in sorted(clausulas, reverse=True) if s > 0.45]
    return sust, adj, clausulas, terr


NOMBRES, TERR, CORTOS = {}, {}, {}
for p in range(num_perfiles):
    sust, adj, cl, terr = nombrar(p)
    if cl and cl[0].startswith('envejecid'):  # adjetivo simple: va junto al sustantivo
        nombre = f"{sust} {cl[0]} {adj}"
        cl = cl[1:]
    else:
        nombre = f"{sust} {adj}" + (f" {cl[0]}" if cl else '')
        cl = cl[1:]
    i = 0
    while nombre in NOMBRES.values() and i < len(cl):
        nombre += f" y {cl[i].split(' ', 1)[1] if cl[i].startswith('con ') else cl[i]}"
        i += 1
    if nombre in NOMBRES.values():
        nombre += f" ({p + 1})"
    NOMBRES[p] = NOMBRES_MANUALES.get(p, nombre)
    TERR[p] = terr
    CORTOS[p] = (
        f"P{p + 1} · "
        + {'rural': 'Rural', 'urbano': 'Urbano', 'mixto': 'Mixto'}[terr]
        + {True: ' participativo', False: ''}[CENTROS_Z.loc[p, 'part_rel'] > 0.2]
        + {True: ' abstencionista', False: ''}[CENTROS_Z.loc[p, 'part_rel'] < -0.2]
    )
COLORES = {p: PALETA[p % len(PALETA)] for p in range(num_perfiles)}

In [ ]:
# Descripción de cada perfil
def pp(v):
    return f"{v:+.1f} pp"


def narrativa(p):
    c = PERFILES.loc[p]
    t = (
        f"Reúne {int(c.n_mun_ult):,} municipios en {ULT}, donde están {c.LN_ult / 1e6:,.1f} millones de personas en la lista nominal "
        f"({c.pct_LN_ult:.1f} % del país). En promedio de las {len(ANIOS)} elecciones, su participación está {pp(c.d_part)} respecto al país. "
    )
    if 'd_brecha_jov' in c:
        t += (
            f"Las personas de 18 a 29 años votan {abs(c.u_brecha_jov):.1f} pp {'menos' if c.u_brecha_jov < 0 else 'más'} que el total de su municipio "
            f"(en el país la brecha es de {pais_ultima['brecha_jov']:+.1f} pp). "
        )
    if 'd_brecha_may' in c:
        t += f"Entre quienes tienen 70 años o más, la brecha es de {c.u_brecha_may:+.1f} pp. "
    if 'u_pct_ln_R' in c:
        t += f"El {c.u_pct_ln_R:.0f} % de su lista nominal vive en secciones rurales" + (
            f" y el {c.u_pct_ln_U:.0f} % en urbanas. " if 'u_pct_ln_U' in c else '. '
        )
    if 'u_brecha_genero' in c and pd.notna(c.u_brecha_genero):
        t += f"Las mujeres votan {abs(c.u_brecha_genero):.1f} pp {'más' if c.u_brecha_genero > 0 else 'menos'} que los hombres (país: {pais_ultima.get('brecha_genero', 0):+.1f} pp)."
    return t.strip()


NARRATIVAS = {p: narrativa(p) for p in range(num_perfiles)}
datos_modelo['nombre_perfil'] = datos_modelo.perfil.map(NOMBRES)

for p in range(num_perfiles):
    print(f"\nPERFIL {p + 1}: {NOMBRES[p]}\n   {textwrap.fill(NARRATIVAS[p], 110, subsequent_indent='   ')}")

tabla = PERFILES[
    ['n_mun_ult', 'pct_LN_ult', 'd_part']
    + [c for c in ['u_brecha_jov', 'u_brecha_may', 'u_pct_ln_R', 'u_pct_ln_60', 'u_brecha_genero'] if c in PERFILES]
].copy()
tabla.index = [NOMBRES[p] for p in range(num_perfiles)]
display(tabla.round(1))

## 10. Reglas para reconocer cada perfil

Un árbol de decisión poco profundo imita el agrupamiento usando las variables en sus unidades originales. Así se obtienen reglas que se pueden aplicar sin correr el modelo. Un bosque aleatorio indica qué variables separan más a los perfiles.

Cómo leer el resultado: la fidelidad es el porcentaje de municipios en que el árbol coincide con el K-Means. Cada regla indica su porcentaje de acierto.

In [ ]:
# Árbol de decisión e importancia de variables
variables_originales = datos_modelo[VARIABLES_SEGMENTACION].fillna(datos_modelo[VARIABLES_SEGMENTACION].median())
ARBOL = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=max(30, len(datos_modelo) // 200), random_state=SEMILLA
).fit(variables_originales, datos_modelo.perfil)
FIDELIDAD = (ARBOL.predict(variables_originales) == datos_modelo.perfil).mean()
print(f"Fidelidad del árbol (coincide con K-Means): {FIDELIDAD:.1%}")


def regla_por_perfil(arbol, nombres_var):
    t = arbol.tree_
    reglas = {}

    def recorrer(nodo, conds):
        if t.children_left[nodo] == -1:
            cls = int(np.argmax(t.value[nodo][0]))
            n_ = t.n_node_samples[nodo]
            pureza = t.value[nodo][0].max() / t.value[nodo][0].sum()
            if cls not in reglas or n_ > reglas[cls][1]:
                reglas[cls] = (conds, n_, pureza)
            return
        v = nombres_var[t.feature[nodo]]
        u = t.threshold[nodo]
        recorrer(t.children_left[nodo], conds + [(v, '≤', u)])
        recorrer(t.children_right[nodo], conds + [(v, '>', u)])

    recorrer(0, [])
    out = {}
    for cls, (conds, n_, pur) in reglas.items():
        simpl = {}
        for v, op, u in conds:  # quedarse con el umbral más estricto por variable y sentido
            k = (v, op)
            simpl[k] = min(simpl.get(k, u), u) if op == '≤' else max(simpl.get(k, u), u)
        out[cls] = (
            'Si ' + ' y '.join(f"{ETIQ.get(v, v).split(' (')[0]} {op} {u:.1f}" for (v, op), u in simpl.items()),
            pur,
            n_,
        )
    return out


REGLAS = regla_por_perfil(ARBOL, VARIABLES_SEGMENTACION)
for p in range(num_perfiles):
    if p in REGLAS:
        print(f"P{p + 1}: {REGLAS[p][0]}. Acierto {REGLAS[p][1]:.0%} en {REGLAS[p][2]:,} casos")

RF = RandomForestClassifier(300, max_depth=10, n_jobs=-1, random_state=SEMILLA).fit(
    variables_originales, datos_modelo.perfil
)
IMPORT = pd.Series(RF.feature_importances_, index=VARIABLES_SEGMENTACION).sort_values(ascending=False)
print("\nImportancia de cada variable para distinguir perfiles:")
display(IMPORT.rename(ETIQ).round(3).to_frame('importancia'))

## 11. ¿Los perfiles forman regiones?

Se construye la matriz de vecinos (dos municipios son vecinos si comparten frontera o un vértice) y se calculan:

- I de Moran global: si los municipios vecinos se parecen entre sí (de -1 a 1)
- LISA con 999 permutaciones: dónde están las zonas concretas de participación alta o baja
- cohesión: qué tan seguido un municipio tiene el mismo perfil que sus vecinos, comparado con lo que pasaría por azar

In [ ]:
# Moran, LISA y cohesión por elección
t0 = time.time()
GEO_I = GEO.set_index('cvegeo')
W_ALL = Queen.from_dataframe(GEO_I, use_index=True, silence_warnings=True)
print(
    f"Matriz Queen: {W_ALL.n:,} municipios, {W_ALL.mean_neighbors:.1f} vecinos en promedio ({time.time() - t0:.0f} s)"
)
CLASE = {1: 'Alto-Alto', 2: 'Bajo-Alto', 3: 'Bajo-Bajo', 4: 'Alto-Bajo'}
ESPACIAL, datos_modelo['lisa_part'], datos_modelo['lisa_jov'] = [], 'Sin polígono', 'Sin polígono'
for a in ANIOS:
    s = datos_modelo[(datos_modelo.anio == a) & datos_modelo.cvegeo_inegi.notna()].drop_duplicates('cvegeo_inegi')
    ids = [i for i in s.cvegeo_inegi if i in W_ALL.neighbors]
    s = s.set_index('cvegeo_inegi').loc[ids]
    w = w_subset(W_ALL, ids, silence_warnings=True)
    w.transform = 'r'
    fila = {'anio': a, 'n': len(ids)}
    for var, col in [('part_rel', 'lisa_part'), ('brecha_jov', 'lisa_jov')]:
        if var not in s:
            continue
        y = s[var].fillna(s[var].median()).values
        mg = Moran(y, w, permutations=999)
        ml = Moran_Local(y, w, permutations=999, seed=SEMILLA)
        cl = np.where(ml.p_sim < 0.05, pd.Series(ml.q).map(CLASE).values, 'No significativo')
        mask = (datos_modelo.anio == a) & datos_modelo.cvegeo_inegi.isin(ids)
        datos_modelo.loc[mask, col] = datos_modelo.loc[mask, 'cvegeo_inegi'].map(pd.Series(cl, index=ids))
        fila[f'moran_{var}'], fila[f'p_{var}'] = mg.I, mg.p_sim
    # Cohesión: qué tan seguido un municipio comparte perfil con sus vecinos, comparado con el azar
    perfiles_vecinos = s.perfil.values
    pos = {c: i for i, c in enumerate(ids)}
    mismos = [
        np.mean([perfiles_vecinos[pos[j]] == perfiles_vecinos[pos[i]] for j in w.neighbors[i]])
        for i in ids
        if w.neighbors[i]
    ]
    esperado = (pd.Series(perfiles_vecinos).value_counts(normalize=True) ** 2).sum()
    fila['vecinos_mismo_perfil'], fila['esperado_azar'] = np.mean(mismos), esperado
    ESPACIAL.append(fila)
ESPACIAL = pd.DataFrame(ESPACIAL).set_index('anio')
ESPACIAL['cohesion_x'] = ESPACIAL.vecinos_mismo_perfil / ESPACIAL.esperado_azar
display(ESPACIAL.round(3))
print(
    f"\nEn {ULT}, un municipio comparte perfil con el {ESPACIAL.loc[ULT, 'vecinos_mismo_perfil']:.0%} de sus vecinos; "
    f"por azar se esperaría el {ESPACIAL.loc[ULT, 'esperado_azar']:.0%} ({ESPACIAL.loc[ULT, 'cohesion_x']:.1f} veces más)."
)

## 12. Vista por estados

Se repite el análisis con los 32 estados. Se suman los conteos de cada estado, se resume la información con PCA y se agrupan los estados con K-Means sobre los componentes que explican al menos el 80 % de la varianza. PCA y K-Means se ajustan con todas las elecciones juntas, para que los ejes y los grupos se puedan comparar entre años.

Al final se dibujan los perfiles municipales en un plano PCA. Ese PCA sólo sirve para visualizar; el K-Means municipal se ajusta sobre las variables estandarizadas.

In [ ]:
# Indicadores por estado y PCA
def EDO_CORTO(e):
    """Nombre corto del estado (sin «de Zaragoza», «de Ocampo», etc.)."""
    return re.sub(r' de (Zaragoza|Ocampo|Ignacio de la Llave)$', '', str(e))


cols_conteo = [c for c in datos_modelo.columns if re.match(r'^(LN|SV|NV|NS)(_|$)', c)]
datos_estados = datos_modelo.groupby(['anio', 'edo'])[cols_conteo].sum().reset_index()
datos_estados['estado'] = datos_estados.edo.map(ENTIDADES).map(EDO_CORTO)
datos_estados['part'] = 100 * datos_estados.SV / (datos_estados.SV + datos_estados.NV)
datos_estados['part_rel'] = datos_estados.part - datos_estados.anio.map(PART_NACIONAL)
for gk, sufs in GRUPOS.items():
    if sufs:
        datos_estados[f'part_{gk}'] = (
            100
            * sum(datos_estados[f'SV_{s}'] for s in sufs)
            / sum(datos_estados[f'SV_{s}'] + datos_estados[f'NV_{s}'] for s in sufs)
        )
        datos_estados[f'brecha_{gk}'] = datos_estados[f'part_{gk}'] - datos_estados.part
if GRUPOS['jov']:
    datos_estados['pct_ln_jov'] = 100 * sum(datos_estados[f'LN_{s}'] for s in GRUPOS['jov']) / datos_estados.LN
if any(lim_inf(e) >= 60 for e in EDADES):
    datos_estados['pct_ln_60'] = (
        100 * sum(datos_estados[f'LN_{e}'] for e in EDADES if lim_inf(e) >= 60) / datos_estados.LN
    )
if 'R' in TIPOS:
    datos_estados['pct_rural'] = 100 * datos_estados.LN_TS_R / datos_estados.LN
if 'U' in TIPOS:
    datos_estados['pct_urbano'] = 100 * datos_estados.LN_TS_U / datos_estados.LN
elif {'R', 'M'} <= set(TIPOS):
    datos_estados['pct_urbano'] = 100 - datos_estados.pct_rural - 100 * datos_estados.LN_TS_M / datos_estados.LN
datos_estados['log_ln'] = np.log10(datos_estados.LN)
if 'brecha_genero' in datos_modelo:
    brecha_genero_estado = (
        datos_modelo.dropna(subset=['brecha_genero'])
        .groupby(['anio', 'edo'])
        .apply(lambda d: np.average(d.brecha_genero, weights=d.LN))
    )
    datos_estados = datos_estados.merge(
        brecha_genero_estado.rename('brecha_genero').reset_index(), on=['anio', 'edo'], how='left'
    )
VARIABLES_ESTADOS = [
    f
    for f in VARIABLES_SEGMENTACION
    if f in datos_estados and datos_estados[f].notna().all() and datos_estados[f].std() > 0
]

estados_estandarizados = StandardScaler().fit_transform(datos_estados[VARIABLES_ESTADOS])
pca_estados = PCA().fit(estados_estandarizados)
comps, componentes_estados = pca_estados.components_.copy(), pca_estados.transform(estados_estandarizados)
for j in range(len(comps)):  # signo determinista: la carga más grande es positiva
    if comps[j][np.argmax(np.abs(comps[j]))] < 0:
        comps[j] *= -1
        componentes_estados[:, j] *= -1
varianza_estados = pca_estados.explained_variance_ratio_ * 100
n_componentes_80 = int(max(2, np.searchsorted(np.cumsum(varianza_estados), 80) + 1))
CARGAS_E = comps.T * np.sqrt(pca_estados.explained_variance_)  # correlación entre variable y componente
datos_estados['pc1'], datos_estados['pc2'] = componentes_estados[:, 0], componentes_estados[:, 1]

In [ ]:
# Agrupamiento de los estados
componentes_80 = componentes_estados[:, :n_componentes_80]
SIL_E = {
    k: silhouette_score(componentes_80, KMeans(k, n_init=20, random_state=SEMILLA).fit_predict(componentes_80))
    for k in range(3, 6)
}
num_segmentos = max(SIL_E, key=SIL_E.get)
kmeans_estados = KMeans(num_segmentos, n_init=50, random_state=SEMILLA).fit(componentes_80)
orden_e = pd.Series(datos_estados.groupby(kmeans_estados.labels_).part_rel.mean()).sort_values(ascending=False).index
datos_estados['segmento'] = pd.Series(kmeans_estados.labels_).map({o: n for n, o in enumerate(orden_e)}).values

# Nombre del segmento: los dos rasgos más distintivos, sin repetir familia (rural/urbano)
RASGO_E = {
    'part_rel': ('Alta participación', 'Baja participación'),
    'brecha_jov': ('jóvenes que votan', 'jóvenes alejados'),
    'brecha_mad': ('50–69 muy activos', '50–69 poco activos'),
    'brecha_may': ('mayores activos', 'mayores que dejan de votar'),
    'pct_ln_jov': ('padrón joven', 'padrón con pocos jóvenes'),
    'pct_ln_60': ('padrón envejecido', 'padrón con pocos mayores'),
    'pct_rural': ('más rural', 'menos rural'),
    'pct_urbano': ('más urbano', 'menos urbano'),
    'log_ln': ('entidades grandes', 'entidades pequeñas'),
    'brecha_genero': ('mujeres votan más', 'brecha de género menor'),
}
FAMILIA_V = {'pct_rural': 'terr', 'pct_urbano': 'terr', 'pct_ln_jov': 'edad', 'pct_ln_60': 'edad'}
medias_segmentos = (
    pd.DataFrame(estados_estandarizados, columns=VARIABLES_ESTADOS).groupby(datos_estados.segmento.values).mean()
)
NOMBRES_SEGMENTOS = {}
for s in range(num_segmentos):
    z = medias_segmentos.loc[s].sort_values(key=np.abs, ascending=False)
    usados, partes = set(), []
    for v, zv in z.items():
        fam = FAMILIA_V.get(v, v)
        if fam in usados:
            continue
        partes.append(RASGO_E.get(v, (v, v))[0 if zv > 0 else 1])
        usados.add(fam)
        if len(partes) == 2:
            break
    n_ = ' · '.join(partes)
    n_ = n_[0].upper() + n_[1:]
    while n_ in NOMBRES_SEGMENTOS.values():
        n_ += ' II'
    NOMBRES_SEGMENTOS[s] = n_
datos_estados['nombre_segmento'] = datos_estados.segmento.map(NOMBRES_SEGMENTOS)
COLORES_SEGMENTOS = {s: PALETA[s % len(PALETA)] for s in range(num_segmentos)}

In [ ]:
# LISA entre estados
GEO_EDO = GEO.assign(edo=GEO.cvegeo.str[:2].astype(int)).dissolve('edo').reset_index()[['edo', 'geometry']]
W_E = Queen.from_dataframe(GEO_EDO, use_index=False, silence_warnings=True)
W_E.transform = 'r'
datos_estados['lisa_pc1'] = 'No significativo'
MORAN_E = {}
for a in ANIOS:
    s = datos_estados[datos_estados.anio == a].set_index('edo').reindex(GEO_EDO.edo)
    ml = Moran_Local(s.pc1.values, W_E, permutations=999, seed=SEMILLA)
    MORAN_E[a] = Moran(s.pc1.values, W_E, permutations=999)
    cl = np.where(ml.p_sim < 0.05, pd.Series(ml.q).map(CLASE).values, 'No significativo')
    datos_estados.loc[datos_estados.anio == a, 'lisa_pc1'] = datos_estados.loc[datos_estados.anio == a, 'edo'].map(
        dict(zip(GEO_EDO.edo, cl))
    )


def desc_pc(j, n=2):
    idx = np.argsort(-np.abs(CARGAS_E[:, j]))[:n]
    return ' y '.join(
        f"{ETIQ.get(VARIABLES_ESTADOS[i], VARIABLES_ESTADOS[i]).split(' (')[0]} ({'+' if CARGAS_E[i, j] > 0 else '−'})"
        for i in idx
    )


print(
    f"PCA estatal: {len(VARIABLES_ESTADOS)} variables, {n_componentes_80} componentes explican al menos 80 %. PC1 {varianza_estados[0]:.0f}% ({desc_pc(0)}), PC2 {varianza_estados[1]:.0f}% ({desc_pc(1)})"
)
print(
    f"K-Means estatal: k = {num_segmentos} (silhouette {SIL_E[num_segmentos]:.2f}; candidatos {', '.join(f'{k}: {v:.2f}' for k, v in SIL_E.items())})"
)
print(
    f"I de Moran de PC1 entre estados, {ULT}: {MORAN_E[ULT].I:.2f} (p = {MORAN_E[ULT].p_sim:.3f}). Con 32 unidades la prueba tiene poca potencia."
)
display(
    datos_estados[datos_estados.anio == ULT]
    .groupby('nombre_segmento')
    .estado.apply(lambda s: ', '.join(sorted(s)))
    .to_frame('Estados en ' + str(ULT))
)

In [ ]:
# Gráficas de la vista estatal
def biplot_estados(a):
    s = datos_estados[datos_estados.anio == a]
    f = go.Figure()
    for g in range(num_segmentos):
        m = s[s.segmento == g]
        if len(m) >= 3:
            try:
                h = ConvexHull(m[['pc1', 'pc2']].values).vertices
                h = np.r_[h, h[0]]
                f.add_trace(
                    go.Scatter(
                        x=m.pc1.values[h],
                        y=m.pc2.values[h],
                        mode='lines',
                        fill='toself',
                        fillcolor=COLORES_SEGMENTOS[g] + '1f',
                        line=dict(color=COLORES_SEGMENTOS[g], width=1, dash='dot'),
                        hoverinfo='skip',
                        showlegend=False,
                    )
                )
            except Exception:
                pass
        f.add_trace(
            go.Scatter(
                x=m.pc1,
                y=m.pc2,
                mode='markers+text',
                name=NOMBRES_SEGMENTOS[g],
                text=m.estado,
                textposition='top center',
                textfont=dict(size=9),
                marker=dict(size=11, color=COLORES_SEGMENTOS[g], line=dict(color='white', width=1.5)),
            )
        )
    fac = np.abs(componentes_estados[:, :2]).max() * 0.8 / np.abs(CARGAS_E[:, :2]).max()
    for j in np.argsort(-np.hypot(CARGAS_E[:, 0], CARGAS_E[:, 1]))[:8]:
        f.add_annotation(
            x=CARGAS_E[j, 0] * fac,
            y=CARGAS_E[j, 1] * fac,
            ax=0,
            ay=0,
            xref='x',
            yref='y',
            axref='x',
            ayref='y',
            arrowhead=3,
            arrowwidth=1.5,
            arrowcolor='#52514e',
            text=ETIQ.get(VARIABLES_ESTADOS[j], VARIABLES_ESTADOS[j]).split(' (')[0],
            font=dict(size=10, color='#0b0b0b'),
            bgcolor='rgba(255,255,255,.85)',
        )
    f.update_layout(
        template='plotly_white',
        height=560,
        title=f'Estados en el plano PCA, {a} (color según el segmento K-Means)',
        xaxis_title=f'PC1 ({varianza_estados[0]:.0f}%)',
        yaxis_title=f'PC2 ({varianza_estados[1]:.0f}%)',
        legend=dict(orientation='h', y=-0.15),
    )
    return f


biplot_estados(ULT).show()

# PCA del nivel municipal: los mismos perfiles K-Means vistos en el plano de componentes
pca_municipios = PCA(2).fit(variables_estandarizadas)
componentes_municipios = pca_municipios.transform(variables_estandarizadas)
CARGAS_M = pca_municipios.components_.T * np.sqrt(pca_municipios.explained_variance_)
for j in range(2):
    if CARGAS_M[np.argmax(np.abs(CARGAS_M[:, j])), j] < 0:
        CARGAS_M[:, j] *= -1
        componentes_municipios[:, j] *= -1
varianza_municipios = pca_municipios.explained_variance_ratio_ * 100
datos_modelo['pcm1'], datos_modelo['pcm2'] = componentes_municipios[:, 0], componentes_municipios[:, 1]
print(
    f"PCA municipal (sólo para visualizar): PC1 {varianza_municipios[0]:.0f}% y PC2 {varianza_municipios[1]:.0f}% de la varianza"
)
datos_estados[
    ['anio', 'edo', 'estado', 'LN', 'SV', 'part', 'part_rel']
    + VARIABLES_ESTADOS
    + ['pc1', 'pc2', 'segmento', 'nombre_segmento', 'lisa_pc1']
].to_csv(f'{DIR_SALIDA}/resultados_segmentacion_estados.csv', index=False, encoding='utf-8-sig')

## 13. Historia de participación de 2009 a 2024

Se sigue el perfil de cada municipio en las seis elecciones, se cuentan los cambios de perfil entre elecciones y se clasifica el historial de cada municipio (en ascenso, en descenso, volátil, participación alta constante, abstención persistente o estable en el promedio).

In [ ]:
# Trayectorias y tipos de historial
TRAY = datos_modelo.pivot_table(index='cvegeo', columns='anio', values='perfil', aggfunc='first')
TRAY_N = TRAY.notna().sum(axis=1)
moda = TRAY.mode(axis=1)[0]
ESTABILIDAD = (TRAY.eq(moda, axis=0).sum(axis=1) / TRAY_N).rename('estabilidad')
print(
    f"Municipios que conservan su perfil en todas las elecciones: {(ESTABILIDAD == 1).mean():.0%}, "
    f"estabilidad media {ESTABILIDAD.mean():.0%}"
)

TRANS = {}
for a, b in zip(ANIOS[:-1], ANIOS[1:]):
    t = TRAY[[a, b]].dropna().astype(int)
    TRANS[(a, b)] = pd.crosstab(t[a], t[b]).reindex(
        index=range(num_perfiles), columns=range(num_perfiles), fill_value=0
    )
print(f"\nCambios de perfil de {ANIOS[-2]} a {ANIOS[-1]} (filas: perfil de origen, columnas: perfil de llegada):")
display(TRANS[(ANIOS[-2], ANIOS[-1])].rename(index=lambda p: f'P{p+1}', columns=lambda p: f'P{p+1}'))

# Tipología del historial (participación relativa al país, elección por elección)
PR = datos_modelo.pivot_table(index='cvegeo', columns='anio', values='part_rel')
xs = np.array(ANIOS, dtype=float)


def pendiente(row):
    m = row.notna().values
    return np.polyfit(xs[m], row.values[m], 1)[0] * 3 if m.sum() >= 3 else np.nan  # pp por elección (3 años)


HIST = pd.DataFrame({'media': PR.mean(axis=1), 'sd': PR.std(axis=1), 'tendencia': PR.apply(pendiente, axis=1)})
q_t, q_s = HIST.tendencia.quantile([0.1, 0.9]).values, HIST.sd.quantile(0.85)


def tipo_hist(r):
    if pd.isna(r.tendencia):
        return 'Historial incompleto'
    if r.tendencia >= max(q_t[1], 1):
        return 'En ascenso'
    if r.tendencia <= min(q_t[0], -1):
        return 'En descenso'
    if r.sd >= q_s:
        return 'Volátil'
    if r.media >= 3:
        return 'Participación alta constante'
    if r.media <= -3:
        return 'Abstención persistente'
    return 'Estable en el promedio'


HIST['historial'] = HIST.apply(tipo_hist, axis=1)
HIST['estabilidad'] = ESTABILIDAD
HIST['perfil_dominante'] = moda.astype(int)
display(pd.crosstab(HIST.historial, HIST.perfil_dominante.map(lambda p: f'P{p+1}')))

## 14. Estrategias y municipios prioritarios

Cada estrategia se activa cuando el perfil cumple una condición medible y trae un indicador para evaluarla. Un perfil recibe hasta tres estrategias, ordenadas por la magnitud del problema.

Los votos a recuperar son las personas que votarían si el municipio alcanzara el promedio nacional. Los jóvenes a recuperar son las personas de 18 a 29 años que votarían si participaran como el resto de su municipio.

In [ ]:
# Catálogo de estrategias y asignación a cada perfil
CATALOGO = [
    dict(
        clave='jov',
        titulo='Primer voto: juventud a las urnas',
        # Se activa cuando la brecha juvenil del perfil es más negativa que la del país
        cuando=lambda c: pais_ultima.get('brecha_jov', 0) - c.get('u_brecha_jov', 0),
        umbral=0.5,
        acciones=[
            'Jornadas en preparatorias y universidades con simulacros de votación',
            'Credencialización en planteles para quienes cumplen 18 años antes de la elección',
            'Contenido breve en redes sociales sobre cómo y dónde votar, con creadores locales',
        ],
        indicador='Brecha de participación 18–29 años vs. total del municipio',
    ),
    dict(
        clave='rur',
        titulo='Cercanía territorial en comunidades rurales',
        cuando=lambda c: (c.get('u_pct_ln_R', 0) - 40) / 10 - c.get('d_part', 0) / 3,
        umbral=0.5,
        acciones=[
            'Difusión por radio comunitaria y perifoneo, en lenguas originarias donde aplique',
            'Visitas casa por casa de capacitadores asistentes electorales para ubicar la casilla',
            'Revisión de la ubicación de casillas para reducir distancias y tiempos de traslado',
        ],
        indicador='Participación en secciones rurales',
    ),
    dict(
        clave='may',
        titulo='Inclusión de personas mayores',
        cuando=lambda c: -(c.get('u_brecha_may', 0) - pais_ultima.get('brecha_may', 0))
        + (c.get('u_pct_ln_60', 0) - pais_ultima.get('pct_ln_60', 0)) / 3,
        umbral=0.8,
        acciones=[
            'Casillas en planta baja con rampas, sillas y fila preferente',
            'Difusión del voto anticipado para personas con discapacidad o que no pueden salir de casa',
            'Redes de acompañamiento con organizaciones vecinales y de personas adultas mayores',
        ],
        indicador='Participación de personas de 70 años o más',
    ),
    dict(
        clave='urb',
        titulo='Ciudad conectada',
        cuando=lambda c: (c.get('u_pct_ln_U', 0) - 60) / 10 - c.get('d_part', 0) / 2,
        umbral=0.5,
        acciones=[
            'Campañas digitales geolocalizadas con la ubicación de la casilla de cada sección',
            'Alianzas con empleadores, transporte público y comercios para difundir la jornada',
            'Mensajes de recordatorio en la semana previa y el día de la elección',
        ],
        indicador='Participación en secciones urbanas',
    ),
    dict(
        clave='rea',
        titulo='Reactivación cívica',
        cuando=lambda c: -c.get('d_part', 0) / 2,
        umbral=1,
        acciones=[
            'Diagnóstico participativo con liderazgos locales sobre motivos de la abstención',
            'Foros y debates locales con candidaturas para conectar elección y vida cotidiana',
            'Seguimiento sección por sección para focalizar recursos',
        ],
        indicador='Participación total vs. promedio nacional',
    ),
    dict(
        clave='gen',
        titulo='Igualdad en la participación',
        cuando=lambda c: abs((c.get('u_brecha_genero', 0) or 0) - pais_ultima.get('brecha_genero', 0)),
        umbral=2,
        acciones=[
            'Mensajes diferenciados para el grupo que participa menos',
            'Horarios y espacios de difusión que respondan a cargas de cuidado',
        ],
        indicador='Brecha de participación mujeres − hombres',
    ),
    dict(
        clave='sem',
        titulo='Semillero cívico',
        cuando=lambda c: c.get('d_part', 0) / 2,
        umbral=1,
        acciones=[
            'Convocar a más personas como funcionarias de casilla y observadoras electorales',
            'Documentar y replicar en perfiles vecinos las prácticas que sostienen la participación',
            'Mantener la difusión para evitar retrocesos',
        ],
        indicador='Participación total y número de voluntarios electorales',
    ),
]
ESTRATEGIAS = {}
for p in range(num_perfiles):
    c = PERFILES.loc[p].to_dict()
    act = sorted([(e['cuando'](c), e) for e in CATALOGO if e['cuando'](c) >= e['umbral']], key=lambda x: -x[0])
    if len(act) < 2:
        act += [(0, e) for e in sorted(CATALOGO, key=lambda e: -e['cuando'](c)) if e not in [a[1] for a in act]][
            : 2 - len(act)
        ]
    ESTRATEGIAS[p] = [e for _, e in act[:3]]
    print(f"P{p + 1} {NOMBRES[p]}: " + '; '.join(e['titulo'] for e in ESTRATEGIAS[p]))

In [ ]:
# Municipios prioritarios y tablas de resultados
datos_ultima = datos_modelo[datos_modelo.anio == ULT].copy()
datos_ultima['votos_a_recuperar'] = (
    (datos_ultima.SV + datos_ultima.NV) * np.maximum(0, pais_ultima['part'] - datos_ultima.part) / 100
).round()
if 'part_jov' in datos_ultima:
    lnj = sum(datos_ultima[f'SV_{e}'] + datos_ultima[f'NV_{e}'] for e in GRUPOS['jov'])
    datos_ultima['jovenes_a_recuperar'] = (lnj * np.maximum(0, datos_ultima.part - datos_ultima.part_jov) / 100).round()
else:
    datos_ultima['jovenes_a_recuperar'] = 0
datos_ultima['estado'] = datos_ultima.edo.map(ENTIDADES)
datos_ultima['municipio'] = (
    datos_ultima.cvegeo_inegi.map(GEO.set_index('cvegeo').nom_mun)
    .fillna(datos_ultima.nombre)
    .fillna(datos_ultima.cvegeo)
)
PRIORIDAD = {}
for p in range(num_perfiles):
    s = datos_ultima[datos_ultima.perfil == p]
    clave = 'votos_a_recuperar' if PERFILES.loc[p, 'd_part'] < 0 else 'jovenes_a_recuperar'
    PRIORIDAD[p] = s.sort_values(clave, ascending=False).head(10)[
        ['municipio', 'estado', 'LN', 'part', 'votos_a_recuperar', 'jovenes_a_recuperar']
    ]
print(
    f"\nSi todos los municipios por debajo del promedio lo alcanzaran, votarían {datos_ultima.votos_a_recuperar.sum() / 1e6:.2f} millones de personas más."
)
display(PRIORIDAD[int(PERFILES.d_part.idxmin())].round(1))

# Tabla final por municipio
SALIDA = datos_ultima[
    ['cvegeo', 'cvegeo_inegi', 'municipio', 'estado', 'LN', 'SV', 'part', 'part_rel']
    + [c for c in VARIABLES_SEGMENTACION if c not in ('part_rel',)]
    + ['perfil', 'nombre_perfil', 'lisa_part', 'votos_a_recuperar', 'jovenes_a_recuperar']
].copy()
SALIDA['perfil'] = SALIDA.perfil + 1
SALIDA = SALIDA.merge(HIST[['historial', 'estabilidad', 'tendencia']], left_on='cvegeo', right_index=True, how='left')
SALIDA.to_csv(f'{DIR_SALIDA}/perfiles_municipales_{ULT}.csv', index=False, encoding='utf-8-sig')
datos_modelo.assign(perfil=datos_modelo.perfil + 1).drop(
    columns=[c for c in datos_modelo.columns if re.match(r'^(LN|SV|NV|NS)_', c)]
).to_csv(f'{DIR_SALIDA}/panel_municipio_eleccion.csv', index=False, encoding='utf-8-sig')
print(f"Guardado: perfiles_municipales_{ULT}.csv y panel_municipio_eleccion.csv")

## 15. Dashboard interactivo

Se genera un solo archivo HTML con menú lateral y nueve secciones, que son Inicio, Resumen, Perfiles, Mapa, Trayectorias y región, Estados (PCA y K-Means), Estrategias, Buscador municipal y Modelo. Antes de armarlo se ajusta la redacción de los textos (nombres de variables, perfiles y estrategias, descripciones y hallazgos) a un registro técnico. Las gráficas se dibujan en el navegador, por lo que el archivo funciona sin conexión; solo la tipografía Poppins se descarga si hay internet.

In [ ]:
# Datos del dashboard
def r2(v, d=2):
    return None if v is None or (isinstance(v, float) and not np.isfinite(v)) else round(float(v), d)


ids_geo = sorted(set(datos_modelo.cvegeo_inegi.dropna()))
pos = {c: i for i, c in enumerate(ids_geo)}
nom = GEO.set_index('cvegeo').nom_mun
POR_ANIO = {}
for a in ANIOS:
    s = (
        datos_modelo[(datos_modelo.anio == a) & datos_modelo.cvegeo_inegi.notna()]
        .drop_duplicates('cvegeo_inegi')
        .set_index('cvegeo_inegi')
        .reindex(ids_geo)
    )
    POR_ANIO[str(a)] = {
        'perfil': [None if pd.isna(v) else int(v) for v in s.perfil],
        'part': [r2(v, 1) for v in s.part],
        'rel': [r2(v, 1) for v in s.part_rel],
        'jov': [r2(v, 1) for v in s.get('brecha_jov', pd.Series(np.nan, index=s.index))],
        'lisa': [None if pd.isna(v) else v for v in s.lisa_part],
        'ln': [None if pd.isna(v) else int(v) for v in s.LN],
    }
historial_mun = (
    HIST.reset_index()
    .merge(datos_modelo[['cvegeo', 'cvegeo_inegi']].drop_duplicates('cvegeo'), on='cvegeo')
    .drop_duplicates('cvegeo_inegi')
    .set_index('cvegeo_inegi')
    .reindex(ids_geo)
)
MUNS = [
    [nom.get(c, c), ENTIDADES.get(int(c[:2]), ''), None if pd.isna(h) else h]
    for c, h in zip(ids_geo, historial_mun.historial)
]


def perfil_js(p):
    c = PERFILES.loc[p]
    return dict(
        id=p,
        nombre=NOMBRES[p],
        corto=CORTOS[p],
        color=COLORES[p],
        narrativa=NARRATIVAS[p],
        n_mun=int(c.n_mun_ult),
        ln=r2(c.LN_ult / 1e6, 2),
        pct_ln=r2(c.pct_LN_ult, 1),
        d_part=r2(c.d_part, 1),
        brecha_jov=r2(c.get('u_brecha_jov'), 1),
        brecha_may=r2(c.get('u_brecha_may'), 1),
        rural=r2(c.get('u_pct_ln_R'), 0),
        urbano=r2(c.get('u_pct_ln_U'), 0),
        genero=r2(c.get('u_brecha_genero'), 1),
        curva=[r2(c.get(f'u_part_{e}'), 1) for e in EDADES],
        estrategias=[dict(t=e['titulo'], a=e['acciones'], k=e['indicador']) for e in ESTRATEGIAS[p]],
        regla=(REGLAS[p][0] + f" ({REGLAS[p][1]:.0%} de acierto)") if p in REGLAS else '',
        prioridad=[
            [
                r.municipio,
                EDO_CORTO(r.estado),
                int(r.LN),
                r2(r.part, 1),
                int(r.votos_a_recuperar),
                int(r.jovenes_a_recuperar),
            ]
            for r in PRIORIDAD[p].itertuples()
        ],
        historial={k: int(v) for k, v in HIST[HIST.perfil_dominante == p].historial.value_counts().items()},
    )


nodos, links = [], dict(source=[], target=[], value=[], color=[])
for i, a in enumerate(ANIOS):
    for p in range(num_perfiles):
        nodos.append(dict(label=f"P{p + 1} ({a})", color=COLORES[p]))
for i, (a, b) in enumerate(zip(ANIOS[:-1], ANIOS[1:])):
    m = TRANS[(a, b)]
    for p in range(num_perfiles):
        for q in range(num_perfiles):
            if m.loc[p, q] > 0:
                links['source'].append(i * num_perfiles + p)
                links['target'].append((i + 1) * num_perfiles + q)
                links['value'].append(int(m.loc[p, q]))
                links['color'].append(COLORES[p] + '55')

reparto = datos_modelo.pivot_table(index='anio', columns='perfil', values='LN', aggfunc='sum')
reparto = reparto.div(reparto.sum(axis=1), axis=0) * 100
p_hi, p_lo = int(PERFILES.d_part.idxmax()), int(PERFILES.d_part.idxmin())
hallazgos = [
    f"Se identificaron <b>{num_perfiles} perfiles</b> en {len(datos_modelo):,} observaciones municipio-elección ({datos_modelo.cvegeo.nunique():,} municipios, {len(ANIOS)} elecciones federales).",
    f"El perfil más participativo, <b>{NOMBRES[p_hi]}</b>, vota {PERFILES.d_part[p_hi]:+.1f} pp respecto al país; el menos participativo, <b>{NOMBRES[p_lo]}</b>, {PERFILES.d_part[p_lo]:+.1f} pp.",
    f"En {ULT}, la participación de 18–29 años fue {pais_ultima.get('part_jov', np.nan):.1f} %, {abs(pais_ultima.get('brecha_jov', 0)):.1f} pp por debajo del total nacional ({pais_ultima['part']:.1f} %).",
    f"Los perfiles forman regiones: un municipio comparte perfil con el {ESPACIAL.loc[ULT, 'vecinos_mismo_perfil']:.0%} de sus vecinos ({ESPACIAL.loc[ULT, 'cohesion_x']:.1f}× lo esperado al azar); I de Moran de la participación = {ESPACIAL.loc[ULT, 'moran_part_rel']:.2f}.",
    f"El {(ESTABILIDAD == 1).mean():.0%} de los municipios mantuvo el mismo perfil en todas las elecciones; el resto muestra trayectorias que permiten anticipar cambios.",
    f"Si cada municipio por debajo del promedio nacional lo alcanzara, votarían <b>{datos_ultima.votos_a_recuperar.sum() / 1e6:.2f} millones</b> de personas más.",
]
DATA = dict(
    equipo=EQUIPO,
    anios=[int(a) for a in ANIOS],
    ult=int(ULT),
    K=num_perfiles,
    ids=ids_geo,
    muns=MUNS,
    por_anio=POR_ANIO,
    perfiles=[perfil_js(p) for p in range(num_perfiles)],
    edades=[ETQ_EDAD[e] for e in EDADES],
    nac=dict(
        part={str(a): r2(INDICADORES_PAIS[a]['part'], 1) for a in ANIOS},
        curvas={str(a): [r2(v, 1) for v in CURVA_NAC[a]] for a in ANIOS},
        ts={ETQ_TS.get(t, t): [r2(TS_NAC.loc[t, a], 1) for a in ANIOS] for t in TIPOS} if TS_NAC is not None else {},
        curva_ult=[r2(pais_ultima[f'part_{e}'], 1) for e in EDADES],
        tipo={str(a): ELECCION_TIPO.get(a, '') for a in ANIOS},
        ln_ult=r2(pais_ultima['LN'] / 1e6, 1),
    ),
    reparto={str(a): [r2(reparto.loc[a, p], 1) for p in range(num_perfiles)] for a in ANIOS},
    sankey=dict(nodos=nodos, links=links),
    hallazgos=hallazgos,
    modelo=dict(
        k=[int(k) for k in METRICAS.index],
        sil=[r2(v, 3) for v in METRICAS.silhouette],
        est=[r2(v, 3) for v in METRICAS.estabilidad],
        db=[r2(v, 2) for v in METRICAS.davies],
        ari_gmm=[r2(v, 2) for v in METRICAS.ari_kmeans_gmm],
        ari_ward=[r2(v, 2) for v in METRICAS.ari_kmeans_ward],
        import_=[[ETIQ.get(k, k), r2(v, 3)] for k, v in IMPORT.items()],
        fidelidad=r2(FIDELIDAD * 100, 1),
        feats=[ETIQ.get(f, f) for f in VARIABLES_SEGMENTACION],
        n=len(datos_modelo),
        rango=list(K_RANGO),
    ),
    espacial=dict(
        moran_part=[r2(ESPACIAL.loc[a, 'moran_part_rel'], 3) for a in ANIOS],
        moran_jov=[
            r2(ESPACIAL.loc[a, 'moran_brecha_jov'], 3) if 'moran_brecha_jov' in ESPACIAL else None for a in ANIOS
        ],
        cohesion=[r2(ESPACIAL.loc[a, 'cohesion_x'], 2) for a in ANIOS],
    ),
)

# Datos para las secciones Inicio y Estrategias
DESCRIPCION_VARIABLES = {
    'part_rel': (
        'Cuántos puntos más o menos vota el municipio que el país en la misma elección.',
        'Si el municipio votó 65 % y el país 60 %, el valor es +5 pp.',
    ),
    'brecha_jov': (
        'Cuántos puntos más o menos votan las personas de 18 a 29 años que el total de su municipio.',
        'Si los jóvenes votaron 45 % y el municipio 55 %, la brecha es -10 pp.',
    ),
    'brecha_adu': (
        'Cuántos puntos más o menos votan las personas de 30 a 49 años que el total de su municipio.',
        'Un valor positivo indica que ese grupo vota más que el promedio de su municipio.',
    ),
    'brecha_mad': (
        'Cuántos puntos más o menos votan las personas de 50 a 69 años que el total de su municipio.',
        'Un valor positivo indica que ese grupo vota más que el promedio de su municipio.',
    ),
    'brecha_may': (
        'Cuántos puntos más o menos votan las personas de 70 años o más que el total de su municipio.',
        'Un valor negativo indica que las personas mayores dejan de votar.',
    ),
    'pct_ln_jov': (
        'Qué porcentaje de la lista nominal tiene de 18 a 29 años.',
        'Un valor alto indica un padrón joven.',
    ),
    'pct_ln_60': (
        'Qué porcentaje de la lista nominal tiene 60 años o más.',
        'Un valor alto indica un padrón envejecido.',
    ),
    'pct_rural': (
        'Qué porcentaje de la lista nominal vive en secciones rurales.',
        'Cerca de 100 % es un municipio casi totalmente rural.',
    ),
    'pct_urbano': (
        'Qué porcentaje de la lista nominal vive en secciones urbanas.',
        'Cerca de 100 % es un municipio totalmente urbano.',
    ),
    'log_ln': (
        'Tamaño del municipio, medido como el logaritmo base 10 de su lista nominal.',
        'Un valor de 3 equivale a unas 1,000 personas y uno de 6 a un millón.',
    ),
    'brecha_genero': (
        'Cuántos puntos más votan las mujeres que los hombres en el municipio.',
        'Un valor de +8 pp indica que las mujeres votaron 8 puntos más.',
    ),
}
DATA['variables'] = [[ETIQ.get(v, v), *DESCRIPCION_VARIABLES.get(v, ('', ''))] for v in VARIABLES_SEGMENTACION]
DATA['validacion'] = [[int(a), r2(val.loc[a].iloc[0], 1), r2(REF_INE.get(a), 1)] for a in val.index]

simulador = []
for p in range(num_perfiles):
    s = datos_ultima[datos_ultima.perfil == p]
    jov_votaron = sum(s[f'SV_{e}'].sum() for e in GRUPOS['jov'])
    jov_base = sum(s[f'SV_{e}'].sum() + s[f'NV_{e}'].sum() for e in GRUPOS['jov'])
    simulador.append(
        dict(base=int(s.SV.sum() + s.NV.sum()), sv=int(s.SV.sum()), jb=int(jov_base), jsv=int(jov_votaron))
    )
DATA['simulador'] = simulador
DATA['pais_jov'] = r2(pais_ultima.get('brecha_jov'), 1)
DATA['catalogo'] = [
    [e['titulo'], e['indicador'], [p for p in range(num_perfiles) if any(e is x for x in ESTRATEGIAS[p])]]
    for e in CATALOGO
]

In [ ]:
# Datos de la vista estatal
def _hull(m):
    try:
        h = ConvexHull(m[['pc1', 'pc2']].values).vertices
        h = np.r_[h, h[0]]
        return [m.pc1.values[h].round(3).tolist(), m.pc2.values[h].round(3).tolist()]
    except Exception:
        return None


fac_e = np.abs(componentes_estados[:, :2]).max() * 0.8 / np.abs(CARGAS_E[:, :2]).max()
top_e = np.argsort(-np.hypot(CARGAS_E[:, 0], CARGAS_E[:, 1]))[:8]
municipios_ultima = datos_modelo[datos_modelo.anio == ULT]
fac_m = np.abs(componentes_municipios).max() * 0.6 / np.abs(CARGAS_M).max()
top_m = np.argsort(-np.hypot(CARGAS_M[:, 0], CARGAS_M[:, 1]))[:8]
DATA['estados'] = dict(
    k=num_segmentos,
    nombres=[NOMBRES_SEGMENTOS[g] for g in range(num_segmentos)],
    colores=[COLORES_SEGMENTOS[g] for g in range(num_segmentos)],
    ve=[r2(v, 1) for v in varianza_estados[:10]],
    cum=[r2(v, 1) for v in np.cumsum(varianza_estados)[:10]],
    n80=n_componentes_80,
    sil=r2(SIL_E[num_segmentos], 2),
    feats=[ETIQ.get(f, f) for f in VARIABLES_ESTADOS],
    cargas=np.round(CARGAS_E[:, :3], 2).tolist(),
    medz=np.round(medias_segmentos.values, 2).tolist(),
    flechas=[
        [
            ETIQ.get(VARIABLES_ESTADOS[j], VARIABLES_ESTADOS[j]).split(' (')[0],
            r2(CARGAS_E[j, 0] * fac_e, 3),
            r2(CARGAS_E[j, 1] * fac_e, 3),
        ]
        for j in top_e
    ],
    pc_desc=[desc_pc(0), desc_pc(1)],
    anios={
        str(a): dict(
            edo=[f"{e:02d}" for e in s_.edo],
            estado=s_.estado.tolist(),
            pc1=s_.pc1.round(3).tolist(),
            pc2=s_.pc2.round(3).tolist(),
            seg=s_.segmento.astype(int).tolist(),
            lisa=s_.lisa_pc1.tolist(),
            part=s_.part.round(1).tolist(),
            rel=s_.part_rel.round(1).tolist(),
            hull=[_hull(s_[s_.segmento == g]) for g in range(num_segmentos)],
            moran=r2(MORAN_E[a].I, 2),
            p=r2(MORAN_E[a].p_sim, 3),
        )
        for a in ANIOS
        for s_ in [datos_estados[datos_estados.anio == a].sort_values('edo')]
    },
    mun=dict(
        x=municipios_ultima.pcm1.round(2).tolist(),
        y=municipios_ultima.pcm2.round(2).tolist(),
        perfil=municipios_ultima.perfil.astype(int).tolist(),
        nombre=(
            municipios_ultima.cvegeo_inegi.map(nom).fillna(municipios_ultima.nombre).fillna(municipios_ultima.cvegeo)
            + ', '
            + municipios_ultima.edo.map(ENTIDADES).map(EDO_CORTO)
        ).tolist(),
        ve=[r2(v, 1) for v in varianza_municipios],
        flechas=[
            [
                ETIQ.get(VARIABLES_SEGMENTACION[j], VARIABLES_SEGMENTACION[j]).split(' (')[0],
                r2(CARGAS_M[j, 0] * fac_m, 3),
                r2(CARGAS_M[j, 1] * fac_m, 3),
            ]
            for j in top_m
        ],
    ),
)
DATA['geo_ids'] = GEO_SIMPLE.cvegeo.tolist()
_ge = GEO_EDO.copy()
_ge['edo'] = _ge.edo.map('{:02d}'.format)
_ge['geometry'] = _ge.geometry.simplify(0.02, preserve_topology=True)
geo_edo_json = re.sub(r'(\d+\.\d{3})\d+', r'\1', _ge.to_json(drop_id=True)).replace('</', '<\\/')
datos_json = json.dumps(DATA, ensure_ascii=False, separators=(',', ':')).replace('</', '<\\/')
geo_json = GEOJSON_TXT.replace('</', '<\\/')

In [ ]:
# Redacción técnica de los textos del dashboard
# Ajusta etiquetas, nombres de perfiles y estrategias, descripciones, reglas y hallazgos.
import re

# Etiqueta completa -> (etiqueta nueva, etiqueta corta nueva, unidad)
ETIQUETAS = {
    'Participación vs. nacional (pp)': ('Participación relativa (pp)', 'Participación relativa', 'pp'),
    'Brecha jóvenes 18–29 (pp)': ('Brecha de participación 18–29 años (pp)', 'Brecha 18–29 años', 'pp'),
    'Brecha adultos 30–49 (pp)': ('Brecha de participación 30–49 años (pp)', 'Brecha 30–49 años', 'pp'),
    'Brecha 50–69 (pp)': ('Brecha de participación 50–69 años (pp)', 'Brecha 50–69 años', 'pp'),
    'Brecha 70+ (pp)': ('Brecha de participación 70 años o más (pp)', 'Brecha 70 años o más', 'pp'),
    '% de LN con 18–29 años': ('Proporción de LN de 18–29 años (%)', 'Proporción de LN de 18–29 años', '%'),
    '% de LN con 60+ años': ('Proporción de LN de 60 años o más (%)', 'Proporción de LN de 60 años o más', '%'),
    '% de LN en secciones rurales': ('Proporción de LN en secciones rurales (%)', 'Proporción de LN en secciones rurales', '%'),
    '% de LN en secciones urbanas': ('Proporción de LN en secciones urbanas (%)', 'Proporción de LN en secciones urbanas', '%'),
    'Tamaño (log10 LN)': ('Tamaño del municipio (log₁₀ LN)', 'Tamaño del municipio', ''),
    'Brecha mujeres − hombres (pp)': ('Brecha de género, mujeres − hombres (pp)', 'Brecha de género', 'pp'),
}
CORTAS = {k.split(' (')[0] if not k.startswith('%') else k: v for k, v in ETIQUETAS.items()}

DESCRIPCIONES = {
    'Participación relativa (pp)': ('Diferencia, en puntos porcentuales, entre la participación del municipio y la participación nacional en la misma elección.',
                                    'Ejemplo: con 65 % en el municipio y 60 % a nivel nacional, el valor es +5 pp.'),
    'Brecha de participación 18–29 años (pp)': ('Diferencia entre la participación del grupo de 18 a 29 años y la participación total del municipio.',
                                                'Ejemplo: 45 % en el grupo de 18 a 29 años y 55 % en el total municipal equivalen a −10 pp.'),
    'Brecha de participación 30–49 años (pp)': ('Diferencia entre la participación del grupo de 30 a 49 años y la participación total del municipio.',
                                                'Un valor positivo indica una participación superior al total municipal.'),
    'Brecha de participación 50–69 años (pp)': ('Diferencia entre la participación del grupo de 50 a 69 años y la participación total del municipio.',
                                                'Un valor positivo indica una participación superior al total municipal.'),
    'Brecha de participación 70 años o más (pp)': ('Diferencia entre la participación de las personas de 70 años o más y la participación total del municipio.',
                                                   'Un valor negativo indica una participación inferior al total municipal.'),
    'Proporción de LN de 18–29 años (%)': ('Porcentaje de la lista nominal con edad de 18 a 29 años.', 'Valores altos indican un padrón joven.'),
    'Proporción de LN de 60 años o más (%)': ('Porcentaje de la lista nominal con 60 años o más.', 'Valores altos indican un padrón envejecido.'),
    'Proporción de LN en secciones rurales (%)': ('Porcentaje de la lista nominal que reside en secciones electorales rurales.',
                                                  'Valores cercanos a 100 % corresponden a municipios predominantemente rurales.'),
    'Proporción de LN en secciones urbanas (%)': ('Porcentaje de la lista nominal que reside en secciones electorales urbanas.',
                                                  'Valores cercanos a 100 % corresponden a municipios predominantemente urbanos.'),
    'Tamaño del municipio (log₁₀ LN)': ('Logaritmo base 10 de la lista nominal del municipio; la escala logarítmica reduce la asimetría entre municipios pequeños y grandes.',
                                        'Un valor de 3 equivale a unas 1,000 personas y uno de 6, a un millón.'),
    'Brecha de género, mujeres − hombres (pp)': ('Diferencia entre la participación de las mujeres y la de los hombres en el municipio.',
                                                 'Un valor de +8 pp indica que las mujeres participaron 8 puntos más que los hombres.'),
}

# Frases del nombre automático de los perfiles
FRASES_PERFIL = [
    (r'muy participativ[oa]s', 'de participación muy alta'),
    (r'participativ[oa]s', 'de participación alta'),
    (r'con abstención alta', 'de participación muy baja'),
    (r'con juventud alejada de las urnas', 'con brecha juvenil amplia'),
    (r'con jóvenes que sí votan', 'con brecha juvenil reducida'),
    (r'donde las personas mayores dejan de votar', 'con baja participación de personas mayores'),
    (r'con personas mayores muy activas', 'con alta participación de personas mayores'),
    (r'donde las mujeres votan más', 'con brecha de género a favor de las mujeres'),
    (r'con brecha de género a favor de hombres', 'con brecha de género a favor de los hombres'),
    (r'(?<=\S) y juventud alejada de las urnas', ' y brecha juvenil amplia'),
    (r'(?<=\S) y jóvenes que sí votan', ' y brecha juvenil reducida'),
]
# Rasgos de los segmentos estatales
FRASES_SEGMENTO = {
    'jóvenes que votan': 'brecha juvenil reducida', 'jóvenes alejados': 'brecha juvenil amplia',
    '50–69 muy activos': 'alta participación de 50–69 años', '50–69 poco activos': 'baja participación de 50–69 años',
    'mayores que dejan de votar': 'baja participación de 70 años o más', 'mayores activos': 'alta participación de 70 años o más',
    'padrón con pocos jóvenes': 'baja proporción de jóvenes', 'padrón con pocos mayores': 'baja proporción de personas mayores',
    'más rural': 'mayor ruralidad', 'menos rural': 'menor ruralidad', 'más urbano': 'mayor urbanización', 'menos urbano': 'menor urbanización',
    'mujeres votan más': 'brecha de género a favor de las mujeres',
}
ACCIONES = {
    'Contenido breve en redes sociales sobre cómo y dónde votar, con creadores locales':
        'Contenidos breves en redes sociales sobre cómo y dónde votar, en colaboración con creadores locales',
    'Convocar a más personas como funcionarias de casilla y observadoras electorales':
        'Convocatoria a más personas para integrar mesas directivas de casilla y participar en la observación electoral',
    'Documentar y replicar en perfiles vecinos las prácticas que sostienen la participación':
        'Documentación de las prácticas que sostienen la participación para replicarlas en perfiles vecinos',
    'Mantener la difusión para evitar retrocesos': 'Continuidad de la difusión para evitar retrocesos',
    'Visitas casa por casa de capacitadores asistentes electorales para ubicar la casilla':
        'Visitas domiciliarias de capacitadores asistentes electorales para informar la ubicación de la casilla',
}
INDICADORES = {
    'Brecha de participación 18–29 años vs. total del municipio': 'Brecha de participación de 18 a 29 años respecto al total municipal',
    'Participación total vs. promedio nacional': 'Participación total respecto al promedio nacional',
    'Brecha de participación mujeres − hombres': 'Brecha de participación entre mujeres y hombres',
    'Participación total y número de voluntarios electorales': 'Participación total y número de personas voluntarias en la organización electoral',
}
ACCIONES_CATALOGO = {
    'Cercanía territorial en comunidades rurales': [
        'Difusión por radio comunitaria y perifoneo, en lenguas originarias donde aplique',
        'Visitas domiciliarias de capacitadores asistentes electorales para informar la ubicación de la casilla',
        'Revisión de la ubicación de casillas para reducir distancias y tiempos de traslado',
    ],
}
NOMBRES_EST = {
    'Primer voto: juventud a las urnas': 'Promoción del voto juvenil',
    'Cercanía territorial en comunidades rurales': 'Acceso a la casilla en zonas rurales',
    'Inclusión de personas mayores': 'Accesibilidad para personas mayores',
    'Ciudad conectada': 'Difusión digital en zonas urbanas',
    'Reactivación cívica': 'Atención a la abstención',
    'Igualdad en la participación': 'Reducción de la brecha de género',
    'Semillero cívico': 'Consolidación de la participación',
}
QUE_ES = {
    'Promoción del voto juvenil': 'Acciones en preparatorias, universidades y redes sociales para que las personas de 18 a 29 años tramiten su credencial y acudan a votar.',
    'Acceso a la casilla en zonas rurales': 'Difusión por radio comunitaria y revisión de la ubicación de casillas para reducir distancias y tiempos de traslado en secciones rurales.',
    'Accesibilidad para personas mayores': 'Casillas accesibles, difusión del voto anticipado y redes de acompañamiento para que las personas mayores puedan votar.',
    'Difusión digital en zonas urbanas': 'Campañas digitales con la ubicación de la casilla y alianzas con empresas y transporte público para recordar la jornada electoral.',
    'Atención a la abstención': 'Diagnóstico con actores locales sobre las causas de la abstención, foros con candidaturas y seguimiento por sección electoral.',
    'Reducción de la brecha de género': 'Mensajes y horarios de difusión dirigidos al grupo, mujeres u hombres, que participa menos, considerando las cargas de cuidado.',
    'Consolidación de la participación': 'Mantener la participación alta, reclutar funcionariado de casilla y observación electoral, y replicar las prácticas del perfil en municipios vecinos.',
}


def por_que(nombre, p, nac_j, nac_g, nac_m):
    d, sobre = p.get('d_part') or 0, lambda v: 'por debajo' if v < 0 else 'por encima'
    if nombre == 'Promoción del voto juvenil':
        return (f"El grupo de 18 a 29 años vota {abs(p['brecha_jov']):.1f} pp {'menos' if p['brecha_jov'] < 0 else 'más'} que el total de su municipio; "
                f"a nivel nacional la diferencia es de {abs(nac_j):.1f} pp.")
    if nombre == 'Acceso a la casilla en zonas rurales':
        return f"El {p.get('rural', 0):.0f} % de su lista nominal vive en secciones rurales."
    if nombre == 'Accesibilidad para personas mayores':
        if nac_m is not None and p.get('brecha_may') is not None and p['brecha_may'] < nac_m:
            return f"Las personas de 70 años o más presentan una brecha de {_sg(p['brecha_may'])} pp, menor a la nacional ({_sg(nac_m)} pp)."
        return 'Una proporción alta de su lista nominal tiene 60 años o más.'
    if nombre == 'Difusión digital en zonas urbanas':
        return f"El {p.get('urbano', 0):.0f} % de su lista nominal vive en secciones urbanas y su participación está {abs(d):.1f} pp {sobre(d)} del promedio nacional."
    if nombre == 'Atención a la abstención':
        return f"Su participación se ubica {abs(d):.1f} pp {sobre(d)} del promedio nacional, considerando todas las elecciones analizadas."
    if nombre == 'Reducción de la brecha de género':
        g = p.get('genero') or 0
        return (f"Las mujeres votan {abs(g):.1f} pp {'más' if g > 0 else 'menos'} que los hombres"
                + (f", frente a {nac_g:.1f} pp a nivel nacional." if nac_g is not None else '.'))
    if nombre == 'Consolidación de la participación':
        return f"Su participación está {abs(d):.1f} pp {sobre(d)} del promedio nacional, por lo que conviene sostenerla y replicar sus prácticas."
    return ''


NUM = {2: 'dos', 3: 'tres', 4: 'cuatro', 5: 'cinco', 6: 'seis', 7: 'siete', 8: 'ocho'}
MENOS = '−'


def _sg(v, d=1):
    return ('+' if v > 0 else MENOS if v < 0 else '') + f'{abs(v):.{d}f}'


def etiqueta(e):
    return ETIQUETAS.get(e, (e,))[0]


def corta(e):
    if e in CORTAS:
        return CORTAS[e][1]
    return ETIQUETAS[e][1] if e in ETIQUETAS else e


def nombre_perfil(n):
    for a, b in FRASES_PERFIL:
        n = re.sub(a, b, n)
    return n


def regla(r):
    if not r:
        return r
    m = re.match(r'^Si (.*?)\s*\((\d+)% de acierto\)\s*$', r)
    cuerpo, acierto = (m.group(1), m.group(2)) if m else (r.replace('Si ', '', 1), None)
    conds = []
    for c in re.split(r'\s+y\s+(?=%|[A-ZÁÉÍÓÚ])', cuerpo):
        mm = re.match(r'^(.*?)\s*(≤|>|<|≥)\s*(-?[\d.]+)$', c.strip())
        if not mm:
            conds.append(c)
            continue
        lab, op, val = mm.groups()
        unidad = CORTAS.get(lab, ETIQUETAS.get(lab, (lab, lab, '')))[2]
        v = float(val)
        conds.append(f"{corta(lab)} {op} {(MENOS if v < 0 else '')}{abs(v):.1f}{(' ' + unidad) if unidad else ''}")
    return 'Si ' + ' y '.join(conds) + (f' ({acierto} % de acierto)' if acierto else '')


def narrativa(p, D, genero_pais):
    n_el = len(D['anios'])
    t = (f"Agrupa {p['n_mun']:,} municipios en {D['ult']}, con {p['ln']:.1f} millones de personas en la lista nominal "
         f"({p['pct_ln']:.1f} % del total nacional). En el promedio de las {NUM.get(n_el, n_el)} elecciones, su participación se ubica "
         f"{abs(p['d_part']):.1f} pp {'por encima' if p['d_part'] >= 0 else 'por debajo'} del promedio nacional. ")
    if p.get('brecha_jov') is not None:
        t += (f"La participación del grupo de 18 a 29 años es {abs(p['brecha_jov']):.1f} pp "
              f"{'inferior' if p['brecha_jov'] < 0 else 'superior'} a la del total municipal, frente a una brecha nacional de {_sg(D['pais_jov'])} pp")
        t += (f", y la del grupo de 70 años o más presenta una diferencia de {_sg(p['brecha_may'])} pp. " if p.get('brecha_may') is not None else '. ')
    if p.get('rural') is not None:
        t += f"El {p['rural']:.0f} % de su lista nominal reside en secciones rurales" + (
            f" y el {p['urbano']:.0f} % en secciones urbanas. " if p.get('urbano') is not None else '. ')
    if p.get('genero') is not None:
        t += (f"Las mujeres participan {abs(p['genero']):.1f} pp {'más' if p['genero'] > 0 else 'menos'} que los hombres"
              + (f" ({_sg(genero_pais)} pp a nivel nacional)." if genero_pais is not None else '.'))
    return t.strip()


def hallazgos(D, viejos):
    P = D['perfiles']
    num = lambda pat, s, g=1: (re.search(pat, s) or [None, None])[g] if s else None
    h = []
    v0 = viejos[0] if viejos else ''
    m = re.search(r'en ([\d,]+) observaciones municipio-elección \(([\d,]+) municipios, (\d+) elecciones', v0)
    if m:
        h.append(f"Se identificaron <b>{D['K']} perfiles</b> a partir de {m.group(1)} observaciones municipio-elección "
                 f"({m.group(2)} municipios en {m.group(3)} elecciones federales).")
    hi = max(range(len(P)), key=lambda i: P[i]['d_part'])
    lo = min(range(len(P)), key=lambda i: P[i]['d_part'])
    h.append(f"El perfil de mayor participación, <b>{P[hi]['corto'].replace(' · ', ' ')}</b>, se ubica {abs(P[hi]['d_part']):.1f} pp "
             f"{'por encima' if P[hi]['d_part'] >= 0 else 'por debajo'} del promedio nacional, mientras que el de menor participación, "
             f"<b>{P[lo]['corto'].replace(' · ', ' ')}</b>, se ubica {abs(P[lo]['d_part']):.1f} pp {'por encima' if P[lo]['d_part'] >= 0 else 'por debajo'}.")
    for v in viejos[2:]:
        m = re.search(r'En (\d{4}), la participación de 18–29 años fue ([\d.]+) %, ([\d.]+) pp por debajo del total nacional \(([\d.]+) %\)', v)
        if m:
            h.append(f"En {m.group(1)}, la participación del grupo de 18 a 29 años fue de {m.group(2)} %, {m.group(3)} pp por debajo del total nacional ({m.group(4)} %).")
            continue
        m = re.search(r'con el (\d+)% de sus vecinos \(([\d.]+)× lo esperado al azar\); I de Moran de la participación = ([\-−]?\d+(?:\.\d+)?)', v)
        if m:
            h.append(f"Los perfiles presentan concentración territorial, ya que, en promedio, un municipio comparte perfil con el {m.group(1)} % "
                     f"de sus vecinos ({m.group(2)} veces lo esperado al azar), y el I de Moran de la participación es de {m.group(3)}.")
            continue
        m = re.search(r'El (\d+)% de los municipios mantuvo el mismo perfil', v)
        if m:
            h.append(f"El {m.group(1)} % de los municipios conservó el mismo perfil en todas las elecciones, y el resto presenta trayectorias que permiten anticipar cambios.")
            continue
        m = re.search(r'<b>([\d.]+) millones</b>', v)
        if m and 'promedio nacional' in v:
            h.append(f"Si cada municipio con participación inferior al promedio nacional lo alcanzara, se sumarían <b>{m.group(1)} millones</b> de votos.")
            continue
        h.append(re.sub(r'(\d)%', r'\1 %', v))
    return h


def ajustar(D):
    """Aplica la redacción técnica a todos los textos del diccionario de datos del dashboard."""
    gen_pais = None
    for p in D['perfiles']:
        m = re.search(r'país: ([+\-−]?[\d.]+) pp', p.get('narrativa', ''))
        if m:
            gen_pais = float(m.group(1).replace('−', '-'))
            break
    try:
        nac_m = round(D['nac']['curva_ult'][-1] - D['nac']['part'][str(D['ult'])], 1)
    except Exception:
        nac_m = None
    D['nac_ref'] = {'genero': gen_pais, 'may': nac_m}
    for p in D['perfiles']:
        p['nombre'] = nombre_perfil(p['nombre'])
        p['narrativa'] = narrativa(p, D, gen_pais)
        p['regla'] = regla(p.get('regla'))
        for e in p['estrategias']:
            e['a'] = [ACCIONES.get(x, x) for x in e['a']]
            e['k'] = INDICADORES.get(e['k'], e['k'])
            e['t'] = NOMBRES_EST.get(e['t'], e['t'])
            e['q'] = QUE_ES.get(e['t'], '')
            e['por'] = por_que(e['t'], p, D.get('pais_jov'), gen_pais, nac_m)
    D['hallazgos'] = hallazgos(D, D.get('hallazgos', []))
    D['catalogo'] = [[NOMBRES_EST.get(c[0], c[0]), INDICADORES.get(c[1], c[1]), c[2]] + ([ACCIONES_CATALOGO[c[0]]] if c[0] in ACCIONES_CATALOGO else [])
                     for c in D.get('catalogo', [])]
    if 'variables' in D:
        nuevas = []
        for v in D['variables']:
            lab = etiqueta(v[0])
            d = DESCRIPCIONES.get(lab, (v[1], v[2] if len(v) > 2 else ''))
            nuevas.append([lab, d[0], d[1]])
        D['variables'] = nuevas
    m = D['modelo']
    m['feats'] = [etiqueta(f) for f in m['feats']]
    m['import_'] = [[etiqueta(r[0]), r[1]] for r in m.get('import_', [])]
    S = D['estados']
    S['feats'] = [etiqueta(f) for f in S['feats']]
    S['flechas'] = [[corta(f[0])] + f[1:] for f in S['flechas']]
    S['mun']['flechas'] = [[corta(f[0])] + f[1:] for f in S['mun']['flechas']]
    def _pc(d):
        partes = []
        for x in re.split(r'\s+y\s+', d):
            mm = re.match(r'^(.*?)\s*\(([+−-])\)$', x.strip())
            partes.append(f"{corta(mm.group(1))} ({mm.group(2)})" if mm else x)
        return ' y '.join(partes)
    S['pc_desc'] = [_pc(d) for d in S['pc_desc']]
    nombres = []
    for n in S['nombres']:
        partes = []
        for x in n.split(' · '):
            y = x[0].lower() + x[1:]
            y = FRASES_SEGMENTO.get(y, y)
            partes.append(y)
        s = ' y '.join(partes)
        nombres.append(s[0].upper() + s[1:])
    S['nombres'] = nombres
    return D


DATA = ajustar(DATA)
datos_json = json.dumps(DATA, ensure_ascii=False, separators=(',', ':')).replace('</', '<\\/')
print('Textos ajustados:', len(DATA['perfiles']), 'perfiles,', len(DATA['variables']), 'variables')

In [ ]:
# Plantilla HTML del dashboard
PLANTILLA = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Perfiles Ciudadanos de Participación</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<style>
:root{
  --bg:#f4faf7; --card:#ffffff; --ink:#12233f; --ink2:#4f5b6e; --ink3:#94a0b2; --line:#e3eee9; --soft:#f1f7f5;
  --uv:#18529D; --uv2:#28AD56; --mag:#18529D; --mor:#28AD56;
  --g1:linear-gradient(135deg,#18529D,#1fa0d8);
  --g2:linear-gradient(135deg,#1c8c6a,#28AD56);
  --g3:linear-gradient(135deg,#18529D,#28AD56);
  --g4:linear-gradient(135deg,#28AD56,#a6d63b);
  --gside:linear-gradient(180deg,#18529D 0%,#1a6aa6 40%,#1d8f86 72%,#28AD56 100%);
  --sombra:0 2px 4px rgba(24,82,157,.05),0 12px 32px rgba(29,143,134,.10);
  --ola:url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 1200 140' preserveAspectRatio='none'><path d='M0 70 C 260 10, 620 150, 1200 40 L1200 140 L0 140Z' fill='%23ffffff' opacity='.35'/><path d='M0 88 C 280 30, 640 160, 1200 62 L1200 140 L0 140Z' fill='%23ffffff'/><path d='M0 112 C 300 60, 660 170, 1200 92 L1200 140 L0 140Z' fill='%2328AD56'/></svg>");
  --olaside:url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 1200 140' preserveAspectRatio='none'><path d='M0 70 C 260 10, 620 150, 1200 40 L1200 140 L0 140Z' fill='%23ffffff' opacity='.3'/><path d='M0 88 C 280 30, 640 160, 1200 62 L1200 140 L0 140Z' fill='%23ffffff'/><path d='M0 112 C 300 60, 660 170, 1200 92 L1200 140 L0 140Z' fill='%231f8f47'/></svg>");
}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--ink);font:14.5px/1.55 Poppins,system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
body::before,body::after{content:"";position:fixed;z-index:0;pointer-events:none;border-radius:50%;filter:blur(10px)}
body::before{width:620px;height:620px;right:-180px;top:-220px;background:radial-gradient(circle,rgba(40,173,86,.16),rgba(40,173,86,0) 65%)}
body::after{width:560px;height:560px;right:20%;bottom:-260px;background:radial-gradient(circle,rgba(24,82,157,.14),rgba(24,82,157,0) 65%)}
.app{display:flex;min-height:100vh;position:relative;z-index:1}

/* ---------- Menú lateral ---------- */
aside{width:282px;flex:none;position:sticky;top:0;height:100vh;padding:22px 16px;background:var(--gside);color:#fff;display:flex;flex-direction:column;gap:4px;box-shadow:6px 0 30px rgba(29,143,134,.30);overflow:hidden}
aside::before{content:"";position:absolute;inset:-40% -60% auto auto;width:360px;height:360px;border-radius:50%;background:radial-gradient(circle,rgba(255,255,255,.25),rgba(255,255,255,0) 60%);animation:flota 9s ease-in-out infinite alternate}
@keyframes flota{to{transform:translate(-80px,140px)}}
.c-top{position:relative;z-index:1;padding:2px 8px 4px;font-size:11.5px;line-height:1.5;opacity:.97}.c-top .uvn{font-weight:700;font-size:13.5px;letter-spacing:.2px;margin-bottom:2px}
.c-top .lg2{display:flex;gap:11px;align-items:center;margin-top:12px;padding-top:12px;border-top:1px solid rgba(255,255,255,.25)}
.c-top .lg{width:40px;height:40px;border-radius:12px;background:#fff;color:#18529D;display:grid;place-items:center;flex:none;box-shadow:0 4px 14px rgba(5,40,60,.25)}.c-top .lg svg{width:22px;height:22px}
.c-top b.eq{display:block;font-size:14px;letter-spacing:.3px;line-height:1.25}.c-top .lg2 small{font-size:11.5px;opacity:.88}
.sep-l{height:1px;background:rgba(255,255,255,.22);margin:12px 6px 10px;position:relative}
.menu a{position:relative;display:flex;align-items:center;gap:12px;padding:10px 14px;border-radius:14px;color:rgba(255,255,255,.88);text-decoration:none;font-weight:500;font-size:14px;white-space:nowrap;cursor:pointer;transition:all .25s}
.menu a:hover{background:rgba(255,255,255,.14);color:#fff;transform:translateX(3px)}
.menu a.on{background:#fff;color:var(--uv);font-weight:600;box-shadow:0 8px 24px rgba(5,40,60,.30)}
.menu a.on svg{color:var(--uv2)}
.menu svg{width:19px;height:19px;flex:none}
.c-bot{margin-top:auto;position:relative;z-index:1;padding:0 8px;margin-bottom:62px}
.c-bot small.k{display:block;font-size:10.5px;font-weight:700;letter-spacing:1px;opacity:.85;margin-bottom:6px}
.c-bot .n{font-size:12.5px;font-weight:600;margin:5px 0;padding-left:10px;border-left:2px solid rgba(255,255,255,.6);line-height:1.35}
.c-bot .f{margin-top:12px;font-size:11px;opacity:.85;line-height:1.5}
aside::after{content:"";position:absolute;left:0;right:0;bottom:0;height:78px;background:var(--olaside) center/100% 100% no-repeat;opacity:.95}

/* ---------- Contenido ---------- */
main{flex:1;min-width:0;padding:26px 30px 44px}
.top{display:flex;justify-content:space-between;align-items:flex-end;gap:16px;flex-wrap:wrap}
.top h1{margin:0;font-size:30px;font-weight:800;letter-spacing:-.6px;background:linear-gradient(90deg,#18529D,#1d8f86 55%,#28AD56);-webkit-background-clip:text;background-clip:text;color:transparent}
.top p{margin:2px 0 0;color:var(--ink2);font-size:13.5px}
.chip-top{display:inline-flex;align-items:center;gap:8px;padding:8px 14px;border-radius:999px;background:#fff;box-shadow:var(--sombra);font-size:12.5px;color:var(--ink2)}
.chip-top i{width:9px;height:9px;border-radius:50%;background:#2bd67b;box-shadow:0 0 0 0 rgba(43,214,123,.6);animation:late 2s infinite}
@keyframes late{70%{box-shadow:0 0 0 9px rgba(43,214,123,0)}}
.vista{display:none}.vista.on{display:block;animation:entra .6s ease both}
@keyframes entra{from{opacity:0;transform:translateY(14px)}to{opacity:1;transform:none}}
.grid{display:grid;grid-template-columns:repeat(12,1fr);gap:18px;margin-top:18px}
.s3{grid-column:span 3}.s4{grid-column:span 4}.s5{grid-column:span 5}.s6{grid-column:span 6}.s7{grid-column:span 7}.s8{grid-column:span 8}.s12{grid-column:span 12}
.card{background:var(--card);border-radius:20px;box-shadow:var(--sombra);padding:20px 22px;min-width:0;position:relative;opacity:0;animation:aparece .7s ease forwards}
@keyframes aparece{from{opacity:0;transform:translateY(16px) scale(.99)}to{opacity:1;transform:none}}
.card h3{margin:0;font-size:16px;font-weight:700;letter-spacing:-.2px}
.card .d{color:var(--ink2);font-size:12.5px;margin-top:2px}
.guia{display:flex;gap:14px;align-items:flex-start;background:linear-gradient(90deg,rgba(24,82,157,.07),rgba(40,173,86,.09));border-radius:18px;padding:14px 18px;margin-top:18px;font-size:13.5px;color:var(--ink2);border-left:4px solid var(--uv2)}
.guia svg{width:22px;height:22px;flex:none;color:var(--uv2);margin-top:1px}
.guia b{color:var(--ink)}
.ctrl{display:flex;gap:10px;align-items:center;flex-wrap:wrap;font-size:13px;color:var(--ink2)}
select,input[type=text]{font:500 13.5px Poppins,system-ui;padding:8px 12px;border:1.5px solid var(--line);border-radius:12px;background:#fff;color:var(--ink);outline:none}
select:focus,input:focus{border-color:var(--mag)}
.num{font-variant-numeric:tabular-nums}

/* KPI con degradado */
.kpi{border-radius:20px;padding:18px 20px;color:#fff;position:relative;overflow:hidden;box-shadow:0 14px 30px -12px rgba(24,82,157,.45);opacity:0;animation:aparece .7s ease forwards}
.kpi::after{content:"";position:absolute;top:0;left:-60%;width:40%;height:100%;background:linear-gradient(100deg,rgba(255,255,255,0),rgba(255,255,255,.35),rgba(255,255,255,0));transform:skewX(-18deg);animation:brillo 4.5s ease-in-out infinite}
@keyframes brillo{0%,60%{left:-60%}100%{left:130%}}
.kpi .l{font-size:12.5px;opacity:.92;font-weight:500}.kpi .v{font-size:34px;font-weight:800;letter-spacing:-1px;line-height:1.15}.kpi .v small{font-size:15px;font-weight:600;margin-left:3px}
.kpi .n{font-size:11.5px;opacity:.85}
.kpi .ic{position:absolute;right:14px;top:14px;width:38px;height:38px;border-radius:12px;background:rgba(255,255,255,.22);display:grid;place-items:center}
.kpi .ic svg{width:20px;height:20px}

/* Gráficas */
.graf{margin-top:10px;position:relative}
.graf svg{display:block;width:100%;overflow:visible}
.eje line{stroke:#e8eef4}.eje text,.txt{fill:var(--ink3);font-size:11.5px;font-family:Poppins,system-ui}
.trazo{stroke-dasharray:1;stroke-dashoffset:1;animation:dibuja 3.2s cubic-bezier(.45,.05,.25,1) forwards}
@keyframes dibuja{to{stroke-dashoffset:0}}
.areaf{opacity:0;animation:sube 2s 1.6s ease forwards}
@keyframes sube{to{opacity:1}}
.pto{opacity:0;animation:sube .6s ease forwards}
.barra{transform-box:fill-box;transform-origin:50% 100%;transform:scaleY(0);animation:crece 2.1s cubic-bezier(.2,.8,.2,1) forwards}
@keyframes crece{to{transform:scaleY(1)}}
.barra-h{transform-box:fill-box;transform-origin:0 50%;transform:scaleX(0);animation:creceh 2.1s cubic-bezier(.2,.8,.2,1) forwards}
@keyframes creceh{to{transform:scaleX(1)}}
.revela{transform-box:fill-box;transform-origin:0 50%;transform:scaleX(0);animation:creceh 3.2s cubic-bezier(.45,.05,.25,1) forwards}
.leyenda{display:flex;flex-wrap:wrap;gap:6px 16px;font-size:12.5px;color:var(--ink2);margin-top:8px}
.leyenda span{display:inline-flex;align-items:center;gap:7px}.leyenda i{width:11px;height:11px;border-radius:4px;display:inline-block}
.pulso{animation:pulso 1.8s ease-in-out infinite}
@keyframes pulso{50%{r:9;opacity:.35}}

.hbar{margin:12px 0}.hbar .et{display:flex;justify-content:space-between;font-size:12.5px;color:var(--ink2);margin-bottom:6px;gap:10px}
.hbar .et b{color:var(--ink)}.hbar .pista{height:12px;border-radius:8px;background:var(--soft);overflow:hidden}
.hbar .pista i{display:block;height:100%;border-radius:8px;width:0;transition:width 1.8s cubic-bezier(.2,.8,.2,1)}

.calor{display:grid;gap:3px;font-size:11.5px;margin-top:10px;min-width:560px}
.calor div{border-radius:6px;padding:6px 4px;text-align:center;font-weight:600;color:var(--ink);opacity:0;animation:sube .8s ease forwards}
.calor .cab{background:none;color:var(--ink2);font-weight:500;opacity:1;animation:none;text-align:left;padding-left:2px}
.calor .cabx{background:none;color:var(--ink2);font-weight:500;opacity:1;animation:none}

table{width:100%;border-collapse:collapse;font-size:12.5px}
th,td{padding:8px 8px;border-bottom:1px solid var(--line);text-align:right}
th:first-child,td:first-child,th:nth-child(2),td:nth-child(2){text-align:left}
th{color:var(--ink2);font-weight:600}
tr.sel td{font-weight:700;background:rgba(40,173,86,.08)}

/* Inicio */
.hero{background:linear-gradient(120deg,#18529D,#1a6aa6 45%,#1d8f86 75%,#28AD56);color:#fff;border-radius:24px;padding:26px 30px 58px;position:relative;overflow:hidden;box-shadow:0 20px 40px -18px rgba(29,143,134,.7);opacity:0;animation:aparece .8s ease forwards}
.hero::after{content:"";position:absolute;right:-80px;top:-80px;width:320px;height:320px;border-radius:50%;background:radial-gradient(circle,rgba(255,255,255,.3),rgba(255,255,255,0) 60%);animation:flota 7s ease-in-out infinite alternate}
.hero::before{content:"";position:absolute;left:0;right:0;bottom:-1px;height:64px;background:var(--ola) center/100% 100% no-repeat;z-index:0}
.hero>*{position:relative;z-index:1}
.hero small{opacity:.85;font-weight:600;letter-spacing:.8px;font-size:12px}
.hero h2{margin:6px 0 8px;font-size:27px;font-weight:800;letter-spacing:-.5px;line-height:1.25;max-width:820px}
.hero p{margin:0;max-width:880px;opacity:.95}
.lista{margin:8px 0 0;padding-left:18px;font-size:13.5px;line-height:1.75;color:var(--ink2)}.lista b{color:var(--ink)}
.formula{margin-top:10px;border-radius:14px;padding:12px;text-align:center;font-weight:700;color:#fff;background:var(--g2);box-shadow:0 10px 22px -10px rgba(31,154,76,.7)}
.vars{display:grid;grid-template-columns:repeat(auto-fill,minmax(250px,1fr));gap:12px;margin-top:12px}
.var{border-radius:16px;padding:14px 16px;background:var(--soft);font-size:13px;line-height:1.5;color:var(--ink2);border-top:4px solid var(--c);opacity:0;animation:aparece .7s ease forwards;transition:transform .2s}
.var:hover{transform:translateY(-3px)}
.var b{display:block;color:var(--ink);font-size:13.5px;margin-bottom:3px}.var em{display:block;margin-top:6px;font-style:normal;color:var(--ink3);font-size:12px}
.pasos{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-top:12px}
.paso{border-radius:16px;padding:16px;background:var(--soft);font-size:13px;color:var(--ink2);line-height:1.5;position:relative}
.paso span{display:grid;place-items:center;width:34px;height:34px;border-radius:12px;color:#fff;font-weight:800;margin-bottom:8px;background:var(--c);box-shadow:0 8px 18px -6px var(--c)}
.paso b{display:block;color:var(--ink);font-size:14px}
.nav-sec{display:grid;grid-template-columns:repeat(8,1fr);gap:10px;margin-top:12px}
.nav-sec div{background:#fff;border-radius:16px;padding:12px 8px;text-align:center;font-size:11.8px;color:var(--ink2);line-height:1.3;cursor:pointer;box-shadow:0 8px 20px -12px rgba(24,82,157,.45);border:1px solid var(--line);transition:transform .2s,box-shadow .2s;opacity:0;animation:aparece .7s ease forwards}
.nav-sec div:hover{transform:translateY(-3px);box-shadow:0 14px 26px -12px rgba(24,82,157,.55)}
.nav-sec .ic{width:38px;height:38px;border-radius:12px;display:grid;place-items:center;margin:0 auto 6px}.nav-sec .ic svg{width:19px;height:19px}
.nav-sec b{display:block;color:var(--ink);font-size:12.5px;margin-bottom:2px}
.grupos-var{display:grid;gap:14px;margin-top:12px;align-items:start}
.grupo{border-radius:16px;background:var(--soft);padding:14px 12px 8px;border-top:4px solid var(--c);opacity:0;animation:aparece .7s ease forwards}
.grupo h4{margin:0 0 10px;font-size:13.5px;display:flex;align-items:center;gap:8px}.grupo h4 span{width:26px;height:26px;border-radius:9px;background:var(--c);display:grid;place-items:center;flex:none}.grupo h4 svg{width:15px;height:15px}
.vchip{background:#fff;border-radius:11px;padding:8px 10px;margin-bottom:7px;cursor:pointer;transition:box-shadow .2s}
.vchip:hover{box-shadow:0 0 0 1.5px var(--c)}
.vchip .vt{display:flex;justify-content:space-between;align-items:center;gap:8px;font-size:12.5px;font-weight:600;color:var(--ink)}
.vchip .vt i{flex:none;width:18px;height:18px;border-radius:50%;border:1.5px solid var(--ink3);position:relative}
.vchip .vt i::before{content:"i";position:absolute;inset:0;display:grid;place-items:center;font-style:normal;font-size:10.5px;font-weight:700;color:var(--ink3)}
.vchip.abierto .vt i::before{content:"−"}
.vchip p{display:none;margin:6px 0 2px;font-size:12px;line-height:1.5;color:var(--ink2)}.vchip p em{display:block;font-style:normal;color:var(--ink3);margin-top:4px}
.vchip.abierto{box-shadow:0 0 0 2px var(--c)}.vchip.abierto p{display:block;animation:entra .35s ease}
.tabs-guia{display:grid;grid-template-columns:repeat(auto-fill,minmax(210px,1fr));gap:10px;margin-top:12px}
.tabs-guia div{border-radius:14px;padding:12px 14px;background:var(--soft);cursor:pointer;font-size:12.5px;color:var(--ink2);transition:all .2s}
.tabs-guia div:hover{background:#fff;box-shadow:var(--sombra);transform:translateY(-2px)}
.tabs-guia b{display:block;color:var(--ink);font-size:13.5px}

/* Perfiles */
.ficha{padding:0;overflow:hidden}
.ficha .cab{padding:18px 22px;color:#fff;background:linear-gradient(120deg,var(--c),var(--c2));position:relative;text-shadow:0 1px 2px rgba(0,0,0,.22)}
.ficha .cab small{font-weight:700;letter-spacing:1px;opacity:.9;font-size:11.5px}
.ficha .cab h4{margin:2px 0 0;font-size:20px;font-weight:800;line-height:1.25;max-width:760px}
.ficha .cab .ico-p{position:absolute;right:20px;top:16px;width:52px;height:52px;border-radius:16px;background:rgba(255,255,255,.2);display:grid;place-items:center}
.ficha .cab .ico-p svg{width:30px;height:30px}
.ficha .cuerpo{padding:18px 22px;display:grid;grid-template-columns:1.1fr 1fr;gap:22px}
.mini{display:grid;grid-template-columns:repeat(4,1fr);gap:10px}
.mini div{background:var(--soft);border-radius:14px;padding:10px 12px}
.mini b{display:block;font-size:20px;font-weight:800;letter-spacing:-.4px}.mini small{font-size:11.5px;color:var(--ink2);line-height:1.25;display:block}
.regla{font-size:12.5px;color:var(--ink2);background:var(--soft);padding:10px 12px;border-radius:12px;margin-top:12px}
.est{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-top:14px}
.est div{border-radius:14px;padding:12px 14px;font-size:12.5px;background:#fff;border:1.5px solid var(--line);position:relative}
.est b{display:block;margin-bottom:4px;font-size:13px;color:var(--ink)}.est ul{margin:0;padding-left:16px;color:var(--ink2)}.est em{display:block;margin-top:6px;color:var(--ink3);font-size:11.5px}
details summary{cursor:pointer;font-size:12.5px;color:var(--mag);font-weight:600;margin-top:12px}

/* Mapa */
.lienzo{position:relative;margin-top:10px}
.lienzo canvas{width:100%;display:block;cursor:crosshair}
.anio-big{position:absolute;right:8px;top:4px;text-align:right;pointer-events:none}
.anio-big b{display:block;font-size:52px;font-weight:800;line-height:1;letter-spacing:-2px;background:linear-gradient(90deg,#18529D,#28AD56);-webkit-background-clip:text;background-clip:text;color:transparent}
.anio-big span{font-size:12.5px;color:var(--ink2)}
.play{width:42px;height:42px;border-radius:50%;border:0;color:#fff;background:var(--g1);display:grid;place-items:center;cursor:pointer;box-shadow:0 10px 22px -8px rgba(24,82,157,.8);transition:transform .2s}
.play:hover{transform:scale(1.08)}.play svg{width:17px;height:17px}
.anios{flex:1;min-width:240px}
.anios input{width:100%;accent-color:var(--mag)}
.marcas{display:flex;justify-content:space-between;font-size:11.5px;color:var(--ink3)}.marcas span{cursor:pointer}.marcas span.on{color:var(--mag);font-weight:700}
.mapa-grid{display:grid;grid-template-columns:1fr 300px;gap:22px;align-items:center;margin-top:10px}.mapa-grid .lienzo{margin-top:0}
.panel-anillos{background:var(--soft);border-radius:18px;padding:16px 16px 12px}
.pa-tit{font-size:13px;font-weight:700;color:var(--ink);text-align:center;margin-bottom:6px}
.leg-anillos div{display:flex;align-items:center;gap:9px;font-size:12.3px;color:var(--ink2);margin:7px 2px;line-height:1.3}
.leg-anillos i{width:12px;height:12px;border-radius:4px;flex:none}.leg-anillos b{margin-left:auto;font-size:15px;color:var(--ink);font-weight:800}
.leg-anillos .escala{display:flex;align-items:center;gap:8px;font-size:11px;color:var(--ink3);margin-top:12px}.leg-anillos .escala i{flex:1;height:9px;border-radius:6px;background:linear-gradient(90deg,#8a2343,#f6f7f9,#2c5a96)}
.cuentas{display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:10px;margin-top:12px}
.cuentas div{border-radius:14px;padding:10px 14px;background:var(--soft);font-size:12px;color:var(--ink2);border-left:4px solid var(--c)}
.cuentas b{display:block;font-size:22px;color:var(--ink);font-weight:800}

/* Simulador / buscador */
.sim{display:grid;grid-template-columns:300px 1fr 300px;gap:26px;margin-top:12px;align-items:start}
.botones{display:grid;grid-template-columns:1fr 1fr;gap:8px}
.botones button{border:0;border-radius:14px;padding:10px 12px;font:600 12.5px Poppins,system-ui;color:var(--ink2);background:var(--soft);cursor:pointer;text-align:left;line-height:1.3;transition:all .2s}
.botones button small{display:block;font-weight:500;color:var(--ink3);font-size:11px}
.botones button.on{color:#fff;background:var(--c);box-shadow:0 10px 22px -10px var(--c)}
.botones button.on small{color:rgba(255,255,255,.85)}
.grande{font-size:42px;font-weight:800;letter-spacing:-1.4px;line-height:1.1;background:linear-gradient(90deg,#18529D,#28AD56);-webkit-background-clip:text;background-clip:text;color:transparent}
input[type=range]{accent-color:var(--mag)}
.caja{background:var(--soft);border-radius:16px;padding:14px 16px}
.etq{display:inline-block;padding:5px 12px;border-radius:999px;color:#fff;font-size:12.5px;font-weight:600;margin-top:8px}
.chips{display:flex;gap:6px;margin-top:12px;flex-wrap:wrap}
.chips span{border-radius:12px;padding:6px 10px;font-size:11.5px;color:#fff;font-weight:600;text-align:center;line-height:1.25;min-width:64px}
.chips span small{display:block;font-weight:500;opacity:.85}

.tres{display:grid;grid-template-columns:repeat(3,1fr);gap:24px}
.mk{font-size:12.5px;font-weight:600;margin-top:12px;color:var(--ink)}.mk span{font-weight:500;color:var(--ink3);margin-left:6px}
.fc{display:flex;flex-direction:column}
.fc .cab h4{font-size:17px;max-width:calc(100% - 70px)}
.fc-cuerpo{padding:16px 20px 6px;display:grid;grid-template-columns:1fr 1.2fr;gap:16px;align-items:center}
.mini.dos{grid-template-columns:1fr 1fr;gap:8px}.fc .mini div{padding:9px 10px}.fc .mini b{font-size:17px;white-space:nowrap}.fc .mini small{font-size:11px}
.fc .eje text{font-size:9.5px}
.fc-g{font-weight:600;font-size:12.5px}.fc-g span{font-weight:400;color:var(--ink3)}
.fc-pie{padding:4px 20px 18px;margin-top:auto}
.chips-e{display:flex;flex-wrap:wrap;gap:6px}.chips-e span{font-size:11.5px;font-weight:600;color:#fff;padding:4px 11px;border-radius:999px}
.ver{margin-top:6px;border:0;color:#fff;font:700 12.8px Poppins,system-ui;padding:9px 16px;border-radius:999px;cursor:pointer;transition:transform .2s}.ver:hover{transform:translateX(3px)}
.velo{position:fixed;inset:0;background:rgba(12,35,70,.28);opacity:0;pointer-events:none;transition:opacity .35s;z-index:60}.velo.on{opacity:1;pointer-events:auto}
.panel{position:fixed;top:0;right:0;bottom:0;width:min(540px,100%);height:auto;padding:0;background:#fff;color:var(--ink);z-index:61;transform:translateX(105%);transition:transform .5s cubic-bezier(.2,.8,.2,1);box-shadow:-20px 0 50px rgba(12,35,70,.25);border-radius:22px 0 0 22px;overflow-y:auto;display:block}
.panel.on{transform:none}
.panel .cab{padding:18px 64px 16px 22px;color:#fff;position:sticky;top:0;z-index:2}.panel .cab small{font-weight:700;letter-spacing:1px;opacity:.9;font-size:11px}.panel .cab h4{margin:2px 0 0;font-size:18px;font-weight:800;line-height:1.25}
.cerrar{position:absolute;right:16px;top:16px;width:36px;height:36px;border-radius:12px;border:0;background:rgba(255,255,255,.22);color:#fff;font-size:22px;line-height:1;cursor:pointer}.cerrar:hover{background:rgba(255,255,255,.35)}
.pn-cuerpo{padding:16px 22px 26px}
.pn-cuerpo h5{margin:18px 0 6px;font-size:13.5px;font-weight:700}
.pn-txt{margin:0;font-size:13px;line-height:1.6;color:var(--ink2)}
.cond{display:inline-block;margin:3px 5px 0 0;padding:4px 10px;border-radius:9px;background:var(--soft);font-size:12px;font-weight:600}
.pn-nota{font-size:11.5px;color:var(--ink3);margin-top:6px}
.pn-est{display:grid;gap:8px}
.ech{border-radius:14px;border:1.5px solid var(--line);background:#fff;transition:box-shadow .2s}.ech.abierto{box-shadow:0 0 0 2px var(--c)}
.ech-t{display:flex;align-items:center;gap:10px;padding:9px 12px;font-size:13px;font-weight:600;cursor:pointer}
.ech-t .ic{width:30px;height:30px;border-radius:10px;display:grid;place-items:center;flex:none}.ech-t .ic svg{width:16px;height:16px}
.ech-t i{margin-left:auto;font-style:normal;color:var(--ink3);font-weight:700}.ech-t i::before{content:"+"}.ech.abierto .ech-t i::before{content:"−"}
.ech-c{display:none;padding:0 14px 12px}.ech.abierto .ech-c{display:block;animation:entra .35s ease}
.ech-c ul{margin:0;padding-left:18px;font-size:12.5px;line-height:1.55;color:var(--ink2)}.ech-c em{display:block;font-style:normal;color:var(--ink3);font-size:11.8px;margin-top:6px}
.pn-tabla{width:100%;border-collapse:collapse;font-size:12.3px;margin-top:6px}.pn-tabla th,.pn-tabla td{padding:7px 6px;border-bottom:1px solid var(--line);text-align:right}.pn-tabla th:first-child,.pn-tabla td:first-child{text-align:left}
.pn-tabla td small{display:block;color:var(--ink3);font-size:11px;white-space:nowrap}
.sim2{display:grid;grid-template-columns:290px minmax(0,620px) 230px;gap:32px;justify-content:space-between;margin-top:12px;align-items:start}
.sim-et{font-size:12.5px;color:var(--ink2)}.sim-pp{font-size:28px;font-weight:800}
.comp{margin-top:16px}.comp .et{display:flex;justify-content:space-between;gap:10px;font-size:12.5px;color:var(--ink2);margin-bottom:6px}.comp .et b{color:var(--ink);white-space:nowrap}
.comp .pista{height:14px;border-radius:8px;background:var(--soft);position:relative;overflow:hidden}
.comp .base{position:absolute;left:0;top:0;bottom:0;background:#cfd8e3;transition:width 1.2s cubic-bezier(.2,.8,.2,1)}
.comp .inc{position:absolute;top:0;bottom:0;border-radius:0 8px 8px 0;transition:left 1.2s,width 1.2s cubic-bezier(.2,.8,.2,1)}
.sim-nota{font-size:11.5px;color:var(--ink3);margin-top:16px;line-height:1.5;max-width:900px}
.matriz{margin-top:10px}
.fila-m{display:grid;grid-template-columns:1fr repeat(var(--k),52px);align-items:center;border-bottom:1px solid var(--line)}
.cab-m{font-size:12px;font-weight:700;color:var(--ink2);padding:4px 0 8px}.cab-m span{justify-self:center;color:#fff;border-radius:999px;padding:2px 10px;font-size:11.5px}
.est-m{display:flex;align-items:center;gap:10px;padding:10px 4px;font-size:13px;font-weight:600;cursor:pointer;min-width:0}
.est-m .ic{width:30px;height:30px;border-radius:10px;display:grid;place-items:center;flex:none}.est-m .ic svg{width:16px;height:16px}
.est-m i{margin-left:auto;font-style:normal;color:var(--ink3);font-weight:700;padding-right:8px}.est-m i::before{content:"+"}.fila-m.abierto .est-m i::before{content:"−"}
.fila-m:hover .est-m .tt{color:var(--uv)}
.punto{justify-self:center;width:14px;height:14px;border-radius:50%}.punto.no{width:10px;height:2px;border-radius:2px;background:#d5dde7}
.det-m{grid-column:1/-1;display:none;padding:0 12px 14px 44px}.fila-m.abierto .det-m{display:block;animation:entra .35s ease}
.det-m ul{margin:0;padding-left:18px;font-size:12.5px;line-height:1.55;color:var(--ink2)}.det-m em{display:block;font-style:normal;color:var(--ink3);font-size:11.8px;margin-top:5px}
.sub-tit{margin-top:6px}.sub-tit h3{margin:0;font-size:17px;font-weight:700}.sub-tit .d{color:var(--ink2);font-size:12.5px;margin-top:2px}
.ec-cuerpo{padding:14px 20px 4px}.ec-n{font-size:12px;font-weight:600;color:var(--ink3);margin-bottom:8px}
.ec-e{display:flex;align-items:center;gap:10px;padding:8px 10px;margin-bottom:6px;border-radius:12px;background:var(--soft)}
.ec-e .ic{width:30px;height:30px;border-radius:10px;display:grid;place-items:center;flex:none}.ec-e .ic svg{width:16px;height:16px}.ec-e b{font-size:13px}
.pa-tabs{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-top:14px}
.pa-tabs button{text-align:left;border:2px solid var(--line);background:#fff;border-radius:14px;padding:10px 14px;font:500 12px Poppins,system-ui;color:var(--ink2);cursor:pointer;transition:all .25s}
.pa-tabs button b{display:block;font-size:13.5px;color:var(--ink);font-weight:700}
.pa-tabs button:hover{border-color:var(--c2)}
.pa-tabs button.on{border-color:transparent;background:linear-gradient(120deg,var(--c),var(--c2));color:rgba(255,255,255,.9);box-shadow:0 10px 22px -12px var(--c2)}.pa-tabs button.on b{color:#fff}
#pa-body{animation:entra .45s ease}
.pa-diag{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-top:14px}
.pa-diag div{border-radius:14px;background:var(--soft);padding:12px 16px;font-size:12.5px;color:var(--ink2);line-height:1.4}.pa-diag b{display:block;font-size:23px;letter-spacing:-.4px}.pa-diag em{font-style:normal;color:var(--ink3)}
.pa-est{display:grid;gap:14px;margin-top:14px;align-items:start}
.pa-card{border-radius:16px;border:1.5px solid var(--line);padding:14px 16px;background:#fff;display:flex;flex-direction:column;gap:9px;transition:box-shadow .25s}.pa-card:hover{box-shadow:0 12px 28px -16px var(--c2)}
.pa-top{display:flex;align-items:center;gap:10px}.pa-top .ic{width:36px;height:36px;border-radius:11px;display:grid;place-items:center;flex:none}.pa-top .ic svg{width:19px;height:19px}.pa-top b{font-size:14.5px;line-height:1.25}
.pa-que{font-size:12.8px;color:var(--ink2);line-height:1.5}
.pa-por{font-size:12.3px;line-height:1.45;border-radius:11px;padding:8px 10px;border-left:3px solid var(--c2);color:var(--ink)}.pa-por small{display:block;font-size:10.5px;font-weight:700;letter-spacing:.6px;color:var(--c)}
.pa-ind{font-size:11.8px;color:var(--ink3)}
.pa-acc button{border:0;background:none;padding:0;font:600 12.5px Poppins,system-ui;color:var(--uv);cursor:pointer}
.pa-acc ul{display:none;margin:8px 0 0;padding-left:18px;font-size:12.3px;line-height:1.55;color:var(--ink2)}.pa-acc.abierto ul{display:block;animation:entra .35s ease}
.pa-h{font-weight:700;font-size:13.5px;margin-top:18px}.pa-h span{font-weight:400;color:var(--ink3);font-size:12px;margin-left:6px}
.pa-prio{display:grid;grid-template-columns:repeat(5,1fr);gap:10px;margin-top:8px}
.pa-prio div{background:var(--soft);border-radius:12px;padding:9px 12px;font-size:11.8px;color:var(--ink2);line-height:1.35}.pa-prio b{display:block;font-size:13px;color:var(--ink)}.pa-prio i{font-style:normal;font-weight:700}
.pa-mas,.pa-cat{margin-top:12px}.pa-mas summary,.pa-cat summary{cursor:pointer;font-size:12.8px;font-weight:600;color:var(--uv)}
.pa-cat{border-top:1px solid var(--line);padding-top:12px;margin-top:18px}
.top,#vistas,footer{max-width:1480px;margin-left:auto;margin-right:auto}
.tk{width:100%;border-collapse:separate;border-spacing:0;font-size:13px;margin-top:10px}
.tk th{font-size:12px;font-weight:700;color:var(--ink);text-align:left;padding:8px 12px;border-bottom:2px solid var(--line);vertical-align:bottom}
.tk th:first-child,.tk td:first-child{text-align:left}
.tk th small{display:block;font-weight:500;color:var(--ink3);font-size:10.5px}
.tk td{padding:7px 12px;border-bottom:1px solid var(--line);text-align:left;vertical-align:middle}
.tk tr.rango td:first-child{box-shadow:inset 3px 0 0 #d9a92f}
.tk tr.sel td{background:linear-gradient(90deg,rgba(24,82,157,.07),rgba(40,173,86,.07));font-weight:400}.tk tr.sel td:first-child{box-shadow:inset 4px 0 0 var(--uv2)}
.tk .kb{display:inline-grid;place-items:center;width:28px;height:28px;border-radius:9px;background:var(--soft);font-weight:800;color:var(--ink)}
.tk tr.sel .kb{background:linear-gradient(135deg,#18529D,#28AD56);color:#fff;box-shadow:0 6px 14px -6px rgba(24,82,157,.7)}
.tk .kt{margin-left:8px;font-size:11px;font-weight:700;color:#fff;background:var(--uv2);padding:2px 8px;border-radius:999px}
.vb{display:flex;align-items:center;gap:10px;min-width:130px}.vb .num{width:44px;font-weight:600;color:var(--ink2)}
.vb i{flex:1;height:8px;border-radius:6px;background:var(--soft);position:relative;overflow:hidden}
.vb i::after{content:"";position:absolute;left:0;top:0;bottom:0;width:0;border-radius:6px;background:var(--c);opacity:.5;transition:width 1.2s cubic-bezier(.2,.8,.2,1)}
.vb i.on::after{width:var(--w)}
.vb.best .num{color:var(--ink);font-weight:800}.vb.best i::after{opacity:1;box-shadow:0 0 10px var(--c)}
.met{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-top:12px}
.met-c{background:var(--soft);border-radius:16px;padding:14px 16px;opacity:0;animation:aparece .7s ease forwards}
.met-h{display:flex;align-items:center;justify-content:space-between;margin-bottom:8px}
.met-n{width:28px;height:28px;border-radius:9px;display:grid;place-items:center;color:#fff;font-weight:800;font-size:13px;background:linear-gradient(135deg,#18529D,#28AD56)}
.met-ic svg{width:20px;height:20px;color:var(--uv)}
.met-c b{display:block;font-size:14px;margin-bottom:4px}.met-c p{margin:0;font-size:12.8px;line-height:1.55;color:var(--ink2)}
.met-v{display:flex;flex-wrap:wrap;gap:4px;margin-top:6px}.met-v i{font-style:normal;font-size:11px;background:#fff;border:1px solid var(--line);border-radius:8px;padding:2px 7px;color:var(--ink)}
.ver{display:inline-flex;align-items:center;gap:8px}.ver svg{width:16px;height:16px}
.pa-acc button{display:inline-flex;align-items:center;gap:8px}.pa-acc button em{font-style:normal}
.mm{width:18px;height:18px;border-radius:6px;background:var(--soft);position:relative;flex:none;box-shadow:inset 0 0 0 1px var(--line)}
.mm::before,.mm::after{content:"";position:absolute;left:4px;right:4px;top:8px;height:2px;border-radius:2px;background:var(--uv)}
.mm::after{transform:rotate(90deg);transition:transform .25s}.pa-acc.abierto .mm::after{transform:rotate(0)}
details>summary{list-style:none;display:inline-flex;align-items:center;gap:8px}details>summary::-webkit-details-marker{display:none}
details>summary::before{content:"";width:18px;height:18px;border-radius:6px;flex:none;background:var(--soft) linear-gradient(var(--uv),var(--uv)) center/10px 2px no-repeat;box-shadow:inset 0 0 0 1px var(--line)}
details:not([open])>summary::before{background:linear-gradient(var(--uv),var(--uv)) center/10px 2px no-repeat,linear-gradient(var(--uv),var(--uv)) center/2px 10px no-repeat var(--soft)}
.dos-v{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-top:6px}.dos-v small{display:block;font-size:11px;color:var(--ink3)}.dos-v b{font-size:21px;font-weight:800}
.def-pp{background:#fff;border-radius:16px;box-shadow:var(--sombra);padding:12px 18px;border-left:4px solid var(--uv);transition:box-shadow .4s}
.def-pp summary{font-weight:700;font-size:13.5px;color:var(--ink)}
.def-c{margin-top:8px;font-size:13px;line-height:1.6;color:var(--ink2)}.def-c p{margin:0 0 6px}.def-c b{color:var(--ink)}
.inf-col{display:flex;flex-direction:column;gap:14px}
.inf{flex:1;display:flex;gap:14px;align-items:center;text-align:left;font:inherit;color:var(--ink);background:#fff;border:0;border-radius:18px;padding:16px 18px;box-shadow:var(--sombra);border-left:4px solid var(--c);cursor:pointer;transition:transform .2s,box-shadow .2s}
.inf:hover{transform:translateY(-2px);box-shadow:0 14px 34px -12px rgba(12,35,70,.28)}
.inf-ic{width:38px;height:38px;border-radius:12px;background:var(--c);display:grid;place-items:center;flex:none;box-shadow:0 8px 18px -8px var(--c)}.inf-ic svg{width:20px;height:20px}
.inf-tx{display:flex;flex-direction:column;gap:4px;min-width:0;flex:1}.inf-tx b{font-size:14.5px}.inf-tx small{font-size:12.5px;color:var(--ink2);line-height:1.5}
.inf-tx em{font-style:normal;font-weight:700;font-size:12px;color:var(--c);display:inline-flex;align-items:center;gap:6px;margin-top:4px}.inf-tx em svg{width:15px;height:15px}
.inf-f{font-weight:700;font-size:13.5px;color:var(--c);background:var(--soft);border-radius:10px;padding:6px 10px;align-self:flex-start;margin-top:2px}
.inf-n{display:flex;gap:14px;flex-wrap:wrap;margin-top:2px}.inf-n span{font-size:11.5px;color:var(--ink3)}.inf-n b{display:block;font-size:15px;color:var(--ink)}
.pn-form{margin:10px 0;background:var(--fg);box-shadow:0 10px 22px -10px var(--fs)}
.def-v{border-left-color:var(--uv2)}.def-o{border-left-color:#c9972a}
.dos-col{columns:2;column-gap:40px}.dos-col li{break-inside:avoid}
.med-c{display:grid;grid-template-columns:1fr 1fr;gap:28px;align-items:center}.med-c .lista{margin-top:0}.nota-f{font-size:12.5px;color:var(--ink3);margin:10px 0 0;line-height:1.5}
@media (max-width:760px){.dos-col{columns:1}.med-c{grid-template-columns:1fr;gap:10px}}
.def-pp.flash{animation:flashpp 1.6s ease}
@keyframes flashpp{0%,60%{box-shadow:0 0 0 3px var(--uv2),0 12px 30px rgba(24,82,157,.2)}100%{box-shadow:var(--sombra)}}
.pp-link{display:inline-block;font-size:.82em;font-weight:700;line-height:1.35;background:#eaf1fa;color:#18529D;border-radius:6px;padding:0 5px;cursor:pointer;vertical-align:baseline;transition:background .15s}
.pp-link:hover{background:#d3e3f6}
.tip{position:fixed;pointer-events:none;background:rgba(12,35,70,.95);color:#fff;padding:8px 12px;border-radius:12px;font-size:12.5px;line-height:1.45;opacity:0;transition:opacity .12s;z-index:90;max-width:300px;box-shadow:0 10px 30px rgba(12,35,70,.3);z-index:90}
.tip b{font-size:13px}
.pildoras{display:grid;grid-template-columns:repeat(6,1fr);gap:8px;margin-top:14px}
.pildoras div{border-radius:14px;padding:8px 6px;text-align:center;font-size:11px;color:var(--ink2);background:var(--soft);opacity:0;animation:aparece .7s ease forwards}
.pildoras b{display:block;font-size:17px;color:var(--ink);letter-spacing:-.3px}
.pildoras .pres{background:linear-gradient(135deg,#18529D,#28AD56);color:rgba(255,255,255,.9);box-shadow:0 10px 20px -10px rgba(29,143,134,.8)}.pildoras .pres b{color:#fff}
.chispa{position:absolute;right:16px;top:60px;overflow:visible}
.card{transition:box-shadow .3s,transform .3s}.card:hover{box-shadow:0 0 0 1.5px rgba(40,173,86,.2),0 18px 40px rgba(24,82,157,.14)}
.ip{flex:none;width:34px;height:34px;border-radius:11px;display:grid;place-items:center}.ip svg{width:20px;height:20px}
.lienzo canvas{filter:drop-shadow(0 14px 24px rgba(24,82,157,.20)) drop-shadow(0 0 2px rgba(24,82,157,.25))}
footer{margin-top:26px;color:var(--ink3);font-size:11.5px;line-height:1.6}
.tres{}@media (max-width:1180px){.met{grid-template-columns:1fr 1fr}.pa-est{grid-template-columns:1fr!important}.pa-prio{grid-template-columns:repeat(2,1fr)}.pa-tabs,.pa-diag{grid-template-columns:1fr 1fr}.sim2{grid-template-columns:1fr}.fc{grid-column:span 12}.nav-sec{grid-template-columns:repeat(4,1fr)}.grupos-var{grid-template-columns:1fr 1fr!important}.mapa-grid{grid-template-columns:1fr}.tres{grid-template-columns:1fr}.s3{grid-column:span 6}.s4,.s5,.s6,.s7,.s8{grid-column:span 12}.ficha .cuerpo{grid-template-columns:1fr}.sim{grid-template-columns:1fr}.pasos{grid-template-columns:1fr 1fr}}
@media (max-width:760px){.met{grid-template-columns:1fr}.fc-cuerpo{grid-template-columns:1fr}.panel{border-radius:0}.nav-sec{grid-template-columns:repeat(2,1fr)}.grupos-var{grid-template-columns:1fr!important}.app{flex-direction:column}aside{position:sticky;top:0;z-index:30;width:100%;height:auto;padding:8px 10px;flex-direction:row;overflow-x:auto;overflow-y:hidden}aside::before,aside::after,.c-top,.sep-l,.c-bot{display:none}.menu{display:flex;gap:4px}.menu a{padding:8px 12px;font-size:13px}.menu a:hover{transform:none}.pildoras{grid-template-columns:repeat(3,1fr)}.chispa{display:none}main{padding:16px}.s3{grid-column:span 12}.mini{grid-template-columns:1fr 1fr}.est{grid-template-columns:1fr}}
</style>
</head>
<body>
<div class="app">
<aside>
  <div class="c-top">
    <div class="uvn">Universidad Veracruzana</div>Facultad de Estadística e Informática<br>Lic. en Ingeniería en Ciencia de Datos
    <div class="lg2"><span class="lg"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><path d="M4 19V9M10 19V5M16 19v-6M22 19H2"/></svg></span><div><b class="eq">THE DATA SCIENTISTS</b><small>Hackathon INE 2026, Reto 1</small></div></div>
  </div>
  <div class="sep-l"></div>
  <nav class="menu" id="menu"></nav>
  <div class="c-bot"><small class="k">INTEGRANTES</small><div class="n">Cristian Axel Izquierdo López</div><div class="n">Laura Yoselin Tonil De La Luz</div><div class="f">Fuente: INE, Conteos Censales de Participación Ciudadana 2009–2024</div></div>
</aside>
<main>
  <div class="top">
    <div><h1>Perfiles ciudadanos de participación</h1><p id="subt"></p></div>
  </div>
  <div id="vistas"></div>
</main>
</div>
<div class="velo" id="velo"></div><div class="panel" id="panel" role="dialog" aria-label="Ficha completa del perfil"></div>
<div class="tip" id="tip"></div>

<script>
const D = __DATOS__;
const GEO = __GEO__;
const GEOE = __GEOE__;
</script>
<script>
/* ======================= utilidades ======================= */
const $ = s => document.querySelector(s);
const P = D.perfiles, K = D.K, ANIOS = D.anios.map(String), ULT = String(D.ult);
// Paleta académica de las gráficas (verde, oro, azul y vino), en armonía con los colores UV del marco
const PAL = ['#4fae80', '#c9972a', '#2c5a96', '#8a2343', '#2a9d8f', '#6b7a8f'];
const PAL2 = ['#93d1b1', '#e9c86a', '#7fa3d6', '#c8697f', '#7fcfc4', '#aab4c2'];
const OSC = ['#2f7d58', '#8a6512', '#1f4372', '#6a1830', '#1d7066', '#4f5a6b'];
P.forEach((p, i) => { p.color = PAL[i]; p.color2 = PAL2[i]; p.osc = OSC[i]; });
const N = v => v == null ? '—' : Number(v).toLocaleString('es-MX');
const SG = (v, d = 1) => v == null ? '—' : (v > 0 ? '+' : v < 0 ? '−' : '') + Math.abs(v).toFixed(d);
const nb = s => (s || '').replace(/ (%|pp)(?!\w)/g, '&nbsp;$1').replace(/(\d) (millones)/g, '$1&nbsp;$2');
const corto = i => P[i].corto.replace(/^P\d+ · /, '');
const CORTAS = [[/^Participación (vs\. nacional|relativa)/, 'Part. relativa'], [/^Brecha (jóvenes|de participación) 18/, 'Brecha 18–29'], [/^Brecha (de participación )?50–69/, 'Brecha 50–69'], [/^Brecha (70\+|de participación 70)/, 'Brecha 70+'], [/(con|de) 18–29 años/, '% 18–29'], [/(60\+ años|60 años o más)/, '% 60+'], [/rurales/, '% rural'], [/urbanas/, '% urbano'], [/^Tamaño/, 'Tamaño'], [/mujeres|género/, 'Brecha género']];
const corta = f => { for (const [r, c] of CORTAS) if (r.test(f)) return c; return f.split(' (')[0]; };
const tip = $('#tip');
function verTip(h, x, y) { tip.innerHTML = h; tip.style.opacity = 1; const w = tip.offsetWidth; tip.style.left = Math.min(x + 16, innerWidth - w - 12) + 'px'; tip.style.top = (y + 16) + 'px'; }
function sinTip() { tip.style.opacity = 0; }
document.addEventListener('mousemove', e => { const t = e.target.closest && e.target.closest('[data-tip]'); if (t) verTip(t.dataset.tip, e.clientX, e.clientY); else if (!e.target.closest('canvas')) sinTip(); });
function contar(el, fin, dec = 0, dur = 2200, pre = '', suf = '') {
  const t0 = performance.now();
  (function paso(t) { const q = Math.min(1, (t - t0) / dur), e = 1 - Math.pow(1 - q, 4);
    el.textContent = pre + (fin * e).toLocaleString('es-MX', {minimumFractionDigits: dec, maximumFractionDigits: dec}) + suf;
    if (q < 1) requestAnimationFrame(paso); })(t0);
}
const ICO = {
  inicio: '<path d="M3 11l9-7 9 7v9a1 1 0 0 1-1 1h-5v-6H9v6H4a1 1 0 0 1-1-1z"/>',
  resumen: '<path d="M4 4h7v7H4zM13 4h7v7h-7zM4 13h7v7H4zM13 13h7v7h-7z"/>',
  perfiles: '<circle cx="9" cy="8" r="3"/><path d="M3 20c0-3.3 2.7-6 6-6s6 2.7 6 6"/><circle cx="17" cy="9" r="2.5"/><path d="M15 14.5c2.8 0 5 2.2 5 5"/>',
  mapa: '<path d="M9 4 3 6v14l6-2 6 2 6-2V4l-6 2-6-2zM9 4v14M15 6v14"/>',
  tray: '<path d="M3 17l6-6 4 4 8-8"/><path d="M15 7h6v6"/>',
  estados: '<circle cx="12" cy="12" r="8"/><path d="M12 4v16M4 12h16"/>',
  estrategias: '<path d="M12 2l2.4 6.9H22l-6 4.6 2.3 7L12 16l-6.3 4.5 2.3-7-6-4.6h7.6z"/>',
  buscar: '<circle cx="11" cy="11" r="6"/><path d="m20 20-4.5-4.5"/>',
  modelo: '<path d="M4 20V10M10 20V4M16 20v-7M22 20H2"/>',
  info: '<circle cx="12" cy="12" r="9"/><path d="M12 11v6M12 7.5v.5"/>',
  panel: '<rect x="3" y="4" width="18" height="16" rx="2.5"/><path d="M14 4v16M17 9h1M17 12h1"/>',
  lista: '<path d="M8 6h13M8 12h13M8 18h13M3 6h.01M3 12h.01M3 18h.01"/>',
  territorio: '<path d="M3 21h18M5 21V10l4-3 4 3v11M13 21V6l6 3v12"/>',
  gente: '<circle cx="12" cy="8" r="3.5"/><path d="M5 20c0-3.9 3.1-7 7-7s7 3.1 7 7"/>',
  urna: '<path d="M4 10h16v10H4zM8 10V6h8v4M10 4h4M9 14h6"/>',
  chart: '<path d="M4 19V9M10 19V5M16 19v-6M22 19H2"/>',
  flecha: '<path d="M12 4v12M7 11l5 5 5-5M5 20h14"/>'
};
const ico = (k, c = 'currentColor', w = 2) => `<svg viewBox="0 0 24 24" fill="none" stroke="${c}" stroke-width="${w}" stroke-linecap="round" stroke-linejoin="round">${ICO[k]}</svg>`;
const ICO_PERFIL = ['<path d="M12 3l2.6 5.3 5.9.9-4.3 4.1 1 5.8L12 16.4 6.8 19.1l1-5.8L3.5 9.2l5.9-.9z"/>', '<path d="M3 20h18M5 20V10l4-3 4 3v10M13 20V6l6 3v11M8 13h2M8 16h2M16 12h1M16 15h1"/>', '<path d="M3 21h18M5 21V7h6v14M13 21V3h6v18M7 10h2M7 13h2M7 16h2M15 6h2M15 9h2M15 12h2M15 15h2"/>', '<circle cx="12" cy="7" r="3"/><path d="M6 21v-2a6 6 0 0 1 12 0v2M17 4l3-2M7 4L4 2"/>'];
const icoPerfil = i => `<svg viewBox="0 0 24 24" fill="none" stroke="#fff" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round">${ICO_PERFIL[i % 4]}</svg>`;
let uid = 0;
const defsBrillo = (color, id, s = 3.6, k = 1.9) => `<filter id="${id}" filterUnits="userSpaceOnUse" x="-300" y="-300" width="4000" height="3000" color-interpolation-filters="sRGB"><feGaussianBlur in="SourceGraphic" stdDeviation="${s}" result="b"/><feColorMatrix in="b" type="matrix" values="1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 ${k} 0" result="g"/><feMerge><feMergeNode in="g"/><feMergeNode in="SourceGraphic"/></feMerge></filter>`;
function escala(a, b, c, d) { return v => c + (v - a) / (b - a || 1) * (d - c); }
function niceTicks(min, max, n = 5) { const paso = Math.pow(10, Math.floor(Math.log10((max - min) / n || 1))); const err = (max - min) / n / paso; const st = paso * (err >= 7.5 ? 10 : err >= 3.5 ? 5 : err >= 1.5 ? 2 : 1); const out = []; for (let v = Math.ceil(min / st) * st; v <= max + 1e-9; v += st) out.push(+v.toFixed(6)); return out; }

/* ---------- gráfica de líneas con brillo ---------- */
function lineas(el, o) {
  const W = el.clientWidth || 600, H = o.h || 300, ml = o.ml || 46, mr = o.mr || 18, mt = 14, mb = 32;
  const todos = o.series.flatMap(s => s.vals.filter(v => v != null));
  let y0 = o.yMin ?? Math.min(...todos), y1 = o.yMax ?? Math.max(...todos); const pad = (y1 - y0) * .12 || 1;
  if (o.yMin == null) y0 -= pad; if (o.yMax == null) y1 += pad;
  const n = o.cats.length, X = i => ml + (n === 1 ? .5 : i / (n - 1)) * (W - ml - mr), Y = escala(y0, y1, H - mb, mt);
  let s = `<svg viewBox="0 0 ${W} ${H}" height="${H}"><defs>`;
  o.series.forEach((se, k) => { const id = 'b' + (++uid); se._f = id; se._g = 'g' + uid; se._s = 's' + uid;
    s += defsBrillo(se.color, id) + `<linearGradient id="${se._g}" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="${se.grad ? se.grad[1] : se.color}" stop-opacity=".30"/><stop offset="1" stop-color="${se.color}" stop-opacity="0"/></linearGradient>`;
    if (se.grad) s += `<linearGradient id="${se._s}" gradientUnits="userSpaceOnUse" x1="${ml}" x2="${W - mr}" y1="0" y2="0">${se.grad.map((c, j) => `<stop offset="${j / (se.grad.length - 1)}" stop-color="${c}"/>`).join('')}</linearGradient>`;
    if (se.dash) s += `<clipPath id="c${se._s}"><rect class="revela" style="animation-delay:${(o.stagger ?? .25) * k}s" x="0" y="-20" width="${W}" height="${H + 40}"/></clipPath>`; });
  s += '</defs><g class="eje">';
  niceTicks(y0, y1, o.nt || 5).forEach(v => s += `<line x1="${ml}" x2="${W - mr}" y1="${Y(v)}" y2="${Y(v)}"/><text x="${ml - 8}" y="${Y(v) + 4}" text-anchor="end">${v}${o.unit === '%' ? ' %' : (o.unit || '')}</text>`);
  o.cats.forEach((c, i) => s += `<text x="${X(i)}" y="${H - 9}" text-anchor="middle">${c}</text>`);
  if (o.yLabel) s += `<text x="${12}" y="${mt + (H - mt - mb) / 2}" transform="rotate(-90 12 ${mt + (H - mt - mb) / 2})" text-anchor="middle">${o.yLabel}</text>`;
  s += '</g>';
  if (o.ref != null) s += `<line x1="${ml}" x2="${W - mr}" y1="${Y(o.ref)}" y2="${Y(o.ref)}" stroke="#12233f" stroke-dasharray="4 4" stroke-width="1.2" opacity=".5"/>`;
  o.series.forEach((se, k) => {
    const pts = se.vals.map((v, i) => v == null ? null : [X(i), Y(v)]);
    let d = ''; pts.forEach((p, i) => { if (!p) return; d += (d && pts[i - 1] ? ' L' : ' M') + p[0].toFixed(1) + ',' + p[1].toFixed(1); });
    const delay = (o.stagger ?? .25) * k;
    if (o.area && d) { const ok = pts.filter(Boolean); s += `<path class="areaf" style="animation-delay:${1.2 + delay}s" d="${d} L${ok.at(-1)[0]},${Y(y0)} L${ok[0][0]},${Y(y0)} Z" fill="url(#${se._g})"/>`; }
    const trazo = se.grad ? `url(#${se._s})` : se.color;
    if (se.dash) s += `<path clip-path="url(#c${se._s})" d="${d}" fill="none" stroke="${trazo}" stroke-width="${se.w || 3}" stroke-dasharray="7 6" stroke-linecap="round" stroke-linejoin="round" filter="url(#${se._f})"/>`;
    else if (!se.soloPuntos) s += `<path class="trazo" style="animation-delay:${delay}s" pathLength="1" d="${d}" fill="none" stroke="${trazo}" stroke-width="${se.w || 3}" stroke-linecap="round" stroke-linejoin="round" filter="url(#${se._f})"/>`;
    pts.forEach((p, i) => { if (!p) return; const t = `<b>${se.name}</b><br>${o.cats[i]}: ${se.vals[i].toFixed(o.dec ?? 1)}${o.unit === '%' ? ' %' : (o.unit || '')}`;
      s += se.soloPuntos ? `<circle class="pto" style="animation-delay:${1.6 + delay + i * .1}s" cx="${p[0]}" cy="${p[1]}" r="${se.r || 8}" fill="none" stroke="${se.color}" stroke-width="2.5" filter="url(#${se._f})"/>`
        : `<circle class="pto" style="animation-delay:${1.3 + delay + i * .08}s" cx="${p[0]}" cy="${p[1]}" r="${se.r || 4.5}" fill="#fff" stroke="${se.grad ? se.grad[Math.round(i / Math.max(1, pts.length - 1) * (se.grad.length - 1))] : se.color}" stroke-width="2.5"/>`;
      s += `<circle cx="${p[0]}" cy="${p[1]}" r="13" fill="transparent" data-tip="${t.replace(/"/g, '&quot;')}"/>`; });
    if (o.pulso && k === 0) { const ul = pts.filter(Boolean).at(-1); if (ul) s += `<circle class="pulso" cx="${ul[0]}" cy="${ul[1]}" r="6" fill="${se.color}" opacity=".6"/>`; }
    if (o.etiquetaFin) { const ul = pts.filter(Boolean).at(-1); if (ul) s += `<text class="pto" style="animation-delay:2.4s" x="${ul[0] + 10}" y="${ul[1] + 4}" font-size="13" font-weight="800" fill="${se.grad ? se.grad.at(-1) : se.color}">${se.vals.filter(v => v != null).at(-1).toFixed(1)}${o.unit === '%' ? ' %' : (o.unit || '')}</text>`; }
  });
  s += '</svg>';
  if (o.series.length > 1 && o.leyenda !== false) s += `<div class="leyenda">${o.series.map(se => `<span><i style="${se.soloPuntos ? `border:2.5px solid ${se.color};background:#fff;border-radius:50%` : se.dash ? `background:repeating-linear-gradient(90deg,${se.color} 0 4px,transparent 4px 7px);height:4px;border-radius:2px;width:18px` : `background:${se.color}`};box-shadow:0 0 8px ${se.color}"></i>${se.name}</span>`).join('')}</div>`;
  el.innerHTML = s;
}

/* ---------- barras apiladas verticales ---------- */
function barrasApiladas(el, o) {
  const W = el.clientWidth || 600, H = o.h || 300, ml = 46, mr = 12, mt = 12, mb = 32, n = o.cats.length;
  const tot = o.cats.map((_, i) => o.series.reduce((a, s) => a + (s.vals[i] || 0), 0)), mx = o.max ?? Math.max(...tot);
  const Y = escala(0, mx, H - mb, mt), bw = (W - ml - mr) / n * .58, X = i => ml + (i + .5) * (W - ml - mr) / n;
  let s = `<svg viewBox="0 0 ${W} ${H}" height="${H}"><defs>${o.series.map((se, k) => defsBrillo(se.color, 'bb' + uid + '_' + k, 2.2, .9)).join('')}</defs><g class="eje">`;
  niceTicks(0, mx, 5).forEach(v => s += `<line x1="${ml}" x2="${W - mr}" y1="${Y(v)}" y2="${Y(v)}"/><text x="${ml - 8}" y="${Y(v) + 4}" text-anchor="end">${v}${o.unit === '%' ? ' %' : (o.unit || '')}</text>`);
  o.cats.forEach((c, i) => s += `<text x="${X(i)}" y="${H - 9}" text-anchor="middle">${c}</text>`);
  s += '</g>';
  o.cats.forEach((c, i) => { let acc = 0; o.series.forEach((se, k) => { const v = se.vals[i] || 0; if (!v) return; const yA = Y(acc + v), yB = Y(acc); acc += v;
    s += `<rect class="barra" style="animation-delay:${i * .12 + k * .08}s" x="${X(i) - bw / 2}" y="${yA}" width="${bw}" height="${Math.max(0, yB - yA - 2)}" rx="5" fill="${se.color}" filter="url(#bb${uid}_${k})" data-tip="<b>${se.name}</b><br>${c}: ${v.toFixed(1)}${o.unit === '%' ? ' %' : (o.unit || '')}"/>`; }); });
  s += '</svg>' + `<div class="leyenda">${o.series.map(se => `<span><i style="background:${se.color};box-shadow:0 0 8px ${se.color}"></i>${se.name}</span>`).join('')}</div>`;
  uid++; el.innerHTML = s;
}

/* ---------- barras horizontales (HTML) ---------- */
function barrasH(el, items, o = {}) {
  const mx = o.max ?? Math.max(...items.map(i => i.v));
  el.innerHTML = items.map((it, i) => `<div class="hbar" data-tip="${(it.tip || `<b>${it.l}</b><br>${it.v}`).replace(/"/g, '&quot;')}"><div class="et"><span>${it.l}</span><b class="num">${it.txt ?? it.v}</b></div><div class="pista"><i style="background:${it.c2 ? `linear-gradient(90deg,${it.c},${it.c2})` : it.c};box-shadow:0 0 12px ${it.c}88" data-w="${100 * it.v / mx}"></i></div></div>`).join('');
  setTimeout(() => el.querySelectorAll('.pista i').forEach((b, k) => setTimeout(() => b.style.width = b.dataset.w + '%', k * 120)), 120);
}

/* ---------- mapa de calor (HTML) ---------- */
function divergente(v, lim) { const t = Math.max(-1, Math.min(1, v / lim)); const a = t < 0 ? [138, 35, 67] : [44, 90, 150]; const f = Math.abs(t); return `rgb(${Math.round(246 + (a[0] - 246) * f)},${Math.round(247 + (a[1] - 247) * f)},${Math.round(249 + (a[2] - 249) * f)})`; }
// anima desde cero un número ya escrito en un elemento (respeta signo, decimales y sufijo como % o pp)
function animarTexto(el, dur = 1400) {
  const m = (el.textContent || '').trim().match(/^([−-]?)([\d,]*\.?\d+)(.*)$/);
  if (!m) return;
  const v = parseFloat(m[2].replace(/,/g, '')), dec = (m[2].split('.')[1] || '').length;
  contar(el, v, dec, dur, m[1], m[3]);
}
// anima los números fijos de una sección recién dibujada (no toca años, numeraciones ni los que ya cuentan solos)
function animarSeccion(raiz) {
  raiz.querySelectorAll('*').forEach(e => {
    if (e.children.length || e.closest('svg,select,option,.marcas,.kb,.paso,.tip')) return;
    const t = (e.textContent || '').trim(), m = t.match(/^([−-]?)([\d,]*\.?\d+)(\s*(?:%|pp|M)?)$/);
    if (!m) return;
    const v = parseFloat(m[2].replace(/,/g, ''));
    if (!v || (/^(19|20)\d\d$/.test(m[2]) && !m[3]) || (Number.isInteger(v) && v < 10 && !m[3] && !m[2].includes('.'))) return;
    animarTexto(e, 1400);
  });
}
function calor(el, filas, cols, z, lim, fmt = v => v.toFixed(2), anchoEt = '220px') {
  let s = `<div class="calor" style="grid-template-columns:${anchoEt} repeat(${cols.length},1fr)"><div class="cab"></div>${cols.map(c => `<div class="cabx">${c}</div>`).join('')}`;
  filas.forEach((f, i) => { s += `<div class="cab">${f}</div>` + cols.map((c, j) => { const v = z[i][j]; const fuerte = Math.abs(v) / lim > .55;
    return `<div style="background:${divergente(v, lim)};color:${fuerte ? '#fff' : 'var(--ink)'};animation-delay:${(i * cols.length + j) * .025}s" data-tip="<b>${f}</b><br>${c}: ${fmt(v)}">${fmt(v)}</div>`; }).join(''); });
  el.innerHTML = s + '</div>';
  el.querySelectorAll('.calor > div[data-tip]').forEach(c => animarTexto(c, 1300));
}

/* ---------- mapas en canvas ---------- */
const LON0 = -118.6, LON1 = -86.5, LAT0 = 14.3, LAT1 = 32.9, KX = Math.cos(23 * Math.PI / 180);
function crearMapa(cv, geo) {
  const st = {cv, ctx: cv.getContext('2d'), geo, caminos: [], dpr: 1, colores: {}, sel: null};
  st.proyectar = () => {
    const w = cv.clientWidth || 800, h = w * ((LAT1 - LAT0) / ((LON1 - LON0) * KX)), dpr = devicePixelRatio || 1;
    cv.width = w * dpr; cv.height = h * dpr; cv.style.height = h + 'px'; st.dpr = dpr;
    const sx = w / ((LON1 - LON0) * KX), px = l => (l - LON0) * KX * sx, py = l => (LAT1 - l) * sx;
    st.caminos = Object.entries(geo).map(([id, rings]) => { const p = new Path2D(); let b = [1e9, 1e9, -1e9, -1e9];
      rings.forEach(r => { for (let i = 0; i < r.length; i += 2) { const x = px(r[i]), y = py(r[i + 1]); i ? p.lineTo(x, y) : p.moveTo(x, y); b = [Math.min(b[0], x), Math.min(b[1], y), Math.max(b[2], x), Math.max(b[3], y)]; } p.closePath(); });
      return {id, p, b}; });
  };
  st.dibujar = (colorDe, t = 1, antes = null, borde = '#ffffff', grosor = .35) => {
    const c = st.ctx; c.setTransform(st.dpr, 0, 0, st.dpr, 0, 0); c.clearRect(0, 0, cv.width, cv.height);
    c.lineWidth = grosor; c.strokeStyle = borde;
    for (const k of st.caminos) { const nu = colorDe(k.id); c.fillStyle = antes && antes[k.id] && antes[k.id] !== nu && t < 1 ? mezclar(antes[k.id], nu, t) : nu; c.fill(k.p); c.stroke(k.p); st.colores[k.id] = nu; }
    if (st.sel) { const k = st.caminos.find(k => k.id === st.sel); if (k) { c.save(); c.shadowColor = '#d9a92f'; c.shadowBlur = 14; c.lineWidth = 2.6; c.strokeStyle = '#12233f'; c.stroke(k.p); c.restore(); } }
  };
  st.animar = (colorDe, dur = 900) => { const antes = {...st.colores}, t0 = performance.now();
    (function paso(t) { const q = Math.min(1, (t - t0) / dur); st.dibujar(colorDe, q, antes); if (q < 1) requestAnimationFrame(paso); })(t0); };
  st.buscar = e => { const r = cv.getBoundingClientRect(), x = e.clientX - r.left, y = e.clientY - r.top; st.ctx.setTransform(1, 0, 0, 1, 0, 0);
    for (const k of st.caminos) { const b = k.b; if (x < b[0] || x > b[2] || y < b[1] || y > b[3]) continue; if (st.ctx.isPointInPath(k.p, x, y)) return k.id; } return null; };
  return st;
}
function hex2rgb(c) { if (c.startsWith('rgb')) return c.match(/\d+/g).map(Number); return [1, 3, 5].map(i => parseInt(c.slice(i, i + 2), 16)); }
function mezclar(a, b, t) { const A = hex2rgb(a), B = hex2rgb(b); return `rgb(${A.map((v, i) => Math.round(v + (B[i] - v) * t)).join(',')})`; }

/* ======================= estructura ======================= */
const TABS = [['inicio', 'Inicio'], ['resumen', 'Resumen'], ['perfiles', 'Perfiles'], ['mapa', 'Mapa'], ['tray', 'Trayectorias y región'], ['estados', 'Estados (PCA y K-Means)'], ['estrategias', 'Estrategias'], ['buscar', 'Buscador municipal'], ['modelo', 'Modelo']];
const GUIA = {
  resumen: 'En la parte superior se presentan los indicadores principales. Debajo se muestran la participación nacional por elección, la distribución de la lista nominal entre los perfiles y la participación nacional por grupo de edad y por tipo de sección. Las líneas punteadas corresponden a elecciones intermedias.',
  perfiles: 'Cada tarjeta resume un perfil con su número de municipios, su lista nominal, su participación respecto al promedio nacional y la brecha de participación del grupo de 18 a 29 años. La ficha completa incluye la descripción, la regla de clasificación, las estrategias asignadas y los municipios prioritarios. Al final se presenta la distribución municipal de la participación por perfil.',
  mapa: 'El botón de reproducción muestra la evolución de los perfiles entre 2009 y 2024; también es posible elegir una elección con la barra o cambiar la variable representada. En el análisis LISA, <b>Alto-Alto</b> identifica municipios con participación superior al promedio nacional rodeados de municipios similares, <b>Bajo-Bajo</b> el caso inverso, y <b>Alto-Bajo</b> o <b>Bajo-Alto</b> municipios que difieren de sus vecinos. Al seleccionar un municipio se abre su detalle en el buscador.',
  tray: 'El diagrama de flujo representa las transiciones de perfil entre elecciones consecutivas; el grosor de cada flujo es proporcional al número de municipios. El I de Moran toma valores entre −1 y 1, y los valores positivos indican que los municipios vecinos presentan comportamientos similares.',
  estados: 'Se replica el análisis con las 32 entidades federativas. El análisis de componentes principales (PCA) resume las variables en dos ejes (PC1 y PC2) para su representación gráfica; las entidades cercanas en el plano presentan características similares y las flechas indican la dirección en la que aumenta cada variable.',
  estrategias: 'Cada estrategia se asigna a un perfil cuando este cumple una condición medible. Por ejemplo, la «Promoción del voto juvenil» se asigna a los perfiles cuya brecha de participación juvenil es más amplia que la nacional; en cada estrategia se indica el dato que motiva su asignación. El simulador estima los votos adicionales que se obtendrían si aumentara la participación del grupo de 18 a 29 años en un perfil.',
  buscar: 'Al seleccionar un municipio se muestran su perfil en cada elección, su participación frente al promedio nacional y las estrategias asignadas a su perfil. También se accede a esta sección al hacer clic en un municipio del mapa o de la gráfica de distribución municipal.',
  modelo: 'Criterios para seleccionar el número de perfiles y descripción de la metodología. El coeficiente de silhouette mide la separación entre grupos y la estabilidad bootstrap su permanencia ante cambios en los datos; en ambos casos, los valores mayores son mejores. En el índice de Davies-Bouldin, los valores menores son mejores.'
};
$('#subt').textContent = `Elecciones federales de ${ANIOS[0]} a ${ULT} en ${N(D.ids.length)} municipios.`;
$('#menu').innerHTML = TABS.map(([k, l]) => `<a data-k="${k}">${ico(k)}${l}</a>`).join('');
$('#vistas').innerHTML = TABS.map(([k]) => `<section class="vista" id="v-${k}">${GUIA[k] ? `<div class="guia">${ico('info')}<div><b>Guía de lectura.</b> ${GUIA[k]}</div></div>` : ''}<div id="c-${k}"></div></section>`).join('');
const hecho = {};
function mostrar(k) {
  document.querySelectorAll('.menu a').forEach(a => a.classList.toggle('on', a.dataset.k === k));
  document.querySelectorAll('.vista').forEach(v => v.classList.toggle('on', v.id === 'v-' + k));
  // cada vez que se entra a una sección se vuelve a dibujar, para que números y gráficas se animen de nuevo
  if (hecho.actual !== k) { if (typeof pausa === 'function' && reloj) pausa(); hecho.actual = k; claveAnillos = null; requestAnimationFrame(() => { R[k](); animarSeccion($('#c-' + k)); }); }
  window.scrollTo({top: 0, behavior: 'smooth'});
}
$('#menu').onclick = e => { const a = e.target.closest('a'); if (a) mostrar(a.dataset.k); };
const card = (cls, titulo, desc, cuerpo, delay = 0) => `<div class="card ${cls}" style="animation-delay:${delay}s">${titulo ? `<h3>${titulo}</h3>` : ''}${desc ? `<div class="d">${desc}</div>` : ''}${cuerpo}</div>`;
const kpi = (grad, sh, l, v, n, icon, delay, extra = '') => `<div class="kpi s3" style="background:${grad};--sh:${sh};animation-delay:${delay}s"><div class="ic">${ico(icon, '#fff')}</div><div class="l">${l}</div><div class="v">${v}</div><div class="n">${n}</div>${extra}</div>`;
function chispa(vals, w = 104, h = 34) { const a = Math.min(...vals), b = Math.max(...vals), X = i => 4 + i / (vals.length - 1) * (w - 8), Y = v => h - 5 - (v - a) / (b - a || 1) * (h - 10); const id = 'ch' + (++uid);
  return `<svg class="chispa" viewBox="0 0 ${w} ${h}" width="${w}" height="${h}"><defs>${defsBrillo('#fff', id, 2.2, 1.6)}</defs><path class="trazo" style="animation-delay:.6s" pathLength="1" d="${vals.map((v, i) => (i ? 'L' : 'M') + X(i).toFixed(1) + ',' + Y(v).toFixed(1)).join(' ')}" fill="none" stroke="#fff" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" filter="url(#${id})"/><circle class="pto" style="animation-delay:2.6s" cx="${X(vals.length - 1)}" cy="${Y(vals.at(-1))}" r="3.4" fill="#fff"/></svg>`; }
function anillos(el, aros, centro, sub) {
  const S = 240, cx = S / 2; let s = `<svg viewBox="0 0 ${S} ${S}" style="width:100%;max-width:250px;display:block;margin:6px auto 0"><defs>${defsBrillo('', 'ba' + (++uid), 3, 1.5)}</defs>`, arcos = [];
  aros.forEach((aro, a) => { const R = aro.r, C = 2 * Math.PI * R; let acc = 0; s += `<circle cx="${cx}" cy="${cx}" r="${R}" fill="none" stroke="#eef3f7" stroke-width="${aro.w}"/>`;
    aro.items.forEach((it, i) => { const L = C * it.v / 100; s += `<circle class="arco" cx="${cx}" cy="${cx}" r="${R}" fill="none" stroke="${it.c}" stroke-width="${aro.w}" transform="rotate(-90 ${cx} ${cx})" style="stroke-dasharray:0 ${C};stroke-dashoffset:${-acc};transition:stroke-dasharray 1.5s cubic-bezier(.3,.7,.2,1) ${.3 + a * .5 + i * .22}s" filter="url(#ba${uid})" data-tip="${it.t.replace(/"/g, '&quot;')}"/>`; arcos.push([Math.max(0, L - 2), C]); acc += L; }); });
  s += `<text x="${cx}" y="${cx + 2}" text-anchor="middle" font-size="27" font-weight="800" fill="#12233f">${centro}</text><text x="${cx}" y="${cx + 22}" text-anchor="middle" font-size="11.5" fill="#4f5b6e">${sub}</text></svg>`;
  el.innerHTML = s; requestAnimationFrame(() => setTimeout(() => el.querySelectorAll('.arco').forEach((c, i) => c.style.strokeDasharray = arcos[i][0] + ' ' + arcos[i][1]), 30));
}

/* ======================= secciones ======================= */
const R = {
inicio() {
  const m = D.modelo, colVar = ['#18529D', '#28AD56', '#1d8f86', '#1fa0d8', '#7cc242', '#1a6aa6'];
  const GRUPOS = [['Participación y género', 'urna', '#18529D', v => /^Participaci|mujeres|g[ée]nero/i.test(v)], ['Brechas por edad', 'perfiles', '#28AD56', v => /^Brecha/i.test(v)], ['Padrón por edad', 'lista', '#1d8f86', v => /^% de LN con|^Proporci[óo]n de LN de/i.test(v)], ['Territorio y tamaño', 'territorio', '#1fa0d8', () => true]];
  const usadas = new Set(), grupos = GRUPOS.map(([n, ic, c, f]) => { const vs = D.variables.filter(v => !usadas.has(v[0]) && f(v[0])); vs.forEach(v => usadas.add(v[0])); return [n, ic, c, vs]; }).filter(g => g[3].length);
  $('#c-inicio').innerHTML = `<div class="grid">
   <div class="hero s12"><small>RETO 1. PERFILES CIUDADANOS DE PARTICIPACIÓN</small><h2>¿Qué tipos de ciudadanía existen según su comportamiento de participación?</h2>
   <p>Se agrupan los municipios de México en perfiles según su comportamiento de participación en las elecciones federales, con el fin de que el INE diseñe campañas focalizadas y mensajes diferenciados para cada perfil. Los perfiles describen a la población de cada municipio y no a personas individuales.</p></div>
   <div class="s4 inf-col">${[
     ['pp', 'info', '#18529D', 'Puntos porcentuales (pp)', 'Unidad que expresa la diferencia absoluta entre dos porcentajes.', ''],
     ['fuente', 'lista', '#28AD56', 'Fuente de datos', 'Información del INE sobre la participación de la ciudadanía en las elecciones federales.', ''],
     ['medicion', 'urna', '#c9972a', 'Medición de la participación', 'Proporción de la ciudadanía con registro de votación que acudió a votar.', '']
   ].map(([k, ic, c, t, d, x], n) => `<button class="inf" data-k="${k}" style="--c:${c};animation-delay:${.08 + n * .06}s"><span class="inf-ic">${ico(ic, '#fff')}</span><span class="inf-tx"><b>${t}</b><small>${d}</small>${x}<em>${ico('panel')}Ver detalle</em></span></button>`).join('')}</div>
   ${card('s8 val-card', 'Validación con el INE', 'Participación nacional calculada por el equipo frente a la reportada por la plataforma de Conteos Censales del INE, elecciones federales de ' + ANIOS[0] + ' a ' + ULT + (D.validacion.every(v => Math.abs(v[1] - v[2]) < 0.05) ? '. Los valores coinciden en las ' + D.validacion.length + ' elecciones.' : '.'), `<div class="graf" id="i-val"></div>`, .2)}
   ${card('s12', 'Variables de segmentación', `${D.variables.length} variables en ${grupos.length} grupos, calculadas para cada municipio en cada elección. Al seleccionar una variable se muestra su definición.`, `<div class="grupos-var" style="grid-template-columns:repeat(${grupos.length},1fr)">${grupos.map(([n, ic, c, vs], gi) => `<div class="grupo" style="--c:${c};animation-delay:${.3 + gi * .08}s"><h4><span>${ico(ic, '#fff')}</span>${n}</h4>${vs.map((v, vi) => `<div class="vchip${gi === 1 && vi === 0 ? ' abierto' : ''}"><div class="vt">${v[0]}<i></i></div><p>${v[1]}${v[2] ? `<em>${v[2]}</em>` : ''}</p></div>`).join('')}</div>`).join('')}</div>`, .25)}
   ${card('s12', 'Proceso metodológico', '', `<div class="pasos">${[['Integración de datos', 'Se agregan los conteos por municipio y elección, y se calculan las tasas de participación.'], ['Construcción de variables', `Se calculan ${m.feats.length} variables por municipio, se acotan los valores extremos (percentiles 1 y 99) y se estandarizan.`], ['Segmentación', `Se evalúan soluciones de 2 a 8 grupos con K-Means y se selecciona la mejor entre ${m.rango[0]} y ${m.rango.at(-1)} con base en silhouette, Davies-Bouldin, Calinski-Harabasz y estabilidad.`], ['Caracterización', `Se obtienen ${K} perfiles, cada uno con nombre, descripción, regla de clasificación y estrategias asignadas.`]].map((p, i) => `<div class="paso" style="--c:${colVar[i]}"><span>${i + 1}</span><b>${p[0]}</b>${p[1]}</div>`).join('')}</div>`, .3)}
  </div>`;
  $('#c-inicio').querySelector('.inf-col').onclick = e => { const b = e.target.closest('.inf'); if (b) abrirInfo(b.dataset.k); };
  $('#c-inicio').querySelector('.grupos-var').onclick = e => { const c = e.target.closest('.vchip'); if (c) c.classList.toggle('abierto'); };
  lineas($('#i-val'), {cats: D.validacion.map(r => r[0]), series: [{name: 'Estimación del equipo', color: '#2a9d8f', grad: ['#2c5a96', '#2a9d8f', '#4fae80'], vals: D.validacion.map(r => r[1]), w: 3.2}, {name: 'Plataforma del INE', color: '#d9a92f', vals: D.validacion.map(r => r[2]), soloPuntos: true, r: 8}], unit: '%', h: 340, yMin: 40, yMax: 66, ml: 44, dec: 1});
},
resumen() {
  const nac = D.nac, u = ULT;
  $('#c-resumen').innerHTML = `<div class="grid">
   ${kpi('var(--g1)', '', 'Perfiles ciudadanos', `<span id="k1">0</span>`, 'obtenidos con K-Means', 'perfiles', .02)}
   ${kpi('var(--g2)', '', 'Municipios analizados', `<span id="k2">0</span>`, `${ANIOS.length} elecciones federales`, 'mapa', .09)}
   ${kpi('var(--g3)', '', `Lista nominal ${u}`, `<span id="k3">0</span><small>millones</small>`, 'ciudadanía con derecho a votar', 'gente', .16)}
   ${kpi('var(--g4)', '', `Participación ${u}`, `<span id="k4">0</span><small>%</small>`, 'coincide con el INE', 'urna', .23, chispa(ANIOS.map(a => nac.part[a]), 92, 28))}
   ${card('s8', 'Participación nacional por elección', 'Porcentaje de votantes en cada elección federal', `<div class="graf" id="r-part"></div><div class="pildoras">${ANIOS.map((a, i) => `<div class="${nac.tipo[a] === 'Presidencial' ? 'pres' : ''}" style="animation-delay:${.9 + i * .1}s"><b>${nac.part[a]} %</b>${a}<br>${nac.tipo[a].toLowerCase()}</div>`).join('')}</div>`, .3)}
   ${card('s4', 'Hallazgos principales', '', `<div id="r-hall"></div>`, .36)}
   ${card('s4', 'Municipios y lista nominal por perfil', `Año ${ULT}. El anillo exterior muestra el porcentaje de la lista nominal y el interior, el porcentaje de municipios.`, `<div id="r-dona"></div><div class="leyenda" id="r-dona-l" style="justify-content:center"></div>`, .42)}
   ${card('s8', 'Distribución de la lista nominal por perfil', 'Porcentaje de la lista nominal en cada perfil, por elección', `<div class="graf" id="r-rep"></div>`, .46)}
   ${card('s7', 'Participación por grupo de edad', 'Promedio nacional. La línea continua corresponde a elecciones presidenciales y la punteada, a intermedias.', `<div class="graf" id="r-edad"></div>`, .5)}
   ${card('s5', 'Participación por tipo de sección', 'Promedio nacional en secciones urbanas, mixtas y rurales.', `<div class="graf" id="r-ts"></div>`, .54)}
  </div>`;
  contar($('#k1'), K); contar($('#k2'), D.ids.length); contar($('#k3'), nac.ln_ult, 1); contar($('#k4'), nac.part[u], 1);
  lineas($('#r-part'), {cats: ANIOS, series: [{name: 'Participación', color: '#2a9d8f', grad: ['#2c5a96', '#2a9d8f', '#4fae80'], vals: ANIOS.map(a => nac.part[a]), w: 4}], unit: '%', area: true, pulso: true, etiquetaFin: true, h: 420, yMin: 35, yMax: 70, mr: 52});
  const nm = P.reduce((a, p) => a + p.n_mun, 0);
  anillos($('#r-dona'), [{r: 100, w: 20, items: P.map((p, i) => ({v: p.pct_ln, c: p.color, t: `<b>P${i + 1} ${corto(i)}</b><br>${p.pct_ln} % de la lista nominal (${p.ln} M)`}))}, {r: 70, w: 14, items: P.map((p, i) => ({v: 100 * p.n_mun / nm, c: p.color2, t: `<b>P${i + 1} ${corto(i)}</b><br>${(100 * p.n_mun / nm).toFixed(1)} % de los municipios (${N(p.n_mun)})`}))}], nac.ln_ult + ' M', 'lista nominal ' + u);
  $('#r-dona-l').innerHTML = P.map((p, i) => `<span><i style="background:${p.color};box-shadow:0 0 8px ${p.color}"></i>P${i + 1}: ${p.pct_ln} % y ${(100 * p.n_mun / nm).toFixed(0)} %</span>`).join('');
  const colH = ['#18529D', '#28AD56', '#1d8f86', '#1fa0d8', '#7cc242', '#1a6aa6'];
  $('#r-hall').innerHTML = D.hallazgos.map((h, i) => `<div style="display:flex;gap:10px;margin:10px 0;font-size:13px;color:var(--ink2);opacity:0;animation:aparece .7s ${.5 + i * .12}s ease forwards"><span style="flex:none;width:26px;height:26px;border-radius:9px;background:${colH[i % 6]};color:#fff;display:grid;place-items:center;font-weight:700;font-size:12px;box-shadow:0 6px 14px -6px ${colH[i % 6]}">${i + 1}</span><div>${nb(h)}</div></div>`).join('');
  barrasApiladas($('#r-rep'), {cats: ANIOS, series: P.map((p, i) => ({name: 'P' + (i + 1) + ' ' + corto(i), color: p.color, vals: ANIOS.map(a => D.reparto[a][i])})), unit: '%', max: 100, h: 300});
  const tonos = ['#a9bfdf', '#6f93c9', '#2c5a96', '#7fcfc4', '#2a9d8f', '#2f7d58'];
  lineas($('#r-edad'), {cats: D.edades, series: ANIOS.map((a, i) => ({name: `${a} (${nac.tipo[a].toLowerCase()})`, color: tonos[i], vals: nac.curvas[a], dash: nac.tipo[a] === 'Intermedia', w: nac.tipo[a] === 'Intermedia' ? 2.4 : 3.2})), unit: '%', h: 300, stagger: .3});
  const tsC = {Urbana: '#2c5a96', Mixta: '#d9a92f', Rural: '#4fae80'};
  lineas($('#r-ts'), {cats: ANIOS, series: Object.entries(nac.ts).map(([k, v]) => ({name: k, color: tsC[k] || '#18529D', vals: v})), unit: '%', h: 300, area: false});
},
perfiles() {
  $('#c-perfiles').innerHTML = `<div class="grid">${P.map((p, i) => `<div class="card s6 ficha fc" style="--c:${p.osc};--c2:${p.color};animation-delay:${i * .1}s">
   <div class="cab"><small>PERFIL ${i + 1} DE ${K}</small><h4>${p.nombre}</h4><div class="ico-p">${icoPerfil(i)}</div></div>
   <div class="fc-cuerpo"><div class="mini dos"><div><b style="color:${p.osc}">${N(p.n_mun)}</b><small>municipios en ${ULT}</small></div><div><b style="color:${p.osc}">${p.ln} M</b><small>lista nominal, ${p.pct_ln} % del total nacional</small></div>
    <div><b style="color:${p.osc}">${SG(p.d_part)} pp</b><small>frente al promedio nacional</small></div><div><b style="color:${p.osc}">${SG(p.brecha_jov)} pp</b><small>brecha 18–29 años</small></div></div>
    <div><div class="fc-g">Participación por grupo de edad, ${ULT} <span>(en gris, el promedio nacional)</span></div><div class="graf" id="pc${i}"></div></div></div>
   <div class="fc-pie"><button class="ver" data-i="${i}" style="background:${p.osc};box-shadow:0 10px 20px -10px ${p.color}">${ico('panel', '#fff')}Ver ficha completa</button></div></div>`).join('')}
   ${card('s12', 'Distribución municipal de la participación por perfil', `Participación en ${ULT}; cada punto representa un municipio y la línea punteada, el promedio nacional (${D.nac.part[ULT]} %). Al hacer clic en un punto se abre el detalle del municipio.`, `<div class="graf" id="g-enj"></div>`, .4)}</div>`;
  P.forEach((p, i) => lineas($('#pc' + i), {cats: D.edades, series: [{name: 'Perfil ' + (i + 1), color: p.color, vals: p.curva, w: 3.2}, {name: 'Promedio nacional', color: '#94a0b2', vals: D.nac.curva_ult, w: 2, r: 3}], unit: '%', h: 190, area: true, ml: 34, mr: 12, leyenda: false, nt: 4}));
  $('#c-perfiles').onclick = e => { const b = e.target.closest('.ver'); if (b) abrirFicha(+b.dataset.i); };
  enjambre();
},
mapa() {
  $('#c-mapa').innerHTML = `<div class="grid">${card('s12', 'Evolución territorial de los perfiles', '', `
   <div class="ctrl" style="margin-top:10px"><button class="play" id="m-play" title="Reproducir">${ICO_PLAY}</button>
    <div class="anios"><input type="range" id="m-anio" min="0" max="${ANIOS.length - 1}" value="${ANIOS.length - 1}"><div class="marcas" id="m-marcas">${ANIOS.map((a, i) => `<span data-i="${i}">${a}</span>`).join('')}</div></div>
    Variable <select id="m-var"><option value="perfil">Perfil</option><option value="rel">Participación relativa al promedio nacional (pp)</option><option value="jov">Brecha de participación 18–29 años (pp)</option><option value="lisa">Conglomerados LISA de participación</option></select></div>
   <div class="mapa-grid"><div class="lienzo"><canvas id="m-cv"></canvas><div class="anio-big"><b id="m-txt">${ULT}</b><span id="m-sub"></span></div></div>
   <div class="panel-anillos"><div class="pa-tit" id="m-tit"></div><div id="m-anillos"></div><div class="leg-anillos" id="m-leg"></div></div></div>`, 0)}</div>`;
  mapaMun = crearMapa($('#m-cv'), GEO_MUN); mapaMun.proyectar();
  const cv = $('#m-cv');
  cv.onmousemove = e => { const id = mapaMun.buscar(e); if (!id || IDX[id] === undefined) return sinTip(); const i = IDX[id], a = ANIOS[mIdx], d = D.por_anio[a], p = d.perfil[i];
    verTip(`<b>${D.muns[i][0]}</b>, ${D.muns[i][1]}<br>${a}: ${p == null ? 'sin información de votación' : 'P' + (p + 1) + ' ' + corto(p)}${d.part[i] != null ? `<br>Participación: ${d.part[i]} % (${SG(d.rel[i])} pp respecto al promedio nacional)<br>Brecha 18–29 años: ${SG(d.jov[i])} pp` : ''}${d.lisa[i] ? '<br>LISA: ' + d.lisa[i] : ''}`, e.clientX, e.clientY); };
  cv.onmouseleave = sinTip;
  cv.onclick = e => { const id = mapaMun.buscar(e); if (id && IDX[id] !== undefined) irMunicipio(IDX[id]); };
  $('#m-anio').oninput = e => { pausa(); pintarMapa(+e.target.value); };
  $('#m-marcas').onclick = e => { if (e.target.dataset.i) { pausa(); pintarMapa(+e.target.dataset.i); } };
  $('#m-var').onchange = () => pintarMapa(mIdx, false);
  $('#m-play').onclick = () => { if (reloj) return pausa(); $('#m-play').innerHTML = ICO_PAUSA; let i = mIdx === ANIOS.length - 1 ? -1 : mIdx;
    const paso = () => { i++; if (i >= ANIOS.length) return pausa(); pintarMapa(i); }; paso(); reloj = setInterval(paso, 1900); };
  pintarMapa(ANIOS.length - 1, false);
},
tray() {
  $('#c-tray').innerHTML = `<div class="grid">${card('s12', 'Transiciones de perfil entre elecciones', 'Municipios que cambian de perfil entre elecciones consecutivas. El grosor de cada flujo es proporcional al número de municipios.', `<div class="graf" id="t-san"></div>`, 0)}
   ${card('s6', 'Trayectoria de participación por perfil', 'Tipo de trayectoria 2009–2024 según el perfil predominante de cada municipio', `<div class="graf" id="t-hist"></div>`, .1)}
   ${card('s6', 'Autocorrelación espacial', 'I de Moran global con vecindad Queen y 999 permutaciones', `<div class="graf" id="t-mor"></div>`, .2)}</div>`;
  sankey($('#t-san'));
  const tipos = [...new Set(P.flatMap(p => Object.keys(p.historial)))];
  const tcol = {'Participación alta constante': '#2f7d58', 'En ascenso': '#93d1b1', 'Estable en el promedio': '#d5dbe3', 'Volátil': '#d9a92f', 'En descenso': '#c8697f', 'Abstención persistente': '#8a2343', 'Historial incompleto': '#eef1f5'};
  const W = $('#t-hist').clientWidth || 560, H = 300, ml = 46, mr = 14, mt = 8, mb = 26, filas = P.length, bh = (H - mt - mb) / filas * .6;
  const tot = P.map(p => Object.values(p.historial).reduce((a, b) => a + b, 0)), mx = Math.max(...tot), X = escala(0, mx, ml, W - mr);
  let s = `<svg viewBox="0 0 ${W} ${H}" height="${H}"><defs>${tipos.map((_, k) => defsBrillo('', 'bh' + k, 2.2, .9)).join('')}</defs><g class="eje">${niceTicks(0, mx, 5).map(v => `<line x1="${X(v)}" x2="${X(v)}" y1="${mt}" y2="${H - mb}"/><text x="${X(v)}" y="${H - 8}" text-anchor="middle">${v}</text>`).join('')}</g>`;
  P.forEach((p, i) => { const y = mt + (i + .5) * (H - mt - mb) / filas; let acc = 0; s += `<text x="${ml - 8}" y="${y + 4}" text-anchor="end" font-size="12" font-weight="600" fill="${p.osc}">P${i + 1}</text>`;
    tipos.forEach((t, k) => { const v = p.historial[t] || 0; if (!v) return; s += `<rect class="barra-h" style="animation-delay:${i * .15 + k * .07}s" x="${X(acc)}" y="${y - bh / 2}" width="${Math.max(0, X(acc + v) - X(acc) - 2)}" height="${bh}" rx="5" fill="${tcol[t] || '#999'}" filter="url(#bh${k})" data-tip="<b>P${i + 1}</b><br>${t}: ${v} municipios"/>`; acc += v; }); });
  $('#t-hist').innerHTML = s + `</svg><div class="leyenda">${tipos.map(t => `<span><i style="background:${tcol[t] || '#999'}"></i>${t}</span>`).join('')}</div>`;
  const e = D.espacial;
  lineas($('#t-mor'), {cats: ANIOS, series: [{name: 'Participación relativa', color: '#2c5a96', vals: e.moran_part}, {name: 'Brecha 18–29 años', color: '#d9a92f', vals: e.moran_jov}], h: 300, yMin: 0, dec: 2});
},
estados() {
  const S = D.estados; S.colores = S.colores.map((_, g) => PAL[g % PAL.length]);
  $('#c-estados').innerHTML = `<div class="ctrl" style="margin-top:16px">Elección <select id="e-anio">${ANIOS.slice().reverse().map(a => `<option>${a}</option>`).join('')}</select></div>
   <div class="grid" id="e-kpis"></div>
   <div class="grid">${card('s7', 'Biplot de componentes principales por entidad', '', `<div class="d" id="e-bi-d"></div><div class="graf" id="e-bi"></div>`, .1)}
   ${card('s5', 'Distribución territorial de los segmentos', '', `<div class="ctrl" style="margin-top:6px">Mapa <select id="e-mv"><option value="seg">Segmento K-Means</option><option value="lisa">Conglomerados LISA sobre PC1</option></select></div><div class="lienzo"><canvas id="e-cv"></canvas></div><div class="leyenda" id="e-leg"></div>`, .15)}
   ${card('s5', 'Varianza explicada', 'Componentes retenidos para el K-Means (al menos 80 % de varianza acumulada)', `<div class="graf" id="e-var"></div>`, .2)}
   ${card('s7', 'Cargas de los componentes principales', 'Correlación de cada variable con PC1, PC2 y PC3', `<div id="e-car"></div>`, .25)}
   ${card('s7', 'Caracterización de los segmentos', 'Desviaciones estándar respecto al promedio de las entidades', `<div id="e-per" style="overflow-x:auto"></div>`, .3)}
   ${card('s5', 'Entidades por segmento', '', `<div class="d" id="e-lis-d"></div><div id="e-lis"></div>`, .35)}
   ${card('s12', 'Perfiles municipales en el plano de componentes principales', '', `<div class="d" id="e-mun-d"></div><div class="graf" id="e-mun"></div>`, .4)}</div>`;
  mapaEdo = crearMapa($('#e-cv'), GEO_EDO); mapaEdo.proyectar();
  $('#e-anio').onchange = dibujaEstados; $('#e-mv').onchange = dibujaEstados; dibujaEstados();
  const xs = S.ve.map((_, i) => 'PC' + (i + 1)), Wv = $('#e-var').clientWidth || 500, Hv = 270, mlv = 40, mt = 14, mb = 28, Y = escala(0, 100, Hv - mb, mt), bw = (Wv - mlv - 10) / xs.length * .6, Xv = i => mlv + (i + .5) * (Wv - mlv - 10) / xs.length;
  let s = `<svg viewBox="0 0 ${Wv} ${Hv}" height="${Hv}"><defs>${defsBrillo('#2c5a96', 'bv')}${defsBrillo('', 'bvb', 2.2, .9)}</defs><g class="eje">${[0, 20, 40, 60, 80, 100].map(v => `<line x1="${mlv}" x2="${Wv - 10}" y1="${Y(v)}" y2="${Y(v)}"/><text x="${mlv - 8}" y="${Y(v) + 4}" text-anchor="end">${v} %</text>`).join('')}${xs.map((x, i) => `<text x="${Xv(i)}" y="${Hv - 8}" text-anchor="middle">${x}</text>`).join('')}</g>`;
  s += `<line x1="${mlv}" x2="${Wv - 10}" y1="${Y(80)}" y2="${Y(80)}" stroke="#8a2343" stroke-dasharray="4 4"/>`;
  S.ve.forEach((v, i) => s += `<rect class="barra" style="animation-delay:${i * .1}s" x="${Xv(i) - bw / 2}" y="${Y(v)}" width="${bw}" height="${Y(0) - Y(v)}" rx="5" fill="${i < S.n80 ? 'url(#gv)' : '#e3e9f0'}" ${i < S.n80 ? 'filter="url(#bvb)"' : ''} data-tip="${xs[i]}: ${v} % (acumulada ${S.cum[i]} %)"/>`);
  s += `<linearGradient id="gv" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#2a9d8f"/><stop offset="1" stop-color="#2c5a96"/></linearGradient>`;
  s += `<path class="trazo" pathLength="1" d="${S.cum.map((v, i) => (i ? 'L' : 'M') + Xv(i) + ',' + Y(v)).join(' ')}" fill="none" stroke="#2c5a96" stroke-width="2.5" filter="url(#bv)"/>`;
  S.cum.forEach((v, i) => s += `<circle class="pto" style="animation-delay:${1.4 + i * .08}s" cx="${Xv(i)}" cy="${Y(v)}" r="4" fill="#fff" stroke="#2c5a96" stroke-width="2"/>`);
  $('#e-var').innerHTML = s + '</svg>';
  calor($('#e-car'), S.feats, ['PC1', 'PC2', 'PC3'], S.cargas, 1, v => v.toFixed(2), '230px');
  calor($('#e-per'), S.nombres, S.feats.map(corta), S.medz, 2, v => v.toFixed(1), '170px');
  const M = S.mun; $('#e-mun-d').textContent = `${N(M.x.length)} municipios en ${ULT}, coloreados por perfil. PC1 explica ${M.ve[0]} % y PC2 ${M.ve[1]} % de la varianza (el PCA se utiliza solo para la representación gráfica; el K-Means opera sobre las variables estandarizadas)`;
  dispersión($('#e-mun'), M.x.map((x, j) => ({x, y: M.y[j], c: P[M.perfil[j]].color, t: `<b>${M.nombre[j]}</b><br>P${M.perfil[j] + 1}`, r: 2.6})), [], M.flechas, `PC1 (${M.ve[0]} %)`, `PC2 (${M.ve[1]} %)`, 520, P.map((p, i) => [`P${i + 1} ${corto(i)}`, p.color]));
},
estrategias() {
  const sim = D.simulador;
  $('#c-estrategias').innerHTML = `<div class="grid">${card('s12', 'Simulador de incremento en la participación juvenil', 'Seleccione un perfil y ajuste el incremento con el control deslizante.', `<div class="sim2">
    <div><div class="botones" id="s-bot">${P.map((p, i) => `<button data-i="${i}" style="--c:${p.osc}">P${i + 1} ${corto(i)}<small>${N(p.n_mun)} municipios</small></button>`).join('')}</div>
     <div class="sim-et" style="margin-top:18px">Incremento en la participación de 18 a 29 años</div><div class="sim-pp"><span id="s-pp">5</span> pp</div><input type="range" id="s-sl" min="0" max="20" value="5" style="width:100%"></div>
    <div><div class="sim-et">Votos juveniles adicionales</div><div class="grande num" id="s-votos">0</div><div id="s-comp"></div></div>
    <div><div id="s-med"></div><div class="caja" id="s-bre" style="margin-top:8px;text-align:center"></div></div></div>
    <div class="sim-nota">Estimación: personas de 18 a 29 años con registro de votación en los municipios del perfil (${ULT}) multiplicadas por el incremento, bajo el supuesto de que la participación del resto de la población no cambia.</div>`, 0)}
   ${card('s12', 'Plan de acción por perfil', 'Seleccione un perfil para consultar su diagnóstico, las estrategias asignadas y los municipios donde conviene empezar.', `<div class="pa-tabs" id="pa-tabs">${P.map((p, i) => `<button data-i="${i}" style="--c:${p.osc};--c2:${p.color}"><b>P${i + 1} ${corto(i)}</b>${p.estrategias.length} estrategias, ${N(p.n_mun)} municipios</button>`).join('')}</div><div id="pa-body"></div>
    <details class="pa-cat"><summary>Ver el catálogo completo de estrategias</summary><div class="d" style="margin-top:6px">Cada punto indica que la estrategia se asigna al perfil. Al seleccionar una estrategia se muestran sus acciones y su indicador de seguimiento.</div><div class="matriz" id="s-mat" style="--k:${K}"></div></details>`, .1)}</div>`;
  const plan = i => { const p = P[i], R_ = D.nac_ref || {};
    document.querySelectorAll('#pa-tabs button').forEach(b => b.classList.toggle('on', +b.dataset.i === i));
    $('#pa-body').innerHTML = `<div class="pa-diag">${[[SG(p.d_part) + ' pp', 'participación respecto al promedio nacional', ''], [SG(p.brecha_jov) + ' pp', 'brecha de participación 18–29 años', `nacional ${SG(D.pais_jov)} pp`], [SG(p.genero) + ' pp', 'brecha entre mujeres y hombres', R_.genero != null ? `nacional ${SG(R_.genero)} pp` : '']].map(x => `<div><b style="color:${p.osc}">${x[0]}</b>${x[1]}${x[2] ? ` <em>(${x[2]})</em>` : ''}</div>`).join('')}</div>
     <div class="pa-est" style="grid-template-columns:repeat(${Math.max(2, p.estrategias.length)},1fr)">${p.estrategias.map((e, j) => `<div class="pa-card" style="--c:${p.osc};--c2:${p.color}"><div class="pa-top"><span class="ic" style="background:${GRAD_E[j % 3]}">${icoEst(e.t)}</span><b>${e.t}</b></div><div class="pa-que">${e.q || ''}</div>${e.por ? `<div class="pa-por" style="background:${p.color}14"><small>POR QUÉ SE ASIGNA</small>${e.por}</div>` : ''}<div class="pa-ind">Indicador: ${e.k}</div><div class="pa-acc"><button><span class="mm"></span><em>Ver acciones (${e.a.length})</em></button><ul>${e.a.map(x => `<li>${x}</li>`).join('')}</ul></div></div>`).join('')}</div>
     <div class="pa-h">Dónde empezar <span>Municipios del perfil con más votos juveniles recuperables</span></div>
     <div class="pa-prio">${p.prioridad.slice(0, 5).map(r => `<div><b>${r[0]}</b>${r[1]}<br><i style="color:${p.osc}">${N(r[5])}</i> votos juveniles</div>`).join('')}</div>
     <details class="pa-mas"><summary>Ver los ${p.prioridad.length} municipios con detalle</summary><div style="overflow-x:auto"><table class="pn-tabla"><tr><th>Municipio</th><th>Lista nominal</th><th>Participación</th><th>Votos recuperables</th><th>Votos juveniles</th></tr>${p.prioridad.map(r => `<tr><td><b>${r[0]}</b><small>${r[1]}</small></td><td>${N(r[2])}</td><td>${Number(r[3]).toFixed(1)} %</td><td>${N(r[4])}</td><td>${N(r[5])}</td></tr>`).join('')}</table></div></details>`;
    $('#pa-body').querySelectorAll('.pa-acc button').forEach(b => b.onclick = () => { const a = b.parentElement; a.classList.toggle('abierto'); b.querySelector('em').textContent = a.classList.contains('abierto') ? 'Ocultar acciones' : `Ver acciones (${a.querySelectorAll('li').length})`; });
  };
  $('#pa-tabs').onclick = e => { const b = e.target.closest('button'); if (b) plan(+b.dataset.i); };
  plan(P.reduce((b, p, i) => (p.brecha_jov ?? 0) < (P[b].brecha_jov ?? 0) ? i : b, 0));
  // matriz de estrategias
  const acciones = {}; P.forEach(p => p.estrategias.forEach(e => { acciones[e.t] = e; }));
  $('#s-mat').innerHTML = `<div class="fila-m cab-m"><div>Estrategia</div>${P.map((p, i) => `<span style="background:${p.osc}" data-tip="<b>P${i + 1}</b><br>${p.nombre}">P${i + 1}</span>`).join('')}</div>`
    + D.catalogo.map((c, j) => { const e = acciones[c[0]] || {a: c[3] || [], k: c[1]};
      return `<div class="fila-m"><div class="est-m"><span class="ic" style="background:${GRAD_E[j % 3]}">${icoEst(c[0])}</span><span class="tt">${c[0]}</span><i></i></div>
       ${P.map((p, i) => c[2].includes(i) ? `<span class="punto" style="background:${p.color};box-shadow:0 0 0 3px ${p.color}33,0 0 10px ${p.color}"></span>` : '<span class="punto no"></span>').join('')}
       <div class="det-m">${e.a.length ? `<ul>${e.a.map(x => `<li>${x}</li>`).join('')}</ul>` : ''}<em>Indicador de seguimiento: ${e.k}</em><em>${c[2].length ? 'Asignada a ' + c[2].map(i => `P${i + 1} ${corto(i)}`).join(', ').replace(/, ([^,]*)$/, ' y $1') + '.' : 'No se asigna a ningún perfil con los datos actuales.'}</em></div></div>`; }).join('');
  $('#s-mat').onclick = e => { const f = e.target.closest('.est-m'); if (f) f.parentElement.classList.toggle('abierto'); };
  // simulador
  let k = P.reduce((b, p, i) => (p.brecha_jov ?? 0) < (P[b].brecha_jov ?? 0) ? i : b, 0), prev = 0;
  const comp = (et, a0, a1, c, c2) => `<div class="comp"><div class="et"><span>${et}</span><b>de ${a0.toFixed(1)} % a ${a1.toFixed(1)} %</b></div><div class="pista"><i class="base" style="width:${a0}%"></i><i class="inc" style="left:${a0}%;width:${Math.max(0, a1 - a0)}%;background:linear-gradient(90deg,${c},${c2});box-shadow:0 0 10px ${c}"></i></div></div>`;
  const correr = () => { document.querySelectorAll('#s-bot button').forEach(b => b.classList.toggle('on', +b.dataset.i === k));
    const pp = +$('#s-sl').value, S = sim[k], c = P[k].color; $('#s-pp').textContent = pp; if (!S.jb || !S.base) return;
    const j0 = 100 * S.jsv / S.jb, t0 = 100 * S.sv / S.base, ex = Math.round(S.jb * pp / 100), j1 = j0 + pp, t1 = 100 * (S.sv + ex) / S.base;
    const el = $('#s-votos'), a = prev, tt = performance.now(); prev = ex; (function paso(t) { const q = Math.min(1, (t - tt) / 700); el.textContent = '+' + N(Math.round(a + (ex - a) * q)); if (q < 1) requestAnimationFrame(paso); })(tt);
    $('#s-comp').innerHTML = comp('Participación del grupo de 18 a 29 años', j0, j1, c, P[k].color2) + comp(`Participación total del perfil ${k + 1}`, t0, t1, c, P[k].color2);
    $('#s-bre').innerHTML = `<div style="font-size:12.5px;color:var(--ink2)">Brecha de participación 18–29 años, perfil ${k + 1}</div><div class="dos-v"><div><small>Actual</small><b>${SG(j0 - t0)} pp</b></div><div><small>Con el incremento</small><b>${SG(j1 - t1)} pp</b></div></div><div style="font-size:12px;color:var(--ink3)">Nivel nacional: ${SG(D.pais_jov)} pp</div>`;
    medidor($('#s-med'), j0, j1, c, P[k].color2); };
  $('#s-bot').onclick = e => { const b = e.target.closest('button'); if (b) { k = +b.dataset.i; correr(); } };
  $('#s-sl').oninput = correr; correr();
},
buscar() {
  $('#c-buscar').innerHTML = `<div class="grid">${card('s12', 'Buscador municipal', '', `<div class="ctrl" style="margin-top:10px"><input type="text" id="b-in" list="b-list" placeholder="Nombre del municipio" style="min-width:340px"><datalist id="b-list">${D.muns.map(m => `<option value="${m[0]}, ${m[1]}">`).join('')}</datalist></div><div id="b-out" style="margin-top:14px;color:var(--ink2)">Escriba el nombre de un municipio para consultar su perfil en cada elección, su participación y las estrategias asignadas.</div>`, 0)}</div>`;
  $('#b-in').oninput = e => { const i = D.muns.findIndex(m => `${m[0]}, ${m[1]}` === e.target.value); if (i >= 0) ficha(i); };
},
modelo() {
  const m = D.modelo;
  $('#c-modelo').innerHTML = `<div class="grid">${card('s12', 'Selección del número de perfiles (k)', `Índices de validación para k = 2 a 8. La franja sombreada corresponde al intervalo establecido en la convocatoria (${m.rango[0]} a ${m.rango.at(-1)}).`, `<div class="tres"><div><div class="mk">Silhouette <span>valores mayores son mejores</span></div><div class="graf" id="o-k1"></div></div><div><div class="mk">Estabilidad bootstrap (ARI) <span>valores mayores son mejores</span></div><div class="graf" id="o-k2"></div></div><div><div class="mk">Davies-Bouldin <span>valores menores son mejores</span></div><div class="graf" id="o-k3"></div></div></div>`, 0)}
   ${card('s12', 'Comparación de soluciones (k = 2 a 8)', 'Las barras muestran el valor relativo de cada índice y el valor destacado es el mejor de la columna. Las filas con borde dorado corresponden al intervalo de la convocatoria.', `<div style="overflow-x:auto"><table class="tk" id="o-tab"></table></div>`, .15)}
   ${card('s12', 'Metodología', 'Etapas del análisis, de los datos a los perfiles.', `<div class="met">${[
     ['Unidad de análisis', `Municipio por elección federal, con ${N(m.n)} observaciones. Las tasas se calculan a partir de los conteos agregados y no como promedio de porcentajes.`, 'mapa'],
     ['Variables', `${m.feats.length} variables de participación, edad, género y territorio, definidas en la sección Inicio. <span class="met-v">${m.feats.map(f => `<i>${f.replace(/ \((pp|%)\)$/, '').replace('Brecha de participación ', 'Brecha ').replace('Proporción de LN ', 'LN ').replace(', mujeres − hombres', '').replace(' (log₁₀ LN)', '')}</i>`).join('')}</span>`, 'chart'],
     ['Preparación', 'Acotamiento de valores extremos mediante winsorización en los percentiles 1 y 99, seguido de estandarización.', 'resumen'],
     ['Segmentación', `K-Means, contrastado con mezclas gaussianas (GMM) y agrupamiento jerárquico de Ward. El valor de k se selecciona entre ${m.rango[0]} y ${m.rango.at(-1)} con silhouette, Calinski-Harabasz, Davies-Bouldin y estabilidad bootstrap.`, 'perfiles'],
     ['Interpretación', `Un árbol de decisión reproduce la segmentación con ${m.fidelidad} % de fidelidad y proporciona reglas de clasificación aplicables sin el modelo.`, 'buscar'],
     ['Análisis espacial', 'Matriz de vecindad Queen entre municipios, I de Moran global e indicadores locales LISA con 999 permutaciones.', 'estados']
   ].map((x, i) => `<div class="met-c" style="animation-delay:${.2 + i * .07}s"><div class="met-h"><span class="met-n">${i + 1}</span><span class="met-ic">${ico(x[2])}</span></div><b>${x[0]}</b><p>${x[1]}</p></div>`).join('')}</div>`, .25)}</div>`;
  {
    const cols = [['Silhouette', m.sil, 1, 3], ['Estabilidad (ARI)', m.est, 1, 3], ['Davies-Bouldin', m.db, -1, 2], ['Acuerdo con GMM (ARI)', m.ari_gmm, 1, 2], ['Acuerdo con Ward (ARI)', m.ari_ward, 1, 2]];
    const colr = ['#2c5a96', '#2a9d8f', '#d9a92f', '#4fae80', '#8a2343'];
    let h = `<thead><tr><th>k</th>${cols.map(c => `<th>${c[0]}<small>${c[2] > 0 ? 'valor óptimo: máximo' : 'valor óptimo: mínimo'}</small></th>`).join('')}</tr></thead><tbody>`;
    m.k.forEach((k, i) => { const enR = k >= m.rango[0] && k <= m.rango.at(-1);
      h += `<tr class="${k === K ? 'sel' : ''}${enR ? ' rango' : ''}"><td><span class="kb">${k}</span>${k === K ? '<span class="kt">seleccionado</span>' : ''}</td>` + cols.map((c, j) => {
        const vs = c[1].filter(v => v != null), lo = Math.min(...vs), hi = Math.max(...vs), v = c[1][i];
        const best = v === (c[2] > 0 ? hi : lo), w = v == null ? 0 : 12 + 88 * (c[2] > 0 ? (v - lo) / (hi - lo || 1) : (hi - v) / (hi - lo || 1));
        return `<td><div class="vb${best ? ' best' : ''}"><span class="num">${v == null ? '—' : Number(v).toFixed(c[3])}</span><i style="--w:${w}%;--c:${colr[j]}"></i></div></td>`; }).join('') + '</tr>'; });
    $('#o-tab').innerHTML = h + '</tbody>';
    requestAnimationFrame(() => setTimeout(() => $('#o-tab').querySelectorAll('.vb i').forEach((b, n) => setTimeout(() => b.classList.add('on'), n * 18)), 300));
  }
  [['o-k1', m.sil, ['#2c5a96', '#2a9d8f'], 3], ['o-k2', m.est, ['#2a9d8f', '#4fae80'], 3], ['o-k3', m.db, ['#c9972a', '#d9a92f'], 2]].forEach(([id, vals, g, dec], j) => {
    const el = $('#' + id), H = 150, ml = 46, mr = 18, W = el.clientWidth || 560, n = m.k.length, X = i => ml + i / (n - 1) * (W - ml - mr);
    lineas(el, {cats: m.k, series: [{name: ['Silhouette', 'Estabilidad', 'Davies-Bouldin'][j], color: g[1], grad: g, vals, w: 3.4}], h: H, ml, mr, dec, nt: 3, stagger: 0});
    const iK = m.k.indexOf(K), xa = X(Math.max(0, m.k.indexOf(m.rango[0]) - .5)), xb = X(Math.min(n - 1, m.k.indexOf(m.rango.at(-1)) + .5));
    el.querySelector('svg').insertAdjacentHTML('afterbegin', `<rect x="${xa}" y="14" width="${xb - xa}" height="${H - 46}" fill="#eef3f9" rx="8"/><line x1="${X(iK)}" x2="${X(iK)}" y1="14" y2="${H - 32}" stroke="#8a2343" stroke-width="1.5" stroke-dasharray="4 4"/>`);
    const pts = el.querySelectorAll('circle.pto'); if (pts[iK]) pts[iK].insertAdjacentHTML('afterend', `<circle class="pulso" cx="${pts[iK].getAttribute('cx')}" cy="${pts[iK].getAttribute('cy')}" r="6" fill="#8a2343" opacity=".5"/><text x="${+pts[iK].getAttribute('cx') + 9}" y="${+pts[iK].getAttribute('cy') + (j === 2 ? 19 : -9)}" font-size="12" font-weight="800" fill="#8a2343" style="paint-order:stroke;stroke:#fff;stroke-width:4px">k = ${K}: ${vals[iK]}</text>`);
  });
}};

/* ---------- medidor circular del simulador ---------- */
function medidor(el, j0, j1, c, c2) {
  const R = 78, C = 2 * Math.PI * R;
  if (!el.querySelector('svg')) el.innerHTML = `<svg viewBox="0 0 200 200" style="width:210px;display:block;margin:0 auto"><defs><linearGradient id="gm" x1="0" y1="0" x2="1" y2="1"><stop offset="0" class="m1"/><stop offset="1" class="m2"/></linearGradient>${defsBrillo('', 'bm', 4, 1.8)}</defs>
    <circle cx="100" cy="100" r="${R}" fill="none" stroke="#eef3f7" stroke-width="16"/>
    <circle class="m-ahora" cx="100" cy="100" r="${R}" fill="none" stroke="url(#gm)" stroke-width="16" stroke-linecap="round" transform="rotate(-90 100 100)" filter="url(#bm)" style="stroke-dasharray:0 ${C};transition:stroke-dasharray 1.2s cubic-bezier(.2,.8,.2,1)"/>
    <circle class="m-antes" cx="100" cy="100" r="${R}" fill="none" stroke="#fff" stroke-opacity=".6" stroke-width="5" stroke-linecap="round" transform="rotate(-90 100 100)" style="stroke-dasharray:0 ${C};transition:stroke-dasharray 1.2s cubic-bezier(.2,.8,.2,1)"/>
    <text x="100" y="98" text-anchor="middle" font-size="33" font-weight="800" fill="#12233f" class="m-txt">0</text><text x="100" y="120" text-anchor="middle" font-size="10" fill="#4f5b6e">participación 18–29 años</text><text x="100" y="136" text-anchor="middle" font-size="10.5" fill="#94a0b2" class="m-sub"></text></svg>`;
  el.querySelector('.m1').setAttribute('stop-color', c); el.querySelector('.m2').setAttribute('stop-color', c2);
  el.querySelector('.m-sub').textContent = `actual: ${j0.toFixed(1)} %`;
  requestAnimationFrame(() => { el.querySelector('.m-ahora').style.strokeDasharray = `${C * Math.min(j1, 100) / 100} ${C}`; el.querySelector('.m-antes').style.strokeDasharray = `${C * j0 / 100} ${C}`; });
  const tx = el.querySelector('.m-txt'), a0 = parseFloat(tx.textContent) || 0, t0 = performance.now();
  (function paso(t) { const q = Math.min(1, (t - t0) / 900); tx.textContent = (a0 + (j1 - a0) * (1 - Math.pow(1 - q, 3))).toFixed(1) + ' %'; if (q < 1) requestAnimationFrame(paso); })(t0);
}

/* ---------- mapa municipal ---------- */
const ICO_PLAY = '<svg viewBox="0 0 24 24" fill="currentColor"><path d="M7 5v14l12-7z"/></svg>', ICO_PAUSA = '<svg viewBox="0 0 24 24" fill="currentColor"><path d="M6 5h4v14H6zM14 5h4v14h-4z"/></svg>';
const IDX = {}; D.ids.forEach((c, i) => IDX[c] = i);
const GEO_MUN = GEO, GEO_EDO = GEOE;
let mapaMun = null, mapaEdo = null, mIdx = ANIOS.length - 1, reloj = null;
const LISA = {'Alto-Alto': '#2c5a96', 'Bajo-Bajo': '#8a2343', 'Alto-Bajo': '#a9bfdf', 'Bajo-Alto': '#d9a0ad', 'No significativo': '#e9eef3'};
function pausa() { clearInterval(reloj); reloj = null; const b = $('#m-play'); if (b) b.innerHTML = ICO_PLAY; }
function pintarMapa(i, animar = true) {
  mIdx = i; const a = ANIOS[i], d = D.por_anio[a], v = $('#m-var').value, SIN = '#e9eef3';
  $('#m-anio').value = i; $('#m-txt').textContent = a; $('#m-sub').textContent = `Elección ${D.nac.tipo[a].toLowerCase()}, participación nacional de ${D.nac.part[a]} %`;
  document.querySelectorAll('#m-marcas span').forEach((s, j) => s.classList.toggle('on', j === i));
  let colorDe, items, tit;
  const sinDato = d.part.filter(x => x == null).length;
  if (v === 'perfil') { colorDe = id => { const k = IDX[id]; const p = k === undefined ? null : d.perfil[k]; return p == null ? SIN : P[p].color; };
    const n = P.map(() => 0); d.perfil.forEach(p => { if (p != null) n[p]++; });
    items = P.map((p, k) => ({l: `P${k + 1} ${corto(k)}`, n: n[k], c: p.color})).concat([{l: 'Sin información de votación', n: d.perfil.filter(p => p == null).length, c: '#cfd8e3'}]); tit = `Municipios por perfil en ${a}`; }
  else if (v === 'lisa') { colorDe = id => { const k = IDX[id]; const l = k === undefined ? null : d.lisa[k]; return l ? LISA[l] || SIN : SIN; };
    items = Object.entries(LISA).map(([l, c]) => ({l, n: d.lisa.filter(x => x === l).length, c: c === '#e9eef3' ? '#cfd8e3' : c})); tit = `Conglomerados LISA en ${a}`; }
  else { const lim = v === 'rel' ? 15 : 12; colorDe = id => { const k = IDX[id]; const x = k === undefined ? null : d[v][k]; return x == null ? SIN : divergente(x, lim); };
    const mas = d[v].filter(x => x != null && x > 0).length, menos = d[v].filter(x => x != null && x <= 0).length;
    items = v === 'rel' ? [{l: 'Participación superior al promedio nacional', n: mas, c: '#2c5a96'}, {l: 'Participación inferior al promedio nacional', n: menos, c: '#8a2343'}, {l: 'Sin información de votación', n: sinDato, c: '#cfd8e3'}]
      : [{l: 'Participación juvenil superior al total municipal', n: mas, c: '#2c5a96'}, {l: 'Participación juvenil inferior al total municipal', n: menos, c: '#8a2343'}, {l: 'Sin información de votación', n: d[v].filter(x => x == null).length, c: '#cfd8e3'}];
    tit = v === 'rel' ? `Participación relativa en ${a}` : `Brecha de participación 18–29 años en ${a}`; }
  $('#m-tit').textContent = tit;
  anillosMapa(items, v);
  $('#m-leg').innerHTML = items.map(it => `<div><i style="background:${it.c};box-shadow:0 0 8px ${it.c}"></i><span>${it.l}</span><b class="num">0</b></div>`).join('')
    + (v === 'rel' || v === 'jov' ? `<div class="escala"><span>menor</span><i></i><span>mayor</span></div>` : '');
  $('#m-leg').querySelectorAll('b.num').forEach((b, k) => contar(b, items[k].n, 0, 1100));
  if (animar) mapaMun.animar(colorDe, 1100); else mapaMun.dibujar(colorDe);
}
let claveAnillos = null;
function anillosMapa(items, clave) {
  const el = $('#m-anillos'), S = 270, cx = S / 2, w = items.length > 3 ? 15 : 20, gap = 6, R0 = cx - 14, mx = Math.max(1, ...items.map(it => it.n)), tot = items.reduce((a, it) => a + it.n, 0);
  if (claveAnillos !== clave) { claveAnillos = clave; const id = 'bm' + (++uid);
    let s = `<svg viewBox="0 0 ${S} ${S}" style="width:100%;max-width:270px;display:block;margin:auto;overflow:visible"><defs>${defsBrillo('', id, 2.8, 1.6)}</defs>`;
    items.forEach((it, k) => { const R = R0 - k * (w + gap), C = 2 * Math.PI * R;
      s += `<circle cx="${cx}" cy="${cx}" r="${R}" fill="none" stroke="#eef3f7" stroke-width="${w}" stroke-linecap="round" stroke-dasharray="${C * .75} ${C}" transform="rotate(-90 ${cx} ${cx})"/>`
        + `<circle class="aro" data-c="${C}" cx="${cx}" cy="${cx}" r="${R}" fill="none" stroke-width="${w}" stroke-linecap="round" transform="rotate(-90 ${cx} ${cx})" filter="url(#${id})" style="stroke-dasharray:0 ${C};transition:stroke-dasharray 1.1s cubic-bezier(.3,.7,.2,1),stroke .6s"/>`
        + `<text class="aro-n" x="${cx - 16}" y="${cx - R + 4}" text-anchor="end" font-size="11.5" font-weight="700" fill="#4f5b6e">0</text>`; });
    const Ri = R0 - items.length * (w + gap) + gap;
    s += `<text class="aro-t" x="${cx}" y="${cx + 5}" text-anchor="middle" font-size="${Ri > 45 ? 22 : 16}" font-weight="800" fill="#12233f">0</text><text x="${cx}" y="${cx + (Ri > 45 ? 22 : 18)}" text-anchor="middle" font-size="9.5" fill="#4f5b6e">municipios</text></svg>`;
    el.innerHTML = s; }
  const aros = el.querySelectorAll('.aro'), nums = el.querySelectorAll('.aro-n');
  requestAnimationFrame(() => items.forEach((it, k) => { const C = +aros[k].dataset.c; aros[k].style.stroke = it.c; aros[k].style.strokeDasharray = `${Math.max(.001, C * .75 * it.n / mx)} ${C}`; aros[k].innerHTML = `<title>${it.l}: ${it.n}</title>`; contar(nums[k], it.n, 0, 1100); }));
  contar(el.querySelector('.aro-t'), tot, 0, 1100);
}
/* ---------- ficha completa en panel lateral ---------- */
const IEST = [[/Primer voto|juvenil/, '<path d="M2 9l10-5 10 5-10 5z"/><path d="M6 11v5c0 1.5 2.7 3 6 3s6-1.5 6-3v-5"/>'], [/rural|territorial|rurales/i, '<path d="M12 21s-7-6.2-7-11a7 7 0 0 1 14 0c0 4.8-7 11-7 11z"/><circle cx="12" cy="10" r="2.5"/>'], [/mayores/i, '<circle cx="12" cy="4.5" r="2"/><path d="M12 7v6M8 9.5h8M10 21l2-8 2 8"/>'], [/Ciudad|urbanas/, '<rect x="7" y="2" width="10" height="20" rx="2.5"/><path d="M11 18h2M10 6h4"/>'], [/Reactivaci|abstención/, '<path d="M21 12a9 9 0 1 1-3-6.7L21 8"/><path d="M21 3v5h-5"/>'], [/Igualdad|género/, '<path d="M12 3v18M6 21h12M4 7h16"/><path d="M7 7l-3 7a3 3 0 0 0 6 0zM17 7l-3 7a3 3 0 0 0 6 0z"/>'], [/Semillero|Consolidación/, '<path d="M12 21v-8"/><path d="M12 13c0-4 3-7 8-7 0 4-3 7-8 7zM12 15c0-3-2.5-5.5-7-5.5 0 3 2.5 5.5 7 5.5z"/>']];
const icoEst = t => `<svg viewBox="0 0 24 24" fill="none" stroke="#fff" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">${(IEST.find(([r]) => r.test(t)) || [0, ICO.estrategias])[1]}</svg>`;
const GRAD_E = ['linear-gradient(135deg,#18529D,#1fa0d8)', 'linear-gradient(135deg,#1c8c6a,#28AD56)', 'linear-gradient(135deg,#18529D,#28AD56)'];
function abrirFicha(i) {
  const p = P[i], acierto = ((p.regla || '').match(/\((\d+) ?% de acierto\)/) || [])[1] || '';
  const conds = (p.regla || '').replace(/^Si /, '').replace(/\s*\(\d+ ?% de acierto\)\s*$/, '').split(/\s+y\s+(?=[%BPT])/);
  $('#panel').style.setProperty('--c', p.color);
  $('#panel').innerHTML = `<div class="cab" style="background:linear-gradient(120deg,${p.osc},${p.color})"><small>PERFIL ${i + 1} DE ${K}</small><h4>${p.nombre}</h4><button class="cerrar" title="Cerrar">×</button></div>
   <div class="pn-cuerpo"><div class="mini">${[[N(p.n_mun), `municipios en ${ULT}`], [p.ln + ' M', `lista nominal, ${p.pct_ln} % del total nacional`], [SG(p.d_part) + ' pp', 'frente al promedio nacional, 2009–2024'], [SG(p.brecha_jov) + ' pp', 'brecha de participación 18–29 años']].map(x => `<div><b style="color:${p.osc}">${x[0]}</b><small>${x[1]}</small></div>`).join('')}</div>
   <h5>Descripción del perfil</h5><p class="pn-txt">${nb(p.narrativa)}</p>
   <h5>Regla de clasificación</h5><div>${conds.map(c => `<span class="cond">${c}</span>`).join('')}</div>${acierto ? `<div class="pn-nota">Árbol de decisión con ${acierto} % de acierto para este perfil</div>` : ''}
   <h5>Estrategias asignadas</h5><div class="pn-est">${p.estrategias.map((e, j) => `<div class="ech${j === 0 ? ' abierto' : ''}"><div class="ech-t"><span class="ic" style="background:${GRAD_E[j % 3]}">${icoEst(e.t)}</span>${e.t}<i></i></div><div class="ech-c">${e.q ? `<p class="pn-txt" style="margin:0 0 6px">${e.q}</p>` : ''}${e.por ? `<div class="pa-por" style="background:${p.color}14;margin-bottom:8px"><small>POR QUÉ SE ASIGNA</small>${e.por}</div>` : ''}<ul>${e.a.map(x => `<li>${x}</li>`).join('')}</ul><em>Indicador: ${e.k}</em></div></div>`).join('')}</div>
   <h5>Municipios prioritarios</h5><div class="pn-nota" style="margin-top:0">Ordenados por el número de votos juveniles recuperables</div>
   <div style="overflow-x:auto"><table class="pn-tabla"><tr><th>Municipio</th><th>Participación</th><th>Votos recuperables</th><th>Votos juveniles</th></tr>${p.prioridad.map(r => `<tr><td><b>${r[0]}</b><small>${r[1]}, LN de ${N(r[2])}</small></td><td>${Number(r[3]).toFixed(1)} %</td><td>${N(r[4])}</td><td>${N(r[5])}</td></tr>`).join('')}</table></div></div>`;
  $('#panel').scrollTop = 0; $('#velo').classList.add('on'); $('#panel').classList.add('on');
}
function abrirInfo(k) {
  const jov = D.nac.part[ULT] + (D.pais_jov || 0), varsPP = D.variables.filter(v => /\(pp\)/.test(v[0])).map(v => v[0].replace(/\s*\(pp\)/, ''));
  const C = {
    pp: ['#18529D', '#1a6aa6', 'DEFINICIÓN', 'Puntos porcentuales (pp)', `
      <p class="pn-txt">El punto porcentual (pp) es la unidad con la que se expresa la <b>diferencia absoluta entre dos magnitudes medidas en porcentaje</b>. Se obtiene restando un porcentaje de otro, por lo que el resultado se lee en la misma escala de 0 a 100.</p>
      <p class="pn-txt" style="margin-top:8px">El resultado conserva el signo, de modo que un valor positivo indica que el primer porcentaje es mayor que el de referencia y uno negativo indica lo contrario.</p>
      <h5>Ejemplo con los datos de ${ULT}</h5>
      <div class="mini">${[[D.nac.part[ULT] + ' %', 'participación nacional'], [jov.toFixed(1) + ' %', 'participación del grupo de 18 a 29 años'], [SG(D.pais_jov) + ' pp', 'brecha del grupo de 18 a 29 años']].map(x => `<div><b style="color:#18529D">${x[0]}</b><small>${x[1]}</small></div>`).join('')}</div>
      <h5>Variables del estudio expresadas en pp</h5><ul class="lista">${varsPP.map(v => `<li>${v}</li>`).join('')}</ul>`],
    fuente: ['#1d8f86', '#28AD56', 'DATOS', 'Fuente de datos', `
      <p class="pn-txt">La información proviene de los Conteos Censales de Participación Ciudadana del Instituto Nacional Electoral, que reportan, para cada sección electoral, cuántas personas de la lista nominal votaron y cuántas no, desagregadas por sexo y grupo de edad.</p>
      <div class="mini">${[[ANIOS.length, 'elecciones federales'], [N(D.ids.length), 'municipios con cartografía del INEGI'], [D.nac.ln_ult + ' M', 'personas en la lista nominal de ' + ULT]].map(x => `<div><b style="color:#1d8f86">${x[0]}</b><small>${x[1]}</small></div>`).join('')}</div>
      <h5>Elecciones analizadas</h5><div>${ANIOS.map(a => `<span class="cond">${a}</span>`).join('')}</div>
      <h5>Características</h5><ul class="lista"><li>Los conteos se agregan del nivel de sección al nivel municipal.</li><li>Se trabaja con datos agregados, sin información de personas individuales.</li><li>La cartografía municipal del INEGI se usa para el mapa y el análisis espacial (LISA).</li></ul>`],
    medicion: ['#8a6512', '#c9972a', 'MÉTODO', 'Medición de la participación', `
      <p class="pn-txt">La participación se define como la proporción de la ciudadanía con registro de votación que acudió a votar, y se calcula para cada municipio, elección, sexo y grupo de edad.</p>
      <div class="formula pn-form">Participación = SV / (SV + NV)</div>
      <h5>Componentes</h5><ul class="lista"><li><b>LN</b>, lista nominal (ciudadanía con derecho a votar)</li><li><b>SV</b>, sí votó</li><li><b>NV</b>, no votó</li><li><b>NS</b>, sin información de votación</li></ul>
      <p class="pn-nota">Los casos NS se excluyen del denominador, por lo que la tasa se calcula solo sobre la ciudadanía con registro de votación.</p>
      <h5>Participación nacional por elección</h5><table class="pn-tabla"><tr><th>Elección</th><th>Participación</th></tr>${ANIOS.map(a => `<tr><td>${a}</td><td>${D.nac.part[a]} %</td></tr>`).join('')}</table>`]
  }[k];
  $('#panel').style.setProperty('--fg', `linear-gradient(120deg,${C[0]},${C[1]})`); $('#panel').style.setProperty('--fs', C[1]);
  $('#panel').innerHTML = `<div class="cab no-pp" style="background:linear-gradient(120deg,${C[0]},${C[1]})"><small>${C[2]}</small><h4>${C[3]}</h4><button class="cerrar" title="Cerrar">×</button></div><div class="pn-cuerpo no-pp">${C[4]}</div>`;
  $('#panel').scrollTop = 0; $('#velo').classList.add('on'); $('#panel').classList.add('on');
}
function cerrarFicha() { $('#velo').classList.remove('on'); $('#panel').classList.remove('on'); }
$('#velo').onclick = cerrarFicha;
$('#panel').onclick = e => { if (e.target.closest('.cerrar')) return cerrarFicha(); const t = e.target.closest('.ech-t'); if (t) t.parentElement.classList.toggle('abierto'); };
addEventListener('keydown', e => { if (e.key === 'Escape') cerrarFicha(); });
/* ---------- enlace de "pp" a su definición ---------- */
function irPP() { abrirInfo('pp'); }
function marcarPP(raiz) {
  const w = document.createTreeWalker(raiz, NodeFilter.SHOW_TEXT, {acceptNode: n => (/\bpp\b/.test(n.nodeValue) && !n.parentElement.closest('svg,script,style,select,option,textarea,.pp-link,#def-pp,.tip,.vchip,.grupos-var,.no-pp,.inf')) ? NodeFilter.FILTER_ACCEPT : NodeFilter.FILTER_REJECT});
  const nodos = []; while (w.nextNode()) nodos.push(w.currentNode);
  nodos.forEach(n => { const f = document.createDocumentFragment(); n.nodeValue.split(/\b(pp)\b/).forEach((s, i) => { if (i % 2) { const a = document.createElement('span'); a.className = 'pp-link'; a.textContent = 'pp'; a.dataset.tip = 'Puntos porcentuales: diferencia entre dos porcentajes. Clic para ver la definición.'; f.appendChild(a); } else if (s) f.appendChild(document.createTextNode(s)); }); n.replaceWith(f); });
}
new MutationObserver(ms => ms.forEach(m => m.addedNodes.forEach(n => { if (n.nodeType === 1) marcarPP(n); else if (n.nodeType === 3 && n.parentElement) marcarPP(n.parentElement); }))).observe(document.body, {childList: true, subtree: true});
document.addEventListener('click', e => { if (e.target.closest('.pp-link')) { e.preventDefault(); e.stopPropagation(); irPP(); } }, true);
function irMunicipio(i) { cerrarFicha(); pausa(); mostrar('buscar'); setTimeout(() => { $('#b-in').value = `${D.muns[i][0]}, ${D.muns[i][1]}`; ficha(i); }, 60); }
function ficha(i) {
  const m = D.muns[i], per = ANIOS.map(a => D.por_anio[a].perfil[i]), ult = per.at(-1), p = ult == null ? null : P[ult], du = D.por_anio[ULT];
  $('#b-out').innerHTML = `<div style="display:grid;grid-template-columns:1fr 1.2fr;gap:24px" class="bus">
   <div><div style="font-size:26px;font-weight:800;color:var(--ink);letter-spacing:-.5px">${m[0]}</div><div>${m[1]}</div>
    ${p ? `<span class="etq" style="background:linear-gradient(90deg,${p.osc},${p.color})">P${ult + 1} ${corto(ult)}</span>` : '<span class="etq" style="background:#94a0b2">Sin información de votación en ' + ULT + '</span>'}
    <div class="mini" style="grid-template-columns:1fr 1fr;margin-top:14px"><div><b>${du.part[i] ?? '—'} %</b><small>participación en ${ULT} (promedio nacional: ${D.nac.part[ULT]} %)</small></div><div><b>${SG(du.jov[i])} pp</b><small>brecha de participación 18–29 años (nacional: ${SG(D.pais_jov)} pp)</small></div></div>
    <div style="margin-top:12px;font-size:13px">Trayectoria 2009–2024: <b style="color:var(--ink)">${m[2] || '—'}</b></div>
    <div class="chips">${per.map((q, j) => `<span style="background:${q == null ? '#d5dde7' : P[q].osc};${q == null ? 'color:#4f5b6e' : ''}">${q == null ? 's/d' : 'P' + (q + 1)}<small>${ANIOS[j]}</small></span>`).join('')}</div></div>
   <div><div style="font-weight:600;font-size:13px;color:var(--ink)">Participación del municipio y promedio nacional</div><div class="graf" id="b-graf"></div></div></div>
   ${p ? `<div style="margin-top:12px;font-weight:700;color:var(--ink)">Estrategias asignadas a su perfil</div><div class="est">${p.estrategias.map(e => `<div style="border-top:4px solid ${p.color}"><b>${e.t}</b><ul>${e.a.map(x => `<li>${x}</li>`).join('')}</ul></div>`).join('')}</div>` : ''}`;
  lineas($('#b-graf'), {cats: ANIOS, series: [{name: m[0], color: p ? p.color : '#2c5a96', vals: ANIOS.map(a => D.por_anio[a].part[i]), w: 3.2}, {name: 'Promedio nacional', color: '#94a0b2', vals: ANIOS.map(a => D.nac.part[a]), w: 2, r: 3}], unit: '%', h: 250, area: true});
}

/* ---------- gráfica de puntos ---------- */
function enjambre() {
  const el = $('#g-enj'), W = el.clientWidth || 1100, filaH = 112, ml = 210, mr = 20, top = 8, r = 2.4, x0 = 20, x1 = 100, d = D.por_anio[ULT], H = top + filaH * K + 28, nac = D.nac.part[ULT];
  const X = v => ml + (Math.max(x0, Math.min(x1, v)) - x0) / (x1 - x0) * (W - ml - mr);
  let s = `<svg viewBox="0 0 ${W} ${H}" height="${H}"><defs>${P.map((p, k) => defsBrillo(p.color, 'be' + k)).join('')}</defs><g class="eje">`;
  for (let v = x0; v <= x1; v += 10) s += `<line x1="${X(v)}" x2="${X(v)}" y1="${top}" y2="${H - 24}"/><text x="${X(v)}" y="${H - 6}" text-anchor="middle">${v} %</text>`;
  s += `</g><line x1="${X(nac)}" x2="${X(nac)}" y1="${top}" y2="${H - 24}" stroke="#12233f" stroke-width="1.5" stroke-dasharray="5 4"/>`;
  const pts = [];
  for (let k = 0; k < K; k++) {
    const cy = top + filaH * (k + .5), ms = D.ids.map((_, i) => i).filter(i => d.perfil[i] === k && d.part[i] != null).sort((a, b) => d.part[a] - d.part[b]), fila = [], sep = r * 2 + .6;
    s += `<text x="0" y="${cy - 4}" font-size="13" font-weight="700" fill="${P[k].osc}">P${k + 1} ${corto(k)}</text><text x="0" y="${cy + 14}" font-size="12" fill="#94a0b2">${N(ms.length)} municipios</text>`;
    ms.forEach(i => { const x = X(d.part[i]); let dy = 0; for (let t = 0; t < 90; t++) { const c = t === 0 ? 0 : (t % 2 ? 1 : -1) * Math.ceil(t / 2) * sep; dy = c; if (!fila.some(q => Math.abs(q[0] - x) < sep && Math.abs(q[1] - c) < sep)) break; } fila.push([x, dy]); pts.push({i, x, cy, dy, k}); });
    const mx = Math.max(1, ...fila.map(f => Math.abs(f[1]))), f = Math.min(1, (filaH / 2 - 6) / mx); pts.filter(p => p.k === k).forEach(p => p.dy *= f);
  }
  s += P.map((p, k) => `<g filter="url(#be${k})">` + pts.map((q, j) => q.k !== k ? '' : `<circle data-j="${j}" cx="${q.x}" cy="${q.cy + q.dy}" r="${r}" fill="${p.color}" style="transform:translate(${(X(nac) - q.x) * .6}px,${-q.dy}px);opacity:0;transition:transform 2.2s cubic-bezier(.2,.8,.2,1) ${(q.x / W) * .8}s,opacity 1.2s ${(q.x / W) * .8}s" data-tip="<b>${D.muns[q.i][0]}</b>, ${D.muns[q.i][1]}<br>Participación en ${ULT}: ${d.part[q.i]} %<br>Perfil P${k + 1}"/>`).join('') + '</g>').join('') + '</svg>';
  el.innerHTML = s;
  setTimeout(() => el.querySelectorAll('circle[data-j]').forEach(c => { c.style.transform = 'none'; c.style.opacity = 1; }), 200);
  el.querySelector('svg').onclick = e => { const j = e.target.dataset && e.target.dataset.j; if (j !== undefined) { sinTip(); irMunicipio(pts[j].i); } };
}

/* ---------- sankey ---------- */
function sankey(el) {
  const s0 = D.sankey, W = el.clientWidth || 1100, H = 440, nA = ANIOS.length, colW = 16, padY = 12, mt = 10, mb = 26;
  const val = s0.nodos.map(() => 0), vin = s0.nodos.map(() => 0);
  s0.links.source.forEach((s, j) => { val[s] += s0.links.value[j]; vin[s0.links.target[j]] += s0.links.value[j]; });
  const v = val.map((x, i) => Math.max(x, vin[i])), tot = Math.max(...ANIOS.map((_, a) => v.slice(a * K, a * K + K).reduce((p, q) => p + q, 0)));
  const esc = (H - mt - mb - padY * (K - 1)) / tot, X = a => 60 + a * (W - 120 - colW) / (nA - 1), pos = [];
  ANIOS.forEach((_, a) => { let y = mt; for (let k = 0; k < K; k++) { const i = a * K + k, h = v[i] * esc; pos[i] = {x: X(a), y, h, so: 0, ti: 0}; y += h + padY; } });
  let s = `<svg viewBox="0 0 ${W} ${H}" height="${H}"><defs>`;
  s0.links.source.forEach((sr, j) => { const tg = s0.links.target[j]; s += `<linearGradient id="sk${j}" x1="0" x2="1"><stop offset="0" stop-color="${P[sr % K].color}"/><stop offset="1" stop-color="${P[tg % K].color}"/></linearGradient>`; });
  s += '</defs>';
  s0.links.source.forEach((sr, j) => { const tg = s0.links.target[j], w = s0.links.value[j] * esc, a = pos[sr], b = pos[tg];
    const y1 = a.y + a.so + w / 2, y2 = b.y + b.ti + w / 2; a.so += w; b.ti += w; const xa = a.x + colW, xb = b.x, cx = (xa + xb) / 2;
    s += `<path d="M${xa},${y1} C${cx},${y1} ${cx},${y2} ${xb},${y2}" fill="none" stroke="url(#sk${j})" stroke-width="${Math.max(1, w)}" stroke-opacity=".38" style="opacity:0;animation:sube 1.4s ${.6 + (sr / K | 0) * .35}s ease forwards" data-tip="<b>Del P${sr % K + 1} en ${ANIOS[sr / K | 0]} al P${tg % K + 1} en ${ANIOS[tg / K | 0]}</b><br>${N(s0.links.value[j])} municipios"/>`; });
  pos.forEach((p, i) => { const k = i % K, a = i / K | 0; s += `<rect x="${p.x}" y="${p.y}" width="${colW}" height="${Math.max(1, p.h)}" rx="5" fill="${P[k].color}" style="opacity:0;animation:sube .8s ${a * .35}s ease forwards" filter="url(#bs${k})" data-tip="<b>P${k + 1} en ${ANIOS[a]}</b><br>${N(Math.round(v[i]))} municipios"/>`; });
  s = s.replace('</defs>', P.map((p, k) => defsBrillo(p.color, 'bs' + k)).join('') + '</defs>');
  ANIOS.forEach((a, i) => s += `<text x="${X(i) + colW / 2}" y="${H - 6}" text-anchor="middle" class="txt" style="font-weight:600">${a}</text>`);
  el.innerHTML = s + `</svg><div class="leyenda">${P.map((p, k) => `<span><i style="background:${p.color};box-shadow:0 0 8px ${p.color}"></i>P${k + 1} ${corto(k)}</span>`).join('')}</div>`;
}

/* ---------- dispersión (biplot) ---------- */
function dispersión(el, pts, hulls, flechas, xl, yl, H, ley) {
  const W = el.clientWidth || 700, ml = 46, mr = 20, mt = 14, mb = 40;
  const xs = pts.map(p => p.x).concat(flechas.map(f => f[1])), ys = pts.map(p => p.y).concat(flechas.map(f => f[2]));
  const xa = Math.min(...xs), xb = Math.max(...xs), ya = Math.min(...ys), yb = Math.max(...ys), px = (xb - xa) * .08, py = (yb - ya) * .08;
  const X = escala(xa - px, xb + px, ml, W - mr), Y = escala(ya - py, yb + py, H - mb, mt);
  let s = `<svg viewBox="0 0 ${W} ${H}" height="${H}"><defs><marker id="fl${++uid}" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#12233f"/></marker></defs><g class="eje">`;
  niceTicks(xa - px, xb + px, 6).forEach(v => s += `<line x1="${X(v)}" x2="${X(v)}" y1="${mt}" y2="${H - mb}"/><text x="${X(v)}" y="${H - mb + 16}" text-anchor="middle">${v}</text>`);
  niceTicks(ya - py, yb + py, 5).forEach(v => s += `<line x1="${ml}" x2="${W - mr}" y1="${Y(v)}" y2="${Y(v)}"/><text x="${ml - 8}" y="${Y(v) + 4}" text-anchor="end">${v}</text>`);
  s += `<text x="${(ml + W - mr) / 2}" y="${H - 4}" text-anchor="middle">${xl}</text><text x="12" y="${(mt + H - mb) / 2}" transform="rotate(-90 12 ${(mt + H - mb) / 2})" text-anchor="middle">${yl}</text></g>`;
  s += `<line x1="${X(0)}" x2="${X(0)}" y1="${mt}" y2="${H - mb}" stroke="#d5dde7"/><line x1="${ml}" x2="${W - mr}" y1="${Y(0)}" y2="${Y(0)}" stroke="#d5dde7"/>`;
  hulls.forEach(h => { if (h.p) s += `<path d="${h.p[0].map((x, i) => (i ? 'L' : 'M') + X(x) + ',' + Y(h.p[1][i])).join(' ')}Z" fill="${h.c}22" stroke="${h.c}" stroke-dasharray="4 4" style="opacity:0;animation:sube 1.2s .8s ease forwards"/>`; });
  pts.forEach((p, i) => s += `<circle cx="${X(p.x)}" cy="${Y(p.y)}" r="${p.r || 7}" fill="${p.c}" ${p.r ? '' : 'stroke="#fff" stroke-width="2"'} style="opacity:0;animation:sube .8s ${Math.min(1.5, i * (p.r ? .0006 : .03))}s ease forwards" data-tip="${p.t.replace(/"/g, '&quot;')}"/>${p.l ? `<text x="${X(p.x)}" y="${Y(p.y) - 11}" text-anchor="middle" font-size="10.5" fill="#4f5b6e" style="opacity:0;animation:sube .8s 1s ease forwards">${p.l}</text>` : ''}`);
  flechas.forEach(f => s += `<line x1="${X(0)}" y1="${Y(0)}" x2="${X(f[1])}" y2="${Y(f[2])}" stroke="#12233f" stroke-width="1.5" marker-end="url(#fl${uid})" class="trazo" pathLength="1"/><text x="${X(f[1])}" y="${Y(f[2]) + (f[2] >= 0 ? -6 : 14)}" text-anchor="${f[1] >= 0 ? 'start' : 'end'}" font-size="11" font-weight="600" fill="#12233f" style="paint-order:stroke;stroke:#fff;stroke-width:3px">${f[0]}</text>`);
  el.innerHTML = s + '</svg>' + (ley ? `<div class="leyenda">${ley.map(l => `<span><i style="background:${l[1]};box-shadow:0 0 8px ${l[1]}"></i>${l[0]}</span>`).join('')}</div>` : '');
}
function dibujaEstados() {
  const S = D.estados, a = $('#e-anio').value, d = S.anios[a], mv = $('#e-mv').value;
  $('#e-kpis').innerHTML = [['var(--g1)', '', 'Segmentos de entidades', S.k, 'K-Means sobre componentes principales', 'estados'], ['var(--g2)', '', 'Varianza en PC1 + PC2', (S.ve[0] + S.ve[1]).toFixed(1) + ' %', 'de la varianza total', 'chart'], ['var(--g3)', '', 'Silhouette', S.sil, 'calidad de la segmentación', 'resumen'], ['var(--g4)', '', 'I de Moran de PC1', d.moran, `${a}, p = ${d.p}`, 'mapa']].map((x, i) => kpi(x[0], x[1], x[2], x[3], x[4], x[5], i * .07)).join('');
  $('#e-kpis').querySelectorAll('.kpi .v').forEach(v => animarTexto(v, 1600));
  $('#e-bi-d').textContent = `Año ${a}. Cada punto representa una entidad, los contornos delimitan los segmentos y las flechas indican la dirección de cada variable. PC1 se asocia con ${S.pc_desc[0]}, y PC2 con ${S.pc_desc[1]}.`;
  dispersión($('#e-bi'), d.pc1.map((x, i) => ({x, y: d.pc2[i], c: S.colores[d.seg[i]], l: d.estado[i], t: `<b>${d.estado[i]}</b><br>${S.nombres[d.seg[i]]}<br>Participación: ${d.part[i]} % (${SG(d.rel[i])} pp respecto al promedio nacional)`})), S.nombres.map((n, g) => ({p: d.hull[g], c: S.colores[g]})), S.flechas, `PC1 (${S.ve[0]} %)`, `PC2 (${S.ve[1]} %)`, 560, S.nombres.map((n, g) => [n, S.colores[g]]));
  const col = {}; d.edo.forEach((e, i) => col[e] = mv === 'seg' ? S.colores[d.seg[i]] : (LISA[d.lisa[i]] || '#e9eef3'));
  mapaEdo.dibujar(id => col[id] || '#e9eef3', 1, null, '#ffffff', 1.2);
  const cv = $('#e-cv'); cv.onmousemove = e => { const id = mapaEdo.buscar(e), i = d.edo.indexOf(id); if (i < 0) return sinTip(); verTip(`<b>${d.estado[i]}</b><br>${S.nombres[d.seg[i]]}<br>LISA sobre PC1: ${d.lisa[i]}<br>Participación: ${d.part[i]} %`, e.clientX, e.clientY); }; cv.onmouseleave = sinTip;
  $('#e-leg').innerHTML = (mv === 'seg' ? S.nombres.map((n, g) => [n, S.colores[g]]) : Object.entries(LISA).filter(([n]) => n === 'No significativo' || d.lisa.includes(n))).map(([n, c]) => `<span><i style="background:${c}"></i>${n}</span>`).join('');
  $('#e-lis-d').textContent = 'Composición en ' + a;
  $('#e-lis').innerHTML = S.nombres.map((n, g) => `<div style="margin:12px 0;border-left:4px solid ${S.colores[g]};padding-left:10px"><b style="font-size:13px">${n}</b><div style="font-size:12.5px;color:var(--ink2)">${d.estado.filter((e, i) => d.seg[i] === g).sort().join(', ') || '—'}</div></div>`).join('');
}

let rt, anchoPrev = innerWidth; addEventListener('resize', () => { if (innerWidth === anchoPrev) return; anchoPrev = innerWidth; clearTimeout(rt); rt = setTimeout(() => { if (mapaMun) { mapaMun.proyectar(); pintarMapa(mIdx, false); } if (mapaEdo) { mapaEdo.proyectar(); dibujaEstados(); } }, 250); });
mostrar('inicio');
</script>
</body>
</html>
"""

In [ ]:
# Guardar el dashboard
def anillos(geom, dec=2):
    """Contornos de un polígono como listas planas [lon, lat, lon, lat, ...] para dibujarlos en el navegador."""
    partes = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
    out = []
    for pol in partes:
        xy = np.round(np.asarray(pol.exterior.coords), dec)
        xy = xy[np.r_[True, np.any(np.diff(xy, axis=0) != 0, axis=1)]]
        if len(xy) >= 4:
            out.append(xy.ravel().tolist())
    return out


GEO_MAPA = {c: anillos(g) for c, g in zip(GEO_SIMPLE.cvegeo, GEO_SIMPLE.geometry) if g is not None}
GEO_MAPA_EDO = {e: anillos(g) for e, g in zip(_ge.edo, _ge.geometry)}
geo_mapa_json = json.dumps(GEO_MAPA, separators=(',', ':'))
geo_mapa_edo_json = json.dumps(GEO_MAPA_EDO, separators=(',', ':'))

html_doc = PLANTILLA.replace('__DATOS__', datos_json).replace('__GEO__', geo_mapa_json).replace('__GEOE__', geo_mapa_edo_json)
RUTA_DASH = f'{DIR_SALIDA}/dashboard_perfiles_ciudadanos.html'
with open(RUTA_DASH, 'w', encoding='utf-8') as fh:
    fh.write(html_doc)
print(f"Dashboard: {RUTA_DASH} ({len(html_doc) / 1e6:.1f} MB)")
if MOSTRAR_DASHBOARD_EN_NOTEBOOK:
    display(
        HTML(
            f'<iframe srcdoc="{html.escape(html_doc, quote=True)}" width="100%" height="1100" style="border:0;border-radius:16px"></iframe>'
        )
    )

## 16. Entregables

Se escribe un resumen metodológico, que sirve de base para el documento técnico, y se empaqueta todo en un ZIP.

In [ ]:
# Resumen metodológico y ZIP
def _tabla(df):
    """Tabla en formato Markdown (o texto simple si falta la librería tabulate)."""
    try:
        return df.to_markdown()
    except Exception:
        return df.to_string()


# Resumen metodológico: base para el documento técnico
with open(f'{DIR_SALIDA}/resumen_metodologico.md', 'w', encoding='utf-8') as fh:
    fh.write(
        f"# Perfiles ciudadanos de participación. {EQUIPO}\n\n## Datos\n"
        f"- Fuente: INE, Conteos Censales de Participación Ciudadana {min(ANIOS)}–{ULT}; familia de archivos «{FAMILIA}».\n"
        f"- {len(datos_modelo):,} observaciones municipio-elección; {datos_modelo.cvegeo.nunique():,} municipios; {cob:.1%} con polígono INEGI.\n\n"
        f"## Validación\n{_tabla(val)}\n\n"
        f"## Variables\n"
        + ''.join(f"- {ETIQ.get(f, f)}\n" for f in VARIABLES_SEGMENTACION)
        + f"\n## Selección de k\n{METRICAS.round(3).to_string()}\n\nk elegido = {num_perfiles}. Fidelidad del árbol = {FIDELIDAD:.1%}.\n\n"
        f"## Análisis espacial\n{ESPACIAL.round(3).to_string()}\n\n## Perfiles\n"
        + ''.join(
            f"\n### P{p + 1}. {NOMBRES[p]}\n{NARRATIVAS[p]}\n\nCómo reconocerlo: {REGLAS.get(p, ('',))[0]}\n\nEstrategias: "
            + '; '.join(e['titulo'] for e in ESTRATEGIAS[p])
            + '\n'
            for p in range(num_perfiles)
        )
    )
zip_salida = shutil.make_archive(f'{BASE}/entregables_reto1', 'zip', DIR_SALIDA)
print("Contenido de la carpeta de salida:")
for f in sorted(glob.glob(f'{DIR_SALIDA}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f"  {os.path.relpath(f, DIR_SALIDA):70s} {os.path.getsize(f) / 1e6:6.2f} MB")
print(f"\nZIP: {zip_salida}")
if EN_COLAB:
    from google.colab import files

    files.download(zip_salida)